# Aman Carbon-AI
Explainable, modular screening of carbon-credit projects using public registry, issuance and claim data.

**How to run.** Upload the Berkeley Voluntary Registry Offsets Database files to `/content` (or a folder listed in Section 1):
`VROD-registry-files--2024-12-year-end.xlsx`, `VROD-registry-files--2025-12.xlsx`, `VROD-registry-files--2026-02.xlsx` and
`Voluntary-Registry-Offsets-Database--v2026-04.xlsx`. Then run all cells from top to bottom in one session. Every result uses seed 42.

**Optional keys.** The on-chain lookup in Section 7 reads `ETHERSCAN_API_KEY` from the environment; no key is stored in this notebook.

**Outputs.** Tables and figures are written to `v4_outputs/`, `v5_outputs/` and `/content/v9_outputs`–`/content/v13_figures`; the last cell zips them for download.

## 1. Setup and data loading
Sets the random seed and loads the Berkeley VROD registry files.

In [ ]:
# Global random seed, set before anything random runs
import random, numpy as np
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

In [ ]:
# === CELL 1 (replacement): find, validate and load the VROD Excel files ===
import os, glob, zipfile
import pandas as pd
import openpyxl

SEARCH_DIRS = [".", "/content", "/content/drive/MyDrive", os.path.expanduser("~/Downloads")]
EXPECTED_MB = {"v2026-04": 15, "2024-12": 44, "2025-12": 50, "2026-02": 51}   # approx. sizes of the originals

def find_file(pattern):
    """Find a file by pattern in common folders; tolerates browser renames like 'name__1_.xlsx' or 'name (1).xlsx'."""
    for d in SEARCH_DIRS:
        hits = sorted(glob.glob(os.path.join(d, pattern)), key=len)   # shortest name = the original, not the (1) copy
        if hits:
            return hits[0]
    raise FileNotFoundError(f"No file matching '{pattern}' in {SEARCH_DIRS}. Upload it or fix the folder.")

def check_excel(path, tag):
    size_mb = os.path.getsize(path) / 1e6
    if not zipfile.is_zipfile(path):
        raise RuntimeError(
            f"{os.path.basename(path)} is damaged ({size_mb:.1f} MB, expected ~{EXPECTED_MB[tag]} MB). "
            "The upload was probably cut off. Delete it, re-upload, and wait until the upload finishes 100%.")
    if size_mb < 0.8 * EXPECTED_MB[tag]:
        print(f"WARNING: {os.path.basename(path)} is only {size_mb:.1f} MB (expected ~{EXPECTED_MB[tag]} MB).")
    print(f"OK  {tag:9s} {size_mb:5.1f} MB  {path}")
    return path

FILE_HARMONIZED = check_excel(find_file("Voluntary-Registry-Offsets-Database--v2026-04*.xlsx"), "v2026-04")
FILE_2024       = check_excel(find_file("VROD-registry-files--2024-12*.xlsx"), "2024-12")
FILE_2025       = check_excel(find_file("VROD-registry-files--2025-12*.xlsx"), "2025-12")
FILE_2026_02    = check_excel(find_file("VROD-registry-files--2026-02*.xlsx"), "2026-02")

# --- 1. Harmonized cross-registry database (header is on row 4) ---
df_harmonized = pd.read_excel(FILE_HARMONIZED, sheet_name="PROJECTS", header=3)
print("\nHarmonized PROJECTS shape:", df_harmonized.shape)

# --- 2. Raw per-registry snapshots ---
raw_files = {"2024-12": FILE_2024, "2025-12": FILE_2025, "2026-02": FILE_2026_02}
registry_sheets = ["Verra Projects", "ACR Projects", "Gold Projects", "CAR Projects", "ART Projects"]
raw_data = {}
for snapshot_date, filepath in raw_files.items():
    wb = openpyxl.load_workbook(filepath, read_only=True)
    available = wb.sheetnames
    wb.close()
    raw_data[snapshot_date] = {}
    for sheet in registry_sheets:
        if sheet in available:
            raw_data[snapshot_date][sheet] = pd.read_excel(filepath, sheet_name=sheet, header=0)
            print(f"{snapshot_date} | {sheet}: {raw_data[snapshot_date][sheet].shape}")
        else:
            print(f"{snapshot_date} | {sheet}: NOT PRESENT")

# --- Sanity check: Kariba (Verra ID 902) should read 'Withdrawn' in every snapshot ---
for snapshot_date in raw_files:
    v = raw_data[snapshot_date]["Verra Projects"]
    row = v[v["ID"] == 902]
    print(f"{snapshot_date} Kariba status: {row['Status'].values[0] if not row.empty else 'NOT FOUND'}")

# Later cells (e.g. the 'Verra VCUS' sheet) should use these variables, e.g.
#   pd.read_excel(FILE_2026_02, sheet_name="Verra VCUS")
# instead of a hard-coded file name.

## 2. Registry panel and status tiers
Builds the multi-registry status panel and classifies each status into normal, watch or risk.

In [ ]:
# --- Corrected: some registries' raw ID column already includes the prefix, others don't ---
registry_config = {
    "Verra Projects": {"prefix": "VCS", "id_col": "ID",         "name_col": "Name",         "status_col": "Status",          "id_has_prefix": False},
    "ACR Projects":   {"prefix": "ACR", "id_col": "Project ID", "name_col": "Project Name", "status_col": "Voluntary Status", "id_has_prefix": True},
    "Gold Projects":  {"prefix": "GS",  "id_col": "GSID",       "name_col": "Project Name", "status_col": "Status",          "id_has_prefix": False},
    "CAR Projects":   {"prefix": "CAR", "id_col": "Project ID", "name_col": "Project Name", "status_col": "Status",          "id_has_prefix": True},
    "ART Projects":   {"prefix": "ART", "id_col": "Program ID", "name_col": "Program Name", "status_col": "Status",          "id_has_prefix": True},
}

panel_rows = []

for snapshot_date, sheets in raw_data.items():
    for sheet_name, df in sheets.items():
        cfg = registry_config[sheet_name]
        for _, row in df.iterrows():
            raw_id = row[cfg["id_col"]]
            key = str(raw_id) if cfg["id_has_prefix"] else f"{cfg['prefix']}{raw_id}"
            panel_rows.append({
                "project_key": key,
                "name": row.get(cfg["name_col"]),
                "registry": cfg["prefix"],
                "snapshot": snapshot_date,
                "status": row.get(cfg["status_col"]),
            })

for _, row in df_harmonized.iterrows():
    panel_rows.append({
        "project_key": str(row["Project ID"]),
        "name": row.get("Project Name"),
        "registry": row.get("Voluntary Registry"),
        "snapshot": "2026-04",
        "status": row.get("Voluntary Status"),
    })

panel_long = pd.DataFrame(panel_rows)
print("Long panel shape:", panel_long.shape)

panel_wide = panel_long.pivot_table(
    index=["project_key", "name", "registry"],
    columns="snapshot",
    values="status",
    aggfunc="first"
).reset_index()

snapshot_order = ["2024-12", "2025-12", "2026-02", "2026-04"]
panel_wide = panel_wide[["project_key", "name", "registry"] + [c for c in snapshot_order if c in panel_wide.columns]]

status_cols = [c for c in snapshot_order if c in panel_wide.columns]
panel_wide["status_changed"] = panel_wide[status_cols].nunique(axis=1, dropna=True) > 1

# also check the 2026-04 column is no longer all NaN for ACR/CAR/ART rows
print("\nRows with a non-null 2026-04 value:", panel_wide["2026-04"].notna().sum(), "out of", len(panel_wide))

changed_projects = panel_wide[panel_wide["status_changed"]].copy()
print(f"Projects with a status change across snapshots: {len(changed_projects)}")
changed_projects.to_csv("status_changed_projects.csv", index=False)
print("Saved to status_changed_projects.csv")

changed_projects.head(20)

In [ ]:
# --- 7. Classify all statuses into three tiers ---

risk_flag_statuses = {
    "Canceled", "cancelled",
    "Withdrawn",
    "Inactive",
    "Rejected by Administrator",
    "Registration request denied",
    "Registration and verification approval request denied",
    "Verification approval request denied",
    "Crediting Period Renewal Request Denied",
    "Crediting Period Renewal and Verification Approval Request Denied",
    "validation_unsuccessful",
}

watch_statuses = {
    "On Hold - see notification letter",
    "Late to verify",
}

# everything else (Listed, Registered, Completed, Under development/validation,
# Transitioned, various "...requested" states, etc.) is treated as normal progression

def classify(status):
    if status in risk_flag_statuses:
        return "risk_flag"
    elif status in watch_statuses:
        return "watch"
    elif pd.isna(status):
        return None
    else:
        return "normal"

panel_long["status_tier"] = panel_long["status"].apply(classify)

# --- 8. Find projects that moved INTO a risk_flag (or watch) status at some point ---
# i.e. their tier sequence across snapshots includes a transition from normal/watch -> risk_flag

def had_risk_transition(group):
    tiers_by_snapshot = group.sort_values("snapshot")[["snapshot", "status_tier"]]
    tiers = tiers_by_snapshot["status_tier"].tolist()
    # True if risk_flag appears anywhere AND it wasn't risk_flag in every single snapshot
    # (we want the transition, not projects that were always flagged, e.g. incomplete history)
    return "risk_flag" in tiers

flagged_keys = (
    panel_long.groupby("project_key")
    .filter(had_risk_transition)["project_key"]
    .unique()
)

candidate_pool = panel_wide[panel_wide["project_key"].isin(flagged_keys)].copy()
print(f"Candidate pool (ever reached a risk-flag status): {len(candidate_pool)}")

# separate out ones that specifically show a normal->risk_flag transition (the real Kariba-style pattern)
status_cols = ["2024-12", "2025-12", "2026-02", "2026-04"]

def moved_into_risk(row):
    seq = [row[c] for c in status_cols if c in row and pd.notna(row[c])]
    tiers = [classify(s) for s in seq]
    for i in range(1, len(tiers)):
        if tiers[i] == "risk_flag" and tiers[i-1] in ("normal", "watch"):
            return True
    return False

candidate_pool["moved_into_risk"] = candidate_pool.apply(moved_into_risk, axis=1)
true_transitions = candidate_pool[candidate_pool["moved_into_risk"]].copy()

print(f"Projects with an observed normal/watch -> risk_flag transition: {len(true_transitions)}")
true_transitions.to_csv("true_risk_transitions.csv", index=False)
print("Saved to true_risk_transitions.csv")

true_transitions.head(20)

In [ ]:
# --- 9. Bring in "Total Credits Issued" from the harmonized file to prioritize by materiality ---

credits_lookup = df_harmonized.set_index(df_harmonized["Project ID"].astype(str))["Total Credits \nIssued"]

true_transitions["total_credits_issued"] = true_transitions["project_key"].map(credits_lookup)

prioritized = true_transitions.sort_values("total_credits_issued", ascending=False)
prioritized.to_csv("prioritized_risk_transitions.csv", index=False)

print("Top 20 by credits issued:")
prioritized[["project_key", "name", "registry", "total_credits_issued", "2024-12", "2025-12", "2026-02", "2026-04"]].head(20)

## 3. Confirmed cases and transaction features (preliminary analysis)
Transaction histories and project-level features for the confirmed cases and type- and size-matched controls.

In [ ]:
# --- 10. Extract full issuance/retirement transaction history for the confirmed CQC projects ---

cqc_project_ids = [2340, 2341, 2371, 2380, 2890, 2981, 4157]  # confirmed by Verra's own announcement

# Use the most recent snapshot (2026-02) for the fullest transaction history
df_vcus = pd.read_excel("VROD-registry-files--2026-02.xlsx", sheet_name="Verra VCUS", header=0)

cqc_transactions = df_vcus[df_vcus["ID"].isin(cqc_project_ids)].copy()
cqc_transactions = cqc_transactions.sort_values(["ID", "Issuance Date"])

print(f"Total transaction rows for CQC cluster: {len(cqc_transactions)}")
print(f"Projects covered: {sorted(cqc_transactions['ID'].unique())}")
print(f"\nDate range: {cqc_transactions['Issuance Date'].min()} to {cqc_transactions['Issuance Date'].max()}")
print(f"Total quantity issued (this cluster): {cqc_transactions['Quantity Issued'].sum():,.0f}")
print(f"Rows with a retirement/cancellation date: {cqc_transactions['Retirement/Cancellation Date'].notna().sum()}")

cqc_transactions.to_csv("cqc_transaction_history.csv", index=False)
print("\nSaved to cqc_transaction_history.csv")

cqc_transactions.head(20)

In [ ]:
# --- 11. Second cluster: Guizhou/Anhui afforestation rejections ---
afforestation_cluster_ids = [1847, 1864, 1865, 2070, 1855, 1930]  # from your prioritized list

afforestation_transactions = df_vcus[df_vcus["ID"].isin(afforestation_cluster_ids)].copy()
print(f"Afforestation cluster transaction rows: {len(afforestation_transactions)}")
print(f"Projects found: {sorted(afforestation_transactions['ID'].unique())}")

# --- 12. Build a control group: legitimate, long-running, high-status projects ---
# Same registry (VCS), similarly large, but Completed/Registered with NO risk-flag transition ever
control_candidates = panel_wide[
    (panel_wide["registry"] == "VCS") &
    (~panel_wide["project_key"].isin(flagged_keys)) &          # never touched a risk-flag status
    (panel_wide["2026-04"].isin(["Registered", "Completed"]))   # currently healthy
].copy()

# bring in credit volume to match scale with our fraud cluster (avoid comparing huge vs tiny projects)
control_candidates["total_credits_issued"] = control_candidates["project_key"].map(credits_lookup)
control_candidates = control_candidates.sort_values("total_credits_issued", ascending=False)

# take a sample roughly matched in size to our positive cluster (a few hundred thousand to a few million credits)
control_sample = control_candidates[
    (control_candidates["total_credits_issued"] > 100_000) &
    (control_candidates["total_credits_issued"] < 3_000_000)
].head(15)

print(f"\nControl group candidates: {len(control_sample)}")
control_ids = control_sample["project_key"].str.replace("VCS", "").astype(int).tolist()

control_transactions = df_vcus[df_vcus["ID"].isin(control_ids)].copy()
print(f"Control group transaction rows: {len(control_transactions)}")

# --- 13. Combine everything into one labeled dataset ---
cqc_transactions["label"] = "confirmed_fraud"
afforestation_transactions["label"] = "flagged_unconfirmed"
control_transactions["label"] = "control_legitimate"

labeled_dataset = pd.concat([cqc_transactions, afforestation_transactions, control_transactions], ignore_index=True)
labeled_dataset.to_csv("labeled_transaction_dataset.csv", index=False)

print(f"\nFinal labeled dataset: {len(labeled_dataset)} rows")
print(labeled_dataset["label"].value_counts())

In [ ]:
# --- 14. Aggregate transactions into project-level features ---

def build_project_features(df, label):
    feats = []
    for pid, group in df.groupby("ID"):
        group = group.sort_values("Issuance Date")
        n_batches = len(group)
        total_issued = group["Quantity Issued"].sum()
        n_retired = group["Retirement/Cancellation Date"].notna().sum()
        pct_retired = n_retired / n_batches if n_batches > 0 else 0

        # issuance lag: days between vintage end and issuance date (longer lag = more suspicious)
        lag_days = (pd.to_datetime(group["Issuance Date"]) - pd.to_datetime(group["Vintage End"])).dt.days
        mean_lag = lag_days.mean()

        # batch size variability (very uneven batch sizes can indicate irregular issuance behavior)
        batch_cv = group["Quantity Issued"].std() / group["Quantity Issued"].mean() if group["Quantity Issued"].mean() > 0 else 0

        feats.append({
            "project_id": pid,
            "label": label,
            "n_batches": n_batches,
            "total_issued": total_issued,
            "pct_retired": pct_retired,
            "mean_issuance_lag_days": mean_lag,
            "batch_size_cv": batch_cv,
        })
    return pd.DataFrame(feats)

feat_fraud = build_project_features(cqc_transactions, "confirmed_fraud")
feat_flagged = build_project_features(afforestation_transactions, "flagged_unconfirmed")
feat_control = build_project_features(control_transactions, "control_legitimate")

project_features = pd.concat([feat_fraud, feat_flagged, feat_control], ignore_index=True)
project_features.to_csv("project_level_features.csv", index=False)
print(project_features.groupby("label")[["n_batches","total_issued","pct_retired","mean_issuance_lag_days","batch_size_cv"]].mean())

# --- 15. Statistical test: is confirmed_fraud meaningfully different from control_legitimate? ---
from scipy.stats import mannwhitneyu

fraud_vals = project_features[project_features["label"]=="confirmed_fraud"]
control_vals = project_features[project_features["label"]=="control_legitimate"]

print("\n--- Mann-Whitney U test: confirmed_fraud vs control_legitimate ---")
for col in ["n_batches","total_issued","pct_retired","mean_issuance_lag_days","batch_size_cv"]:
    a = fraud_vals[col].dropna()
    b = control_vals[col].dropna()
    if len(a) >= 3 and len(b) >= 3:
        stat, p = mannwhitneyu(a, b, alternative="two-sided")
        flag = " <-- significant (p<0.05)" if p < 0.05 else ""
        print(f"{col:28s} fraud_median={a.median():>12,.1f}  control_median={b.median():>12,.1f}  p={p:.4f}{flag}")
    else:
        print(f"{col:28s} not enough data (n_fraud={len(a)}, n_control={len(b)})")

In [ ]:
# --- 16. Build a properly SIZE-MATCHED control group (fixes the selection bias above) ---

fraud_sizes = feat_fraud["total_issued"].values
print("Fraud cluster size range:", fraud_sizes.min(), "-", fraud_sizes.max(), "median:", pd.Series(fraud_sizes).median())

# Match control projects within a similar size band to the fraud cluster (not just "large")
size_lower = fraud_sizes.min() * 0.3
size_upper = fraud_sizes.max() * 3

control_candidates_matched = panel_wide[
    (panel_wide["registry"] == "VCS") &
    (~panel_wide["project_key"].isin(flagged_keys)) &
    (panel_wide["2026-04"].isin(["Registered", "Completed"]))
].copy()
control_candidates_matched["total_credits_issued"] = control_candidates_matched["project_key"].map(credits_lookup)

control_matched = control_candidates_matched[
    (control_candidates_matched["total_credits_issued"] >= size_lower) &
    (control_candidates_matched["total_credits_issued"] <= size_upper)
].sample(n=min(20, len(control_candidates_matched)), random_state=42)  # random, not sorted-by-size, to avoid re-introducing bias

print(f"\nSize-matched control group: {len(control_matched)} projects")

control_ids_matched = control_matched["project_key"].str.replace("VCS", "").astype(int).tolist()
control_transactions_matched = df_vcus[df_vcus["ID"].isin(control_ids_matched)].copy()
feat_control_matched = build_project_features(control_transactions_matched, "control_legitimate")

print(feat_control_matched[["total_issued","n_batches"]].describe())

# --- Re-run the Mann-Whitney test on the CORRECTED, size-matched comparison ---
print("\n--- Mann-Whitney U test (size-matched): confirmed_fraud vs control_legitimate ---")
for col in ["n_batches","total_issued","pct_retired","mean_issuance_lag_days","batch_size_cv"]:
    a = feat_fraud[col].dropna()
    b = feat_control_matched[col].dropna()
    if len(a) >= 3 and len(b) >= 3:
        stat, p = mannwhitneyu(a, b, alternative="two-sided")
        flag = " <-- significant (p<0.05)" if p < 0.05 else ""
        print(f"{col:28s} fraud_median={a.median():>12,.1f}  control_median={b.median():>12,.1f}  p={p:.4f}{flag}")

In [ ]:
# --- 18. Build a control group matched on BOTH project type AND size (fixed for non-numeric IDs like VCSOPR2) ---

df_vcs_projects = pd.read_excel("VROD-registry-files--2026-02.xlsx", sheet_name="Verra Projects", header=0)
type_lookup = df_vcs_projects.set_index("ID")["Project Type"]

# Safe conversion: anything that isn't a clean "VCS" + number (e.g. VCSOPR2) becomes NaN instead of crashing
control_candidates_matched["numeric_id"] = pd.to_numeric(
    control_candidates_matched["project_key"].str.replace("VCS", "", regex=False),
    errors="coerce"
)
control_candidates_matched = control_candidates_matched.dropna(subset=["numeric_id"]).copy()
control_candidates_matched["numeric_id"] = control_candidates_matched["numeric_id"].astype(int)

control_candidates_matched["project_type"] = control_candidates_matched["numeric_id"].map(type_lookup)

control_type_matched = control_candidates_matched[
    (control_candidates_matched["project_type"] == "Energy demand") &
    (control_candidates_matched["total_credits_issued"] >= size_lower) &
    (control_candidates_matched["total_credits_issued"] <= size_upper)
].copy()

print(f"Energy demand + size-matched control candidates: {len(control_type_matched)}")

if len(control_type_matched) < 8:
    print("Too few — widening size band to get enough Energy demand controls...")
    control_type_matched = control_candidates_matched[
        (control_candidates_matched["project_type"] == "Energy demand")
    ].copy()
    print(f"All available Energy demand controls (any size): {len(control_type_matched)}")

control_ids_type_matched = control_type_matched["numeric_id"].tolist()
control_transactions_type_matched = df_vcus[df_vcus["ID"].isin(control_ids_type_matched)].copy()
feat_control_type_matched = build_project_features(control_transactions_type_matched, "control_legitimate")

print(f"\nFinal type-matched control group: {len(feat_control_type_matched)} projects")
print(feat_control_type_matched[["total_issued","mean_issuance_lag_days"]].describe())

print("\n--- Mann-Whitney U test (type + size matched): confirmed_fraud vs control_legitimate ---")
for col in ["n_batches","total_issued","pct_retired","mean_issuance_lag_days","batch_size_cv"]:
    a = feat_fraud[col].dropna()
    b = feat_control_type_matched[col].dropna()
    if len(a) >= 3 and len(b) >= 3:
        stat, p = mannwhitneyu(a, b, alternative="two-sided")
        flag = " <-- significant (p<0.05)" if p < 0.05 else ""
        print(f"{col:28s} fraud_median={a.median():>12,.1f}  control_median={b.median():>12,.1f}  p={p:.4f}{flag}")
    else:
        print(f"{col:28s} not enough data (n_fraud={len(a)}, n_control={len(b)})")

In [ ]:
# --- Add the second confirmed cluster and expand the labeled dataset ---

guizhou_confirmed_ids = [1847, 1864, 1865, 2070]  # officially confirmed by Verra, Dec 2025

guizhou_confirmed_transactions = df_vcus[df_vcus["ID"].isin(guizhou_confirmed_ids)].copy()
feat_guizhou_confirmed = build_project_features(guizhou_confirmed_transactions, "confirmed_fraud")

# Combine with CQC into one unified confirmed-fraud group
feat_fraud_expanded = pd.concat([feat_fraud, feat_guizhou_confirmed], ignore_index=True)
print(f"Expanded confirmed-fraud sample: n={len(feat_fraud_expanded)} (was n={len(feat_fraud)})")

# Re-run the type+size-matched test with the larger, two-scandal-type sample
print("\n--- Mann-Whitney U test (EXPANDED, n=10, two fraud types) ---")
for col in ["n_batches","total_issued","pct_retired","mean_issuance_lag_days","batch_size_cv"]:
    a = feat_fraud_expanded[col].dropna()
    b = feat_control_type_matched[col].dropna()
    if len(a) >= 3 and len(b) >= 3:
        stat, p = mannwhitneyu(a, b, alternative="two-sided")
        flag = " <-- significant (p<0.05)" if p < 0.05 else ""
        print(f"{col:28s} fraud_median={a.median():>12,.1f}  control_median={b.median():>12,.1f}  p={p:.4f}{flag}")

In [ ]:
# --- Rebuild control group to match the TYPE COMPOSITION of the expanded fraud sample ---
# Fraud sample is now: 6 Energy demand (CQC) + 4 Agriculture Forestry (Guizhou)

# Get Energy demand controls (already have these: control_type_matched)
# Now also get Agriculture Forestry controls, size-matched to the Guizhou cluster specifically
guizhou_sizes = feat_guizhou_confirmed["total_issued"].values
g_size_lower = guizhou_sizes.min() * 0.3
g_size_upper = guizhou_sizes.max() * 3

control_candidates_matched["project_type"] = control_candidates_matched["numeric_id"].map(type_lookup)

forestry_controls = control_candidates_matched[
    (control_candidates_matched["project_type"] == "Agriculture Forestry and Other Land Use") &
    (control_candidates_matched["total_credits_issued"] >= g_size_lower) &
    (control_candidates_matched["total_credits_issued"] <= g_size_upper)
].sample(n=min(15, len(control_candidates_matched)), random_state=42)

print(f"Forestry controls found: {len(forestry_controls)}")

forestry_control_ids = forestry_controls["numeric_id"].tolist()
forestry_control_transactions = df_vcus[df_vcus["ID"].isin(forestry_control_ids)].copy()
feat_forestry_control = build_project_features(forestry_control_transactions, "control_legitimate")

# Combine BOTH control groups to match the fraud sample's type composition (60% energy, 40% forestry)
feat_control_expanded = pd.concat([feat_control_type_matched, feat_forestry_control], ignore_index=True)
print(f"Expanded, type-composition-matched control group: n={len(feat_control_expanded)}")

# --- Re-run the test, properly controlled this time ---
print("\n--- Mann-Whitney U test (n=10 fraud, TYPE-COMPOSITION-MATCHED control) ---")
for col in ["n_batches","total_issued","pct_retired","mean_issuance_lag_days","batch_size_cv"]:
    a = feat_fraud_expanded[col].dropna()
    b = feat_control_expanded[col].dropna()
    if len(a) >= 3 and len(b) >= 3:
        stat, p = mannwhitneyu(a, b, alternative="two-sided")
        flag = " <-- significant (p<0.05)" if p < 0.05 else ""
        print(f"{col:28s} fraud_median={a.median():>12,.1f}  control_median={b.median():>12,.1f}  p={p:.4f}{flag}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import roc_auc_score

# --- Build project-level feature matrix (one row per PROJECT, not per transaction) ---
feat_all = pd.concat([feat_fraud_expanded, feat_control_expanded], ignore_index=True)
feat_all["label"] = (feat_all["label"] == "confirmed_fraud").astype(int)

feature_cols = ["n_batches", "total_issued", "pct_retired", "mean_issuance_lag_days", "batch_size_cv"]
X = feat_all[feature_cols].fillna(0).values
y = feat_all["label"].values

print(f"Dataset: {len(y)} projects ({y.sum()} fraud, {(y==0).sum()} control)")

# --- Leave-One-Out CV (required given n=10 positive cases — a random split would be meaningless) ---
loo = LeaveOneOut()
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, scale_pos_weight=(y==0).sum()/y.sum(), eval_metric="logloss", random_state=42),
}

results = {}
for name, model in models.items():
    y_true_loo, y_score_loo = [], []
    for train_idx, test_idx in loo.split(X):
        model.fit(X[train_idx], y[train_idx])
        prob = model.predict_proba(X[test_idx])[0][1]
        y_true_loo.append(y[test_idx][0])
        y_score_loo.append(prob)
    auc_score = roc_auc_score(y_true_loo, y_score_loo)
    results[name] = auc_score
    print(f"{name}: LOOCV AUC = {auc_score:.3f}")

## 4. Status ontology
Controlled vocabulary mapping every raw status string to one of three tiers.

In [ ]:
# --- Controlled vocabulary: maps every raw status string to one of 3 normalized states ---
STATUS_ONTOLOGY = {
    "Listed": "normal", "Registered": "normal", "Completed": "normal",
    "Under development": "normal", "Under validation": "normal", "under_validation": "normal",
    "validated": "normal", "Transitioned": "normal", "Gold Standard Certified Design": "normal",
    "Gold Standard Certified Project": "normal", "Transfer to Another GHG Program": "normal",
    "Units Transferred from Approved GHG Program": "normal", "Requantification Requested": "normal",
    "Registration requested": "normal", "Verification approval requested": "normal",
    "Crediting Period Renewal Requested": "normal", "Crediting Period Renewal and Verification Approval Requested": "normal",
    "On Hold - see notification letter": "watch", "Late to verify": "watch",
    "Canceled": "risk_flag", "cancelled": "risk_flag", "Withdrawn": "risk_flag", "Inactive": "risk_flag",
    "Rejected by Administrator": "risk_flag", "Registration request denied": "risk_flag",
    "Registration and verification approval request denied": "risk_flag",
    "Verification approval request denied": "risk_flag",
    "Crediting Period Renewal Request Denied": "risk_flag",
    "Crediting Period Renewal and Verification Approval Request Denied": "risk_flag",
    "validation_unsuccessful": "risk_flag",
}
UNMAPPED = set(panel_long["status"].dropna().unique()) - set(STATUS_ONTOLOGY.keys())
print(f"Unmapped statuses (should be empty): {UNMAPPED}")
ontology_table = pd.DataFrame(list(STATUS_ONTOLOGY.items()), columns=["raw_status", "normalized_state"])
ontology_table.to_csv("status_ontology.csv", index=False)
print(ontology_table)

In [ ]:
STATUS_ONTOLOGY["Registration and verification approval requested"] = "normal"
UNMAPPED = set(panel_long["status"].dropna().unique()) - set(STATUS_ONTOLOGY.keys())
print(f"Unmapped statuses (should be empty): {UNMAPPED}")

## 5. Repaired panel, candidate pool and leakage-safe features

In [ ]:
# ===== V4-1  Setup: seed FIRST, single config, output folder =====
import os, re, json, random, platform, unicodedata, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

V4 = {
    "SNAPSHOTS": ["2024-12", "2025-12", "2026-02", "2026-04"],
    "AS_OF_CUTOFF": "2024-12-31",   # features may only use information available at the first snapshot
    "MATCH_K": 5,                   # controls per confirmed-fraud project
    "MATCH_CALIPER_SD": 0.5,        # max |log-size difference| in pooled-SD units
    "N_BOOT": 2000,
    "N_PERM": 500,
    "ALPHA": 0.05,
    "OUT_DIR": "v4_outputs",
    "CQC_IDS": [2340, 2341, 2371, 2380, 2890, 2981, 4157],
    "GUIZHOU_IDS": [1847, 1864, 1865, 2070],
    "IGNORE_PROPONENTS": {"Multiple Proponents"},   # generic label, not a developer
    "CONTAMINATION_PATTERNS": [r"c[\s\-]?quest", r"\bcqc\b", r"\btlc\b", r"baiheng"],
}
os.makedirs(V4["OUT_DIR"], exist_ok=True)
SNAPS = V4["SNAPSHOTS"]
CUTOFF = pd.Timestamp(V4["AS_OF_CUTOFF"]) if V4["AS_OF_CUTOFF"] else None

# Okabe-Ito colorblind-safe palette
CB = {"black": "#000000", "orange": "#E69F00", "sky": "#56B4E9", "green": "#009E73",
      "yellow": "#F0E442", "blue": "#0072B2", "vermillion": "#D55E00", "purple": "#CC79A7", "grey": "#999999"}

def out(fname):
    return os.path.join(V4["OUT_DIR"], fname)

def save_csv(df, fname, index=False):
    df.to_csv(out(fname), index=index)
    print(f"  -> saved {out(fname)}")

for _name in ["panel_long", "df_harmonized", "df_vcus", "STATUS_ONTOLOGY"]:
    assert _name in globals(), f"`{_name}` not found — run the original v3 cells first."
print("V4 setup OK | seed", GLOBAL_SEED, "| cutoff", CUTOFF.date() if CUTOFF is not None else None)

In [ ]:
# ===== V4-2  Gold Standard join defect: diagnose root cause, then repair keys =====
# H1 float-ID artefact: "GS1234.0" never matches "GS1234"
# H2 formatting: case / whitespace / hyphens / leading zeros
# H3 registry label mismatch ("GS" vs e.g. "GOLD") -> splits rows under the OLD composite key
# H4 different ID scheme -> detected by exact normalized-name match
KNOWN_PREFIXES = ["VCS", "ACR", "CAR", "ART", "GS"]

def normalize_key(k):
    if pd.isna(k):
        return np.nan
    s = str(k).strip().upper()
    s = re.sub(r"\.0+$", "", s)            # H1
    s = re.sub(r"[\s\-_]", "", s)          # H2
    m = re.match(r"^([A-Z]+)0*(\d+)$", s)  # H2 leading zeros
    return f"{m.group(1)}{m.group(2)}" if m else s

def registry_from_key(k):
    s = str(k).upper()
    for p in KNOWN_PREFIXES:
        if s.startswith(p):
            return p
    return "OTHER"

def normalize_name(n):
    if pd.isna(n):
        return ""
    n = unicodedata.normalize("NFKD", str(n)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", n.lower())

pl = panel_long.copy()
pl["key_raw"] = pl["project_key"].astype(str)
pl["key_norm"] = pl["project_key"].map(normalize_key)
pl["registry_label_as_given"] = pl["registry"].astype(str)
pl["registry_std"] = pl["key_norm"].map(registry_from_key)

print("H3 — registry labels as given, by snapshot:")
print(pl.groupby(["snapshot", "registry_label_as_given"]).size().unstack(0).fillna(0).astype(int))

diag_rows = []
for reg in KNOWN_PREFIXES:
    raw_raw = set(pl.loc[(pl.snapshot == "2026-02") & (pl.registry_std == reg), "key_raw"])
    har_raw = set(pl.loc[(pl.snapshot == "2026-04") & (pl.registry_std == reg), "key_raw"])
    raw_n = set(pl.loc[(pl.snapshot == "2026-02") & (pl.registry_std == reg), "key_norm"])
    har_n = set(pl.loc[(pl.snapshot == "2026-04") & (pl.registry_std == reg), "key_norm"])
    diag_rows.append({
        "registry": reg, "n_raw_2026-02": len(raw_n), "n_harmonized_2026-04": len(har_n),
        "overlap_before_normalization": len(raw_raw & har_raw),
        "overlap_after_normalization": len(raw_n & har_n),
        "raw_only_after_norm": len(raw_n - har_n), "harmonized_only_after_norm": len(har_n - raw_n),
        "example_raw_key": next(iter(sorted(raw_raw)), None), "example_harmonized_key": next(iter(sorted(har_raw)), None),
    })
join_diag = pd.DataFrame(diag_rows)
print("\nH1/H2 — key overlap between the last raw snapshot and the harmonized file:")
print(join_diag.to_string(index=False))
save_csv(join_diag, "gs_join_diagnostic.csv")

# H4 — unique exact normalized-name match for keys still unmatched
crosswalk = {}
for reg in KNOWN_PREFIXES:
    raw_side = pl[(pl.snapshot == "2026-02") & (pl.registry_std == reg)][["key_norm", "name"]].drop_duplicates("key_norm")
    har_side = pl[(pl.snapshot == "2026-04") & (pl.registry_std == reg)][["key_norm", "name"]].drop_duplicates("key_norm")
    raw_only = raw_side[~raw_side.key_norm.isin(har_side.key_norm)].assign(nm=lambda d: d["name"].map(normalize_name))
    har_only = har_side[~har_side.key_norm.isin(raw_side.key_norm)].assign(nm=lambda d: d["name"].map(normalize_name))
    raw_only = raw_only[raw_only.nm.str.len() > 10]
    har_only = har_only[har_only.nm.str.len() > 10]
    raw_unique = raw_only.drop_duplicates("nm", keep=False)
    har_unique = har_only.drop_duplicates("nm", keep=False)
    merged = har_unique.merge(raw_unique, on="nm", suffixes=("_har", "_raw"))
    for _, r in merged.iterrows():
        crosswalk[r["key_norm_har"]] = r["key_norm_raw"]
print(f"\nH4 — unique name-based crosswalk entries found: {len(crosswalk)}")
save_csv(pd.DataFrame(list(crosswalk.items()), columns=["harmonized_key", "mapped_to_raw_key"]), "name_crosswalk.csv")

pl["project_key_v4"] = pl["key_norm"].replace(crosswalk)
pl["status_tier"] = pl["status"].map(STATUS_ONTOLOGY)
unmapped = set(pl["status"].dropna()) - set(STATUS_ONTOLOGY)
assert not unmapped, f"Unmapped statuses: {unmapped}"

dup = pl.dropna(subset=["status"]).groupby(["project_key_v4", "snapshot"])["status"].nunique()
conflicts = dup[dup > 1].reset_index()
print(f"Within-snapshot status conflicts after key repair: {len(conflicts)} (resolved by first occurrence, listed in CSV)")
save_csv(conflicts, "within_snapshot_conflicts.csv")
panel_long_v4 = pl

In [ ]:
# ===== V4-3  Rebuild the panel on repaired keys; funnel; transitions by registry x snapshot pair =====
status_by_snap = (panel_long_v4.dropna(subset=["status"])
                  .groupby(["project_key_v4", "snapshot"])["status"].first().unstack("snapshot"))
all_keys = panel_long_v4["project_key_v4"].dropna().unique()
panel_v4 = status_by_snap.reindex(all_keys)
panel_v4 = panel_v4[[c for c in SNAPS if c in panel_v4.columns]]
panel_v4.index.name = "project_key"
panel_v4 = panel_v4.reset_index()
meta = panel_long_v4.groupby("project_key_v4").agg(name=("name", "first"), registry=("registry_std", "first"))
panel_v4 = panel_v4.join(meta, on="project_key")
tiers_v4 = panel_v4[SNAPS].apply(lambda col: col.map(STATUS_ONTOLOGY))

def moved_into_risk_gap_skipping(tier_row):
    t = [x for x in tier_row if pd.notna(x)]
    return any(t[i] == "risk_flag" and t[i - 1] in ("normal", "watch") for i in range(1, len(t)))

def moved_into_risk_strict(tier_row):
    t = list(tier_row)
    return any(pd.notna(t[i]) and pd.notna(t[i - 1]) and t[i] == "risk_flag" and t[i - 1] in ("normal", "watch")
               for i in range(1, len(t)))

panel_v4["n_snapshots_observed"] = panel_v4[SNAPS].notna().sum(axis=1)
panel_v4["missing_2026-04"] = panel_v4["2026-04"].isna()
panel_v4["raw_status_changed"] = panel_v4[SNAPS].nunique(axis=1, dropna=True) > 1
panel_v4["tier_changed"] = tiers_v4.nunique(axis=1, dropna=True) > 1
panel_v4["ever_risk_flag"] = (tiers_v4 == "risk_flag").any(axis=1)
panel_v4["moved_into_risk"] = tiers_v4.apply(moved_into_risk_gap_skipping, axis=1)
panel_v4["moved_into_risk_strict"] = tiers_v4.apply(moved_into_risk_strict, axis=1)

print("Did the key repair close the Gold Standard gap? Missing 2026-04 status, by registry:")
gs_gap = panel_v4.groupby("registry")["missing_2026-04"].agg(["sum", "count"])
gs_gap["pct_missing"] = (100 * gs_gap["sum"] / gs_gap["count"]).round(1)
print(gs_gap)
save_csv(gs_gap.reset_index(), "missing_2026_04_by_registry_v4.csv")

funnel = pd.DataFrame([
    ("All project keys (after key repair)", len(panel_v4)),
    ("... with >=1 observed status", int((panel_v4.n_snapshots_observed >= 1).sum())),
    ("... observed in >=2 snapshots", int((panel_v4.n_snapshots_observed >= 2).sum())),
    ("Raw status string changed", int(panel_v4.raw_status_changed.sum())),
    ("Tier changed (routine within-tier lifecycle steps excluded)", int(panel_v4.tier_changed.sum())),
    ("Ever in risk_flag tier", int(panel_v4.ever_risk_flag.sum())),
    ("Moved normal/watch -> risk_flag (v3 rule, gap-skipping) = CANDIDATE POOL", int(panel_v4.moved_into_risk.sum())),
    ("Moved normal/watch -> risk_flag (strict adjacent snapshots)", int(panel_v4.moved_into_risk_strict.sum())),
], columns=["step", "n_projects"])
print("\nCandidate-pool funnel (exclusion rule made explicit):")
print(funnel.to_string(index=False))
save_csv(funnel, "candidate_pool_funnel_v4.csv")

ev = []
for a, b in zip(SNAPS[:-1], SNAPS[1:]):
    m = tiers_v4[a].notna() & tiers_v4[b].notna()
    ev.append(pd.DataFrame({"pair": f"{a}->{b}", "registry": panel_v4.loc[m, "registry"],
                            "from": tiers_v4.loc[m, a], "to": tiers_v4.loc[m, b]}))
events = pd.concat(ev, ignore_index=True)
transition_matrix_v4 = pd.crosstab(events["from"], events["to"]).reindex(
    index=["normal", "watch", "risk_flag"], columns=["normal", "watch", "risk_flag"], fill_value=0)
print(f"\nTransition matrix (EVENTS across {events['pair'].nunique()} adjacent snapshot pairs, "
      f"{len(events):,} events, {panel_v4.loc[(panel_v4.n_snapshots_observed >= 2)].shape[0]:,} projects with >=2 observations):")
print(transition_matrix_v4)
print("Row totals:", transition_matrix_v4.sum(axis=1).to_dict())
by_reg_pair = (events.assign(kind=events["from"] + "->" + events["to"])
               .pivot_table(index=["registry", "pair"], columns="kind", aggfunc="size", fill_value=0))
save_csv(transition_matrix_v4, "transition_matrix_v4.csv", index=True)
save_csv(by_reg_pair, "transitions_by_registry_and_pair_v4.csv", index=True)
candidate_pool_v4 = panel_v4[panel_v4["moved_into_risk"]].copy()
save_csv(candidate_pool_v4, "candidate_pool_v4.csv")

In [ ]:
# ===== V4-4  Label contamination: are un-named C-Quest / Baiheng projects in the CONTROL group? =====
if "df_vcs_projects" not in globals():
    df_vcs_projects = pd.read_excel(FILE_2026_02, sheet_name="Verra Projects", header=0)
vp = df_vcs_projects.copy()
vp["ID_num"] = pd.to_numeric(vp["ID"], errors="coerce")
text_cols = [c for c in vp.columns if str(c).lower() in ("name", "proponent", "project proponent", "developer")]
print("Columns searched for developer names:", text_cols)
blob = vp[text_cols].fillna("").astype(str).agg(" ".join, axis=1).str.lower()
pattern = "|".join(V4["CONTAMINATION_PATTERNS"])
vp["developer_linked_to_confirmed_fraud"] = blob.str.contains(pattern, regex=True)

known_fraud = set(V4["CQC_IDS"] + V4["GUIZHOU_IDS"])
proponent_col = next((c for c in text_cols if "propon" in str(c).lower()), None)
if proponent_col:
    fraud_proponents = set(vp.loc[vp.ID_num.isin(known_fraud), proponent_col].dropna().astype(str)) - V4["IGNORE_PROPONENTS"]
    vp.loc[vp[proponent_col].astype(str).isin(fraud_proponents), "developer_linked_to_confirmed_fraud"] = True
    print("Proponents of the confirmed-fraud projects:", fraud_proponents)

linked_ids = set(vp.loc[vp.developer_linked_to_confirmed_fraud & ~vp.ID_num.isin(known_fraud), "ID_num"].dropna().astype(int))
print(f"Projects linked to a confirmed-fraud developer but NOT labelled fraud: {len(linked_ids)}")
save_csv(vp.loc[vp.ID_num.isin(linked_ids), ["ID"] + text_cols + ["Status"] if "Status" in vp.columns else ["ID"] + text_cols],
         "developer_linked_projects.csv")
if "feat_all" in globals():
    in_v3_controls = sorted(set(feat_all.loc[feat_all["label"] == 0, "project_id"].astype(int)) & linked_ids)
    print(f"...of which sit in the v3 CONTROL group: {len(in_v3_controls)} -> {in_v3_controls}")
    print("If this is > 0, the v3 control group is contaminated. v4 excludes them (V4-6).")

In [ ]:
# ===== V4-5  Leakage-safe features: only information available at the cutoff =====
def build_features_asof(tx, cutoff=None):
    tx = tx.copy()
    for c in ["Issuance Date", "Vintage End", "Retirement/Cancellation Date"]:
        tx[c] = pd.to_datetime(tx[c], errors="coerce")
    tx["Quantity Issued"] = pd.to_numeric(tx["Quantity Issued"], errors="coerce").fillna(0)
    if cutoff is not None:
        tx = tx[tx["Issuance Date"] <= cutoff]
        retired = tx["Retirement/Cancellation Date"].notna() & (tx["Retirement/Cancellation Date"] <= cutoff)
    else:
        retired = tx["Retirement/Cancellation Date"].notna()
    tx = tx.assign(_retired=retired.astype(int), _retired_qty=np.where(retired, tx["Quantity Issued"], 0),
                   _lag=(tx["Issuance Date"] - tx["Vintage End"]).dt.days)
    has_type = "Project Type" in tx.columns
    rows = []
    for pid, g in tx.groupby("ID"):
        q = g["Quantity Issued"]
        rows.append({
            "project_id": int(pid),
            "project_type": g["Project Type"].mode().iat[0] if has_type and g["Project Type"].notna().any() else np.nan,
            "n_batches": len(g),
            "total_issued": q.sum(),
            "share_batches_retired": g["_retired"].mean(),
            "share_credits_retired": g["_retired_qty"].sum() / q.sum() if q.sum() > 0 else np.nan,
            "mean_issuance_lag_days": g["_lag"].mean(),
            "median_issuance_lag_days": g["_lag"].median(),
            "batch_size_cv": q.std(ddof=1) / q.mean() if len(g) > 1 and q.mean() > 0 else np.nan,
            "first_issuance": g["Issuance Date"].min(),
        })
    return pd.DataFrame(rows)

vcus = df_vcus.copy()
vcus["ID"] = pd.to_numeric(vcus["ID"], errors="coerce")
vcus = vcus.dropna(subset=["ID"])
feat_asof = build_features_asof(vcus, CUTOFF)
feat_full = build_features_asof(vcus, None)
if feat_asof["project_type"].isna().all() and "type_lookup" in globals():
    feat_asof["project_type"] = feat_asof["project_id"].map(type_lookup)
    feat_full["project_type"] = feat_full["project_id"].map(type_lookup)

fraud_cluster = {**{i: "CQC" for i in V4["CQC_IDS"]}, **{i: "Guizhou" for i in V4["GUIZHOU_IDS"]}}
for fdf in (feat_asof, feat_full):
    fdf["cluster"] = fdf["project_id"].map(fraud_cluster)
    fdf["is_fraud"] = fdf["cluster"].notna().astype(int)

present_full = set(feat_full.loc[feat_full.is_fraud == 1, "project_id"])
present_asof = set(feat_asof.loc[feat_asof.is_fraud == 1, "project_id"])
print(f"Confirmed-fraud projects with VCU rows (full history): {len(present_full)} of {len(fraud_cluster)}")
print(f"Confirmed-fraud projects with VCU rows before {CUTOFF.date() if CUTOFF is not None else 'N/A'}: {len(present_asof)}")
print("Missing from VCU file entirely:", sorted(set(fraud_cluster) - present_full))

## 6. Matching, statistics and machine learning (preliminary analysis)

In [ ]:
# ===== V4-6  Matching: exact on project type, nearest-neighbour on log(size), caliper, k:1, no replacement =====
def smd(a, b):
    a, b = pd.Series(a, dtype=float).dropna(), pd.Series(b, dtype=float).dropna()
    s = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return np.nan if s == 0 or np.isnan(s) else (a.mean() - b.mean()) / s

vcs_status = panel_v4[panel_v4["registry"] == "VCS"].copy()
vcs_status["ID_num"] = pd.to_numeric(vcs_status["project_key"].str.replace("VCS", "", regex=False), errors="coerce")
healthy_now = vcs_status[vcs_status[SNAPS].ffill(axis=1).iloc[:, -1].isin(["Registered", "Completed"])]
eligible_ids = set(healthy_now.loc[~healthy_now["ever_risk_flag"] & (tiers_v4.loc[healthy_now.index] != "watch").all(axis=1),
                                   "ID_num"].dropna().astype(int))
eligible_ids -= linked_ids
eligible_ids -= set(fraud_cluster)

def build_matched_sample(feat, k=V4["MATCH_K"], caliper_sd=V4["MATCH_CALIPER_SD"]):
    f = feat[feat["total_issued"] > 0].copy()
    f["log_size"] = np.log10(f["total_issued"])
    cases = f[f.is_fraud == 1].sort_values("log_size", ascending=False)
    pool = f[(f.is_fraud == 0) & f.project_id.isin(eligible_ids)]
    caliper = caliper_sd * pd.concat([cases.log_size, pool.log_size]).std()
    used, rows, report = set(), [], []
    for _, c in cases.iterrows():
        cand = pool[(pool.project_type == c.project_type) & ~pool.project_id.isin(used)].copy()
        cand["dist"] = (cand.log_size - c.log_size).abs()
        chosen = cand[cand.dist <= caliper].nsmallest(k, "dist")
        used |= set(chosen.project_id)
        rows.append(c.to_frame().T.assign(match_set=c.project_id, role="case"))
        rows.append(chosen.drop(columns="dist").assign(match_set=c.project_id, role="control", cluster=c.cluster))
        report.append({"case": c.project_id, "cluster": c.cluster, "type": c.project_type,
                       "n_controls": len(chosen), "max_log10_size_gap": chosen.dist.max() if len(chosen) else np.nan})
    matched = pd.concat(rows, ignore_index=True)
    for col in ["log_size", "n_batches", "total_issued", "share_credits_retired", "share_batches_retired",
                "mean_issuance_lag_days", "batch_size_cv", "is_fraud"]:
        matched[col] = pd.to_numeric(matched[col], errors="coerce")
    bal = []
    ctrl = matched[matched.role == "control"]
    bal.append({"variable": "log10(total issued)", "SMD_before": smd(cases.log_size, pool.log_size),
                "SMD_after": smd(cases.log_size, ctrl.log_size)})
    for t in cases.project_type.dropna().unique():
        bal.append({"variable": f"type = {t}", "SMD_before": smd(cases.project_type == t, pool.project_type == t),
                    "SMD_after": smd(cases.project_type == t, ctrl.project_type == t)})
    return matched, pd.DataFrame(report), pd.DataFrame(bal), caliper

matched_v4, match_report, balance_v4, caliper_used = build_matched_sample(feat_asof)
print(f"Caliper = {caliper_used:.3f} log10 units | eligible control pool = {len(eligible_ids)} projects")
print(match_report.to_string(index=False))
print("\nBalance (|SMD| < 0.1 is the usual target):")
print(balance_v4.round(3).to_string(index=False))
save_csv(match_report, "matching_report_v4.csv"); save_csv(balance_v4, "balance_diagnostics_v4.csv")
save_csv(matched_v4, "matched_sample_v4.csv")

In [ ]:
# ===== V4-7  Statistics on the matched, leakage-safe sample =====
TEST_FEATURES = ["n_batches", "share_credits_retired", "mean_issuance_lag_days", "batch_size_cv"]

def rank_biserial(a, b):
    """Positive = values tend to be HIGHER in group a (fraud). Note: v3 used the opposite sign."""
    u1 = mannwhitneyu(a, b, alternative="two-sided").statistic
    return 2 * u1 / (len(a) * len(b)) - 1

def boot_rb(a, b, n_boot, alphas, seed=GLOBAL_SEED):
    rng = np.random.default_rng(seed)
    vals = np.array([rank_biserial(rng.choice(a, len(a)), rng.choice(b, len(b))) for _ in range(n_boot)])
    return {al: np.percentile(vals, [100 * al / 2, 100 * (1 - al / 2)]) for al in alphas}

def matched_set_rank_test(df, col, n_perm, seed=GLOBAL_SEED):
    rng = np.random.default_rng(seed)
    sets = []
    for _, g in df.dropna(subset=[col]).groupby("match_set"):
        if (g.role == "case").sum() == 1 and (g.role == "control").sum() >= 1:
            r = g[col].rank(method="average").values
            sets.append(((r[(g.role == "case").values][0] - 1) / (len(g) - 1), (r - 1) / (len(g) - 1)))
    if not sets:
        return np.nan, np.nan, 0
    obs = np.mean([s[0] for s in sets])
    null = np.array([np.mean([rng.choice(s[1]) for s in sets]) for _ in range(n_perm)])
    p = (np.sum(np.abs(null - 0.5) >= abs(obs - 0.5)) + 1) / (n_perm + 1)
    return obs, p, len(sets)

def effect_size_table(fraud_df, ctrl_df, features, matched_df=None, label=""):
    m = len(features)
    rows = []
    for col in features:
        a, b = fraud_df[col].dropna().values.astype(float), ctrl_df[col].dropna().values.astype(float)
        p = mannwhitneyu(a, b, alternative="two-sided").pvalue
        cis = boot_rb(a, b, V4["N_BOOT"], [V4["ALPHA"], V4["ALPHA"] / m])
        row = {"analysis": label, "feature": col, "n_fraud": len(a), "n_control": len(b),
               "fraud_median": np.median(a), "control_median": np.median(b), "raw_p": p,
               "rank_biserial_r": rank_biserial(a, b),
               "CI95_low": cis[V4["ALPHA"]][0], "CI95_high": cis[V4["ALPHA"]][1],
               f"CI{100 * (1 - V4['ALPHA'] / m):.0f}_bonf_low": cis[V4["ALPHA"] / m][0],
               f"CI{100 * (1 - V4['ALPHA'] / m):.0f}_bonf_high": cis[V4["ALPHA"] / m][1]}
        if matched_df is not None:
            row["matched_set_mean_rank"], row["matched_set_perm_p"], row["n_sets"] = matched_set_rank_test(matched_df, col, V4["N_PERM"])
        rows.append(row)
    t = pd.DataFrame(rows)
    t["fdr_bh_p"] = multipletests(t["raw_p"], alpha=V4["ALPHA"], method="fdr_bh")[1]
    t["sig_fdr"] = t["fdr_bh_p"] < V4["ALPHA"]
    bl, bh = [c for c in t.columns if c.endswith("_bonf_low")][0], [c for c in t.columns if c.endswith("_bonf_high")][0]
    t["adj_CI_excludes_0"] = (t[bl] > 0) | (t[bh] < 0)
    return t

cases_v4 = matched_v4[matched_v4.role == "case"]
ctrls_v4 = matched_v4[matched_v4.role == "control"]
stats_v4 = effect_size_table(cases_v4, ctrls_v4, TEST_FEATURES, matched_v4, "v4 matched, as-of cutoff")

ids_case, ids_ctrl = set(cases_v4.project_id), set(ctrls_v4.project_id)
stats_full = effect_size_table(feat_full[feat_full.project_id.isin(ids_case)],
                               feat_full[feat_full.project_id.isin(ids_ctrl)],
                               TEST_FEATURES + ["share_batches_retired"], None, "sensitivity: full history")
stats_all = pd.concat([stats_v4, stats_full], ignore_index=True)
pd.set_option("display.width", 250)
print(stats_all.round(4).to_string(index=False))
save_csv(stats_all, "effect_sizes_v4.csv")
print("\nCompare like with like: raw_p vs CI95, and fdr_bh_p vs the Bonferroni-width CI.")

In [ ]:
# ===== V4-8  ML evaluation: scaled models, uncertainty, calibration baseline, permutation, cluster hold-out =====
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import roc_auc_score, brier_score_loss

ML_FEATURES = ["n_batches", "share_credits_retired", "mean_issuance_lag_days", "batch_size_cv"]
LOG_FEATURES = ["n_batches", "mean_issuance_lag_days"]

def design_matrix(df, features):
    X = df[features].astype(float).copy()
    for c in LOG_FEATURES:
        if c in X:
            X[c] = np.log1p(X[c].clip(lower=0))
    return X.values

MODELS = {
    "LogReg (standardized, unweighted)": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                                               LogisticRegression(max_iter=5000)),
    "LogReg (standardized, class-balanced)": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                                                   LogisticRegression(max_iter=5000, class_weight="balanced")),
    "Random Forest": lambda: make_pipeline(SimpleImputer(strategy="median"),
                                           RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=GLOBAL_SEED)),
}
try:
    from xgboost import XGBClassifier
    MODELS["XGBoost (shallow)"] = lambda: XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.05,
                                                        eval_metric="logloss", random_state=GLOBAL_SEED)
except ImportError:
    pass

def loocv_predict(factory, X, y):
    p = np.zeros(len(y))
    for tr, te in LeaveOneOut().split(X):
        m = factory(); m.fit(X[tr], y[tr]); p[te] = m.predict_proba(X[te])[:, 1]
    return p

def auc_ci(y, p, n_boot=V4["N_BOOT"], seed=GLOBAL_SEED):
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    vals = [roc_auc_score(y[i], p[i]) for i in
            (np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]) for _ in range(n_boot))]
    return np.percentile(vals, [2.5, 97.5])

def cluster_holdout_auc(factory, df, X, y):
    """Train on one fraud scheme, test on the other."""
    res = {}
    for held in ["CQC", "Guizhou"]:
        te = (df["cluster"] == held).values
        if te.sum() == 0 or len(np.unique(y[te])) < 2 or len(np.unique(y[~te])) < 2:
            res[held] = np.nan; continue
        m = factory(); m.fit(X[~te], y[~te])
        res[held] = roc_auc_score(y[te], m.predict_proba(X[te])[:, 1])
    return res

def evaluate(df, features, label):
    df = df.reset_index(drop=True)
    X, y = design_matrix(df, features), df["is_fraud"].astype(int).values
    prev = y.mean(); brier_ref = prev * (1 - prev)
    rows, preds = [], {}
    for name, fac in MODELS.items():
        p = loocv_predict(fac, X, y); preds[name] = p
        lo, hi = auc_ci(y, p)
        b = brier_score_loss(y, p)
        ch = cluster_holdout_auc(fac, df, X, y)
        rows.append({"dataset": label, "model": name, "n": len(y), "n_fraud": int(y.sum()),
                     "LOOCV_AUC": roc_auc_score(y, p), "AUC_95CI_low": lo, "AUC_95CI_high": hi,
                     "Brier": b, "Brier_no_skill": brier_ref, "Brier_skill_score": 1 - b / brier_ref,
                     "mean_pred_prob": p.mean(), "prevalence": prev,
                     "AUC_holdout_CQC": ch["CQC"], "AUC_holdout_Guizhou": ch["Guizhou"]})
    return pd.DataFrame(rows), preds, X, y

ml_v4, preds_v4, X_v4, y_v4 = evaluate(matched_v4, ML_FEATURES, "v4 matched, as-of")
ml_tables = [ml_v4]
if "feat_all" in globals():
    v3 = feat_all.copy().rename(columns={"label": "is_fraud", "pct_retired": "share_credits_retired"})
    v3["cluster"] = v3["project_id"].astype(int).map(fraud_cluster)
    type_map = feat_full.set_index("project_id")["project_type"]
    v3.loc[v3.cluster.isna(), "cluster"] = v3.loc[v3.cluster.isna(), "project_id"].astype(int).map(type_map).map(
        lambda t: "CQC" if t == "Energy demand" else "Guizhou")
    ml_v3, _, _, _ = evaluate(v3, ML_FEATURES, "v3 sample, v4 protocol (NB: v3 retired = share of batches)")
    ml_tables.append(ml_v3)
ml_all = pd.concat(ml_tables, ignore_index=True)
print(ml_all.round(3).to_string(index=False))
save_csv(ml_all, "ml_evaluation_v4.csv")
print("\nBrier_skill_score <= 0 means no better than always predicting the base rate.")

fac = MODELS["LogReg (standardized, unweighted)"]
obs_auc = roc_auc_score(y_v4, preds_v4["LogReg (standardized, unweighted)"])
rng = np.random.default_rng(GLOBAL_SEED)
null_aucs = []
for i in range(V4["N_PERM"]):
    y_perm = rng.permutation(y_v4)
    null_aucs.append(roc_auc_score(y_perm, loocv_predict(fac, X_v4, y_perm)))
perm_p = (np.sum(np.array(null_aucs) >= obs_auc) + 1) / (V4["N_PERM"] + 1)
print(f"\nLOOCV AUC {obs_auc:.3f} | label-permutation p = {perm_p:.4f} ({V4['N_PERM']} permutations)")
save_csv(pd.DataFrame({"null_auc": null_aucs}), "auc_permutation_null_v4.csv")

## 7. Exposure, on-chain checks and inclusive-finance lens

In [ ]:
# ===== V4-11  Sustainability impact: how many tCO2e sit behind the candidate pool? =====
# (1 credit = 1 tCO2e claimed). This is EXPOSURE, not proof of over-crediting.
def find_col(df, *must):
    for c in df.columns:
        if all(m.lower() in str(c).lower().replace("\n", " ") for m in must):
            return c
    return None

c_iss, c_ret = find_col(df_harmonized, "total credits", "issued"), find_col(df_harmonized, "total credits", "retired")
c_type = find_col(df_harmonized, "type") or "Type"
h = df_harmonized.copy()
h["key"] = h["Project ID"].map(normalize_key).replace(crosswalk)
h[c_iss] = pd.to_numeric(h[c_iss], errors="coerce")
if c_ret:
    h[c_ret] = pd.to_numeric(h[c_ret], errors="coerce")
pool_keys = set(candidate_pool_v4.project_key)
fraud_keys = {f"VCS{i}" for i in fraud_cluster}
groups = {"All projects in panel": h, "Candidate pool (moved into risk)": h[h.key.isin(pool_keys)],
          "Confirmed fraud (10 projects)": h[h.key.isin(fraud_keys)]}
impact = pd.DataFrame([{"group": g, "n_projects": d.key.nunique(),
                        "credits_issued_MtCO2e": d[c_iss].sum() / 1e6,
                        "credits_retired_MtCO2e": d[c_ret].sum() / 1e6 if c_ret else np.nan}
                       for g, d in groups.items()])
impact["share_of_all_issued_%"] = 100 * impact.credits_issued_MtCO2e / impact.credits_issued_MtCO2e.iloc[0]
print(impact.round(3).to_string(index=False))
print("Retired credits = offset claims already made by buyers against these projects.")
save_csv(impact, "sustainability_exposure_v4.csv")
by_type = (h[h.key.isin(pool_keys)].groupby(c_type)[c_iss].agg(["count", "sum"])
           .rename(columns={"count": "n_projects", "sum": "credits_issued"}).sort_values("credits_issued", ascending=False))
save_csv(by_type, "candidate_pool_exposure_by_type_v4.csv", index=True)
print(by_type.head(10))

In [ ]:
# ===== V4-13  On-chain and Carbonmark retrieval protocol with a provenance log =====
# Set your key as an environment variable:  os.environ["ETHERSCAN_API_KEY"] = "..."  (never in the notebook)
import requests, hashlib

ONCHAIN = {"chain_id": 137, "network": "Polygon PoS", "endpoint": "https://api.etherscan.io/v2/api",
           "contracts": {"BCT_pool": "0x2F800Db0fdb5223b3C3f354886d907A671414A7F"},
           "page_size": 100, "startblock": 0, "endblock": 99999999}
provenance = []

def fetch_token_transfers(contract, api_key, page=1):
    params = {"chainid": ONCHAIN["chain_id"], "module": "account", "action": "tokentx", "contractaddress": contract,
              "startblock": ONCHAIN["startblock"], "endblock": ONCHAIN["endblock"], "page": page,
              "offset": ONCHAIN["page_size"], "sort": "desc", "apikey": api_key}
    r = requests.get(ONCHAIN["endpoint"], params=params, timeout=20)
    data = r.json() if r.ok else {}
    rows = pd.DataFrame(data.get("result", []) if isinstance(data.get("result"), list) else [])
    provenance.append({"source": "Etherscan v2", "contract": contract, "http_status": r.status_code,
                       "api_status": data.get("status"), "n_rows": len(rows),
                       "block_min": rows["blockNumber"].astype(int).min() if len(rows) else None,
                       "block_max": rows["blockNumber"].astype(int).max() if len(rows) else None,
                       "retrieved_utc": datetime.now(timezone.utc).isoformat(),
                       "sha256_of_payload": hashlib.sha256(r.content).hexdigest()})
    return rows

def carbonmark_check(registry, number):
    url = f"https://api.carbonmark.com/carbonProjects/{registry}-{number}"
    try:
        r = requests.get(url, headers={"Accept": "application/json"}, timeout=15)
        code = r.status_code
    except requests.RequestException as e:
        code = f"error: {e.__class__.__name__}"
    provenance.append({"source": "Carbonmark", "query": url, "http_status": code,
                       "tokenized_listing_found": code == 200, "retrieved_utc": datetime.now(timezone.utc).isoformat()})
    return code

key = os.environ.get("ETHERSCAN_API_KEY")
if key:
    bct = fetch_token_transfers(ONCHAIN["contracts"]["BCT_pool"], key)
    print(f"BCT transfers retrieved: {len(bct)}")
else:
    print("ETHERSCAN_API_KEY not set — skipping on-chain pull.")

carbonmark_check("VCS", 674)                 # positive control first
for pid in sorted(fraud_cluster):            # then ALL 10 confirmed-fraud projects
    carbonmark_check("VCS", pid)
prov = pd.DataFrame(provenance)
print(prov.drop(columns=[c for c in ["sha256_of_payload"] if c in prov], errors="ignore").to_string(index=False))
save_csv(prov, "onchain_provenance_log_v4.csv")
with open(out("onchain_protocol_v4.json"), "w") as fh:
    json.dump(ONCHAIN, fh, indent=2)

In [ ]:
# ===== V4-15  Inclusive-finance lens: WHO is behind the at-risk credits? =====
import requests, unicodedata, re

def _norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z]", "", s)

# 1) World Bank income groups (official API; fallback list only if offline — verify it before quoting)
ALIASES = {"turkiye": "turkiye", "vietnam": "vietnam", "laos": "laopdr", "drcongo": "congodemrep",
           "democraticrepublicofthecongo": "congodemrep", "republicofthecongo": "congorep",
           "congo": "congorep", "egypt": "egyptarabrep", "iran": "iranislamicrep", "southkorea": "korearep",
           "tanzania": "tanzania", "kyrgyzstan": "kyrgyzrepublic", "gambia": "gambiathe", "yemen": "yemenrep",
           "cotedivoire": "cotedivoire", "bolivia": "bolivia", "venezuela": "venezuelarb", "russia": "russianfederation",
           "slovakia": "slovakrepublic", "micronesia": "micronesiafedsts", "stlucia": "stlucia"}
income = {}
try:
    r = requests.get("https://api.worldbank.org/v2/country", params={"format": "json", "per_page": 400}, timeout=20)
    for c in r.json()[1]:
        if c["region"]["value"] != "Aggregates":
            income[_norm(c["name"])] = c["incomeLevel"]["value"]
    INCOME_SOURCE = "World Bank API (retrieved " + datetime.now(timezone.utc).date().isoformat() + ")"
except Exception as e:
    INCOME_SOURCE = f"OFFLINE — World Bank API unavailable ({e.__class__.__name__}); income groups not assigned"
print("Income-group source:", INCOME_SOURCE)

def income_group(country):
    k = _norm(country)
    k = ALIASES.get(k, k)
    return income.get(k, "Unmatched" if income else "Unknown")

# 2) Build the project table (same key logic as V4-11)
c_iss = find_col(df_harmonized, "total credits", "issued")
c_ret = find_col(df_harmonized, "total credits", "retired")
hh = df_harmonized.copy()
hh["key"] = hh["Project ID"].map(normalize_key).replace(crosswalk)
hh[c_iss] = pd.to_numeric(hh[c_iss], errors="coerce").fillna(0)
hh[c_ret] = pd.to_numeric(hh[c_ret], errors="coerce").fillna(0)
hh["income_group"] = hh["Country"].map(income_group)
hh["low_or_lower_middle"] = hh["income_group"].isin(["Low income", "Lower middle income"])
hh["household_community"] = hh["Scope"].eq("Household & Community")

unmatched = sorted(hh.loc[hh.income_group == "Unmatched", "Country"].dropna().unique())
if unmatched:
    print(f"Countries not matched to a WB income group ({len(unmatched)}) — add to ALIASES:", unmatched[:25])

def lens(d, label):
    iss = d[c_iss].sum()
    return {"group": label, "n_projects": d.key.nunique(), "n_countries": d["Country"].nunique(),
            "issued_MtCO2e": iss / 1e6, "retired_MtCO2e": d[c_ret].sum() / 1e6,
            "%_issued_household_community": 100 * d.loc[d.household_community, c_iss].sum() / iss if iss else np.nan,
            "%_issued_low_or_lower_middle_income": (100 * d.loc[d.low_or_lower_middle, c_iss].sum() / iss
                                                     if iss and income else np.nan)}   # NaN if WB API was offline

pool_keys = set(candidate_pool_v4.project_key)
fraud_keys = {f"VCS{i}" for i in fraud_cluster}
inclusive = pd.DataFrame([lens(hh, "All projects"),
                          lens(hh[hh.key.isin(pool_keys)], "Candidate pool (113)"),
                          lens(hh[hh.key.isin(fraud_keys)], "Confirmed fraud (7 CQC + 4 Guizhou)")])
print(inclusive.round(2).to_string(index=False))
save_csv(inclusive, "inclusive_finance_lens_v4.csv")

by_country = (hh[hh.key.isin(pool_keys)].groupby(["Country", "income_group"])
              .agg(n_projects=("key", "nunique"), issued=(c_iss, "sum"), retired=(c_ret, "sum"))
              .sort_values("issued", ascending=False).reset_index())
print("\nCandidate pool by country:"); print(by_country.head(12).to_string(index=False))
save_csv(by_country, "candidate_pool_by_country_v4.csv")

# 3) Figure: where at-risk credits come from vs the whole market
fig, ax = plt.subplots(figsize=(6, 2.6))
lab = ["Household & Community scope", "Low / lower-middle income countries"]
x = np.arange(2); w = 0.38
ax.bar(x - w/2, inclusive.iloc[0, 5:7].astype(float), w, color=CB["grey"], label="All projects")
ax.bar(x + w/2, inclusive.iloc[1, 5:7].astype(float), w, color=CB["vermillion"], hatch="//", label="Candidate pool (113)")
ax.set_xticks(x); ax.set_xticklabels(lab); ax.set_ylabel("% of credits issued"); ax.set_ylim(0, 100)
ax.legend(frameon=False, fontsize=8); ax.set_title("Who stands behind at-risk carbon credits?")
plt.tight_layout(); plt.savefig(out("fig_inclusive_finance_v4.png"), dpi=300); plt.show()
print("Income-group source for the report footnote:", INCOME_SOURCE)

## 8. Early warning, sensitivity, triage and fairness

In [ ]:
# ===== V5-1  Early warning: lead time, predictive value of the watch state, temporal hold-out =====
import time, hashlib
from statsmodels.stats.proportion import proportion_confint
V5_T0 = time.time(); V5_TIMES = {}
V5 = {"OUT_DIR": "v5_outputs", "N_BOOT": 1000, "CV_REPEATS": 5, "SEED": GLOBAL_SEED}
os.makedirs(V5["OUT_DIR"], exist_ok=True)
def out5(f): return os.path.join(V5["OUT_DIR"], f)
def save5(df, f, index=False):
    df.to_csv(out5(f), index=index); print(f"  -> saved {out5(f)}")
def wilson(k, n):
    if n == 0: return (np.nan, np.nan)
    lo, hi = proportion_confint(k, n, method="wilson"); return (lo, hi)
def tick(name): V5_TIMES[name] = round(time.time() - V5_T0, 1)

SNAP_DATE = {"2024-12": pd.Timestamp("2024-12-31"), "2025-12": pd.Timestamp("2025-12-31"),
             "2026-02": pd.Timestamp("2026-02-28"), "2026-04": pd.Timestamp("2026-04-30")}

# (a) Next-interval risk entry by origin state, per adjacent pair
ew = []
for a, b in zip(SNAPS[:-1], SNAPS[1:]):
    both = tiers_v4[a].notna() & tiers_v4[b].notna()
    for origin in ["normal", "watch"]:
        o = both & (tiers_v4[a] == origin)
        n, k = int(o.sum()), int((o & (tiers_v4[b] == "risk_flag")).sum())
        lo, hi = wilson(k, n)
        ew.append({"interval": f"{a}->{b}", "months": round((SNAP_DATE[b] - SNAP_DATE[a]).days / 30.4),
                   "origin_state": origin, "n_projects": n, "entered_risk": k, "rate_%": 100 * k / n if n else np.nan,
                   "CI95_low_%": 100 * lo, "CI95_high_%": 100 * hi})
ew = pd.DataFrame(ew)
print("Next-interval risk entry by origin state:"); print(ew.round(3).to_string(index=False))
save5(ew, "early_warning_by_interval.csv")

# (b) Eventual entry (by 2026-04) for projects observed at the first snapshot
base = tiers_v4["2024-12"]
later_risk = (tiers_v4[SNAPS[1:]] == "risk_flag").any(axis=1)
rr_rows = []
for origin in ["normal", "watch"]:
    o = base == origin
    n, k = int(o.sum()), int((o & later_risk).sum()); lo, hi = wilson(k, n)
    rr_rows.append({"state_Dec2024": origin, "n": n, "entered_risk_by_Apr2026": k, "rate_%": 100 * k / n,
                    "CI95_low_%": 100 * lo, "CI95_high_%": 100 * hi})
rr = pd.DataFrame(rr_rows)
p_w, p_n = rr.loc[1, "entered_risk_by_Apr2026"] / rr.loc[1, "n"], rr.loc[0, "entered_risk_by_Apr2026"] / rr.loc[0, "n"]
if p_w > 0 and p_n > 0:
    se = np.sqrt((1 - p_w) / rr.loc[1, "entered_risk_by_Apr2026"] + (1 - p_n) / rr.loc[0, "entered_risk_by_Apr2026"])
    RR_WATCH = (p_w / p_n, np.exp(np.log(p_w / p_n) - 1.96 * se), np.exp(np.log(p_w / p_n) + 1.96 * se))
else:
    RR_WATCH = (np.nan, np.nan, np.nan)
print(rr.round(3).to_string(index=False))
print(f"Relative risk (watch vs normal, Dec 2024 -> Apr 2026): {RR_WATCH[0]:.2f} [{RR_WATCH[1]:.2f}, {RR_WATCH[2]:.2f}]")
save5(rr, "early_warning_eventual.csv")

# (c) Lead time: did the 113 candidates pass through 'watch' before 'risk_flag'?
lead = []
for idx in panel_v4.index[panel_v4["moved_into_risk"]]:
    seq = [(s, tiers_v4.at[idx, s]) for s in SNAPS if pd.notna(tiers_v4.at[idx, s])]
    first_risk = next(s for s, t in seq if t == "risk_flag" and any(t2 in ("normal", "watch") for s2, t2 in seq if s2 < s))
    watch_before = [s for s, t in seq if t == "watch" and s < first_risk]
    lead.append({"project_key": panel_v4.at[idx, "project_key"], "registry": panel_v4.at[idx, "registry"],
                 "first_risk_snapshot": first_risk, "watch_before": bool(watch_before),
                 "lead_months": round((SNAP_DATE[first_risk] - SNAP_DATE[watch_before[0]]).days / 30.4) if watch_before else 0,
                 "confirmed_fraud": panel_v4.at[idx, "project_key"] in {f"VCS{i}" for i in fraud_cluster}})
lead = pd.DataFrame(lead)
print(f"\nCandidates with a watch state BEFORE risk: {lead.watch_before.sum()} of {len(lead)} "
      f"({100 * lead.watch_before.mean():.1f}%); median lead when present: "
      f"{lead.loc[lead.watch_before, 'lead_months'].median() if lead.watch_before.any() else 0} months")
print(lead[lead.confirmed_fraud][["project_key", "first_risk_snapshot", "watch_before", "lead_months"]].to_string(index=False))
save5(lead, "lead_time_candidates.csv")

# (d) Temporal hold-out of the simple rule "watch now -> risk next": learn on interval 1, test on intervals 2-3
test = ew[ew.interval != f"{SNAPS[0]}->{SNAPS[1]}"]
tp = test.loc[test.origin_state == "watch", "entered_risk"].sum(); fn = test.loc[test.origin_state == "normal", "entered_risk"].sum()
flagged = test.loc[test.origin_state == "watch", "n_projects"].sum()
print(f"\nHold-out (2025-12 -> 2026-04): 'watch' rule recall = {tp}/{tp + fn} risk entries; precision = {tp}/{flagged}")
EW_SUMMARY = {"watch_recall_holdout": (int(tp), int(tp + fn)), "watch_precision_holdout": (int(tp), int(flagged)),
              "RR_watch": RR_WATCH, "share_with_watch_before": float(lead.watch_before.mean())}
tick("V5-1")

In [ ]:
# ===== V5-2  Sensitivity of the candidate pool to the status ontology =====
def pool_under(mapping):
    t = panel_v4[SNAPS].apply(lambda c: c.map(mapping))
    return set(panel_v4.loc[t.apply(moved_into_risk_gap_skipping, axis=1), "project_key"])

variants = {"Baseline (published ontology)": dict(STATUS_ONTOLOGY)}
v = dict(STATUS_ONTOLOGY); v["On Hold - see notification letter"] = "risk_flag"; variants["On Hold counted as risk"] = v
v = dict(STATUS_ONTOLOGY); v["Late to verify"] = "normal"; variants["Late to verify counted as normal"] = v
v = dict(STATUS_ONTOLOGY); v["Inactive"] = "watch"; variants["Inactive counted as watch"] = v
v = dict(STATUS_ONTOLOGY); v["Withdrawn"] = "watch"; variants["Withdrawn counted as watch"] = v
v = {k: ("risk_flag" if any(w in k.lower() for w in ["reject", "denied", "cancel", "unsuccessful"]) else
         ("watch" if val != "normal" else "normal")) for k, val in STATUS_ONTOLOGY.items()}
variants["Strict: only registry rejections/denials/cancellations"] = v

base_pool = pool_under(variants["Baseline (published ontology)"])
cqc_keys = {f"VCS{i}" for i, c in fraud_cluster.items() if c == "CQC"}
gz_keys = {f"VCS{i}" for i, c in fraud_cluster.items() if c == "Guizhou"}
iss_by_key = hh.groupby("key")[c_iss].sum()
onto = []
for name, mp in variants.items():
    p = pool_under(mp)
    onto.append({"variant": name, "pool_size": len(p), "jaccard_vs_baseline": len(p & base_pool) / len(p | base_pool),
                 "CQC_recovered": f"{len(p & cqc_keys)}/{len(cqc_keys)}", "Guizhou_recovered": f"{len(p & gz_keys)}/{len(gz_keys)}",
                 "issued_MtCO2e": iss_by_key.reindex(list(p)).sum() / 1e6})
onto = pd.DataFrame(onto)
print(onto.round(3).to_string(index=False)); save5(onto, "ontology_sensitivity.csv")
tick("V5-2")

In [ ]:
# ===== V5-3  Matching sensitivity + full balance table (Love plot) =====
def match_custom(feat, k=5, caliper_sd=0.5, exclude_linked=True, seed=GLOBAL_SEED):
    f = feat[feat["total_issued"] > 0].copy(); f["log_size"] = np.log10(f["total_issued"])
    cases = f[f.is_fraud == 1].sort_values("log_size", ascending=False)
    elig = eligible_ids if exclude_linked else (eligible_ids | linked_ids)
    pool = f[(f.is_fraud == 0) & f.project_id.isin(elig)]
    cal = caliper_sd * pd.concat([cases.log_size, pool.log_size]).std() if caliper_sd else np.inf
    used, rows = set(), []
    for _, c in cases.iterrows():
        cand = pool[(pool.project_type == c.project_type) & ~pool.project_id.isin(used)].copy()
        cand["dist"] = (cand.log_size - c.log_size).abs()
        ch = cand[cand.dist <= cal].nsmallest(k, "dist"); used |= set(ch.project_id)
        rows.append(c.to_frame().T.assign(match_set=c.project_id, role="case"))
        rows.append(ch.drop(columns="dist").assign(match_set=c.project_id, role="control", cluster=c.cluster))
    m = pd.concat(rows, ignore_index=True)
    for col in ["log_size", "n_batches", "total_issued", "share_credits_retired", "mean_issuance_lag_days", "batch_size_cv", "is_fraud"]:
        m[col] = pd.to_numeric(m[col], errors="coerce")
    return m, pool

def rf_loocv_auc(m):
    X, y = design_matrix(m.reset_index(drop=True), ML_FEATURES), m["is_fraud"].astype(int).values
    fac = lambda: make_pipeline(SimpleImputer(strategy="median"),
                                RandomForestClassifier(n_estimators=200, min_samples_leaf=2, random_state=GLOBAL_SEED, n_jobs=-1))
    return roc_auc_score(y, loocv_predict(fac, X, y))

specs = [("Primary: k=5, caliper 0.5 SD", 5, 0.5, True), ("Tighter caliper 0.25 SD", 5, 0.25, True),
         ("Wider caliper 1.0 SD", 5, 1.0, True), ("k=3", 3, 0.5, True), ("k=10", 10, 0.5, True),
         ("Developer-linked projects NOT excluded", 5, 0.5, False)]
msens = []
for name, k, cal, excl in specs:
    m, _ = match_custom(feat_asof, k, cal, excl)
    ca, co = m[m.role == "case"], m[m.role == "control"]
    t = effect_size_table(ca, co, TEST_FEATURES, None, name)
    row = {"specification": name, "n_controls": len(co), "size_SMD": smd(ca.log_size, co.log_size),
           "RF_LOOCV_AUC": rf_loocv_auc(m), "min_FDR_q": t.fdr_bh_p.min()}
    for _, r in t.iterrows():
        row[f"r_{r.feature}"] = r.rank_biserial_r
    msens.append(row)
msens = pd.DataFrame(msens)
print(msens.round(3).to_string(index=False)); save5(msens, "matching_sensitivity.csv")

# Full balance table: every feature, before (vs eligible pool) and after matching
m_primary, pool_primary = match_custom(feat_asof)
cases_p = m_primary[m_primary.role == "case"]; ctrl_p = m_primary[m_primary.role == "control"]
pool_primary = pool_primary.copy()
love = []
for col, lab in [("log_size", "log10 credits issued"), ("n_batches", "Issuance batches"),
                 ("share_credits_retired", "Share of credits retired"), ("mean_issuance_lag_days", "Mean issuance lag"),
                 ("batch_size_cv", "Batch-size CV")]:
    love.append({"variable": lab, "SMD_before": smd(cases_p[col], pool_primary[col]), "SMD_after": smd(cases_p[col], ctrl_p[col]),
                 "matching_variable": col == "log_size"})
for t_ in cases_p.project_type.dropna().unique():
    love.append({"variable": f"type = {t_}", "SMD_before": smd(cases_p.project_type == t_, pool_primary.project_type == t_),
                 "SMD_after": smd(cases_p.project_type == t_, ctrl_p.project_type == t_), "matching_variable": True})
love = pd.DataFrame(love); print(love.round(3).to_string(index=False)); save5(love, "balance_full_love.csv")
tick("V5-3")

In [ ]:
# ===== V5-4  Claim-consistency agent: issuance vs the project's OWN claimed annual reductions =====
# Over-issuance (the C-Quest mechanism) means issuing more than the project itself claimed it would reduce.
vp24 = raw_data["2024-12"]["Verra Projects"].copy()
vp24["ID_num"] = pd.to_numeric(vp24["ID"], errors="coerce")
vp24["EAER"] = pd.to_numeric(vp24["Estimated Annual Emission Reductions"], errors="coerce")
tx = df_vcus.copy(); tx["ID"] = pd.to_numeric(tx["ID"], errors="coerce")
for c in ["Issuance Date", "Vintage Start", "Vintage End"]:
    tx[c] = pd.to_datetime(tx[c], errors="coerce")
tx["Quantity Issued"] = pd.to_numeric(tx["Quantity Issued"], errors="coerce").fillna(0)
tx = tx[tx["Issuance Date"] <= CUTOFF]
g = tx.groupby("ID").agg(issued=("Quantity Issued", "sum"), v_start=("Vintage Start", "min"), v_end=("Vintage End", "max"))
g["vintage_years"] = ((g.v_end - g.v_start).dt.days / 365.25).clip(lower=0.25)
claims = g.join(vp24.drop_duplicates("ID_num").set_index("ID_num")[["EAER"]], how="left")
claims["issuance_to_claim_ratio"] = claims["issued"] / (claims["EAER"] * claims["vintage_years"])
claims.loc[~np.isfinite(claims["issuance_to_claim_ratio"]), "issuance_to_claim_ratio"] = np.nan
claims.index.name = "project_id"; claims = claims.reset_index()
claim_map = claims.set_index("project_id")["issuance_to_claim_ratio"]

for fdf in (feat_asof, matched_v4):
    fdf["issuance_to_claim_ratio"] = pd.to_numeric(fdf["project_id"], errors="coerce").map(claim_map)
ca_, co_ = matched_v4[matched_v4.role == "case"], matched_v4[matched_v4.role == "control"]
claim_test = effect_size_table(ca_, co_, ["issuance_to_claim_ratio"], matched_v4, "claim consistency (as of Dec 2024)")
print(claim_test[["feature", "n_fraud", "n_control", "fraud_median", "control_median", "rank_biserial_r",
                  "CI95_low", "CI95_high", "raw_p", "matched_set_mean_rank", "matched_set_perm_p"]].round(3).to_string(index=False))
print("Confirmed cases:"); print(ca_[["project_id", "cluster", "issuance_to_claim_ratio"]].round(2).to_string(index=False))
pop_ratio = claims["issuance_to_claim_ratio"].dropna()
print(f"Population (VCS, n={len(pop_ratio)}): median {pop_ratio.median():.2f}; share > 1.0 = {100 * (pop_ratio > 1).mean():.1f}%")
save5(claims, "claim_consistency_all_vcs.csv"); save5(claim_test, "claim_consistency_test.csv")
tick("V5-4")

In [ ]:
# ===== V5-5  PROSPECTIVE early-warning model: information at Dec 2024 -> entered risk by Apr 2026? =====
# Much larger sample than the 10 confirmed cases; outcome = registry risk transition (surrogate for integrity events).
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
vcs_panel = panel_v4[panel_v4.registry == "VCS"].copy()
vcs_panel["project_id"] = pd.to_numeric(vcs_panel.project_key.str.replace("VCS", "", regex=False), errors="coerce")
vcs_panel["tier_2024"] = tiers_v4.loc[vcs_panel.index, "2024-12"]
pro = vcs_panel[vcs_panel.tier_2024.isin(["normal", "watch"])][["project_id", "project_key", "tier_2024", "moved_into_risk"]]
pro = pro.merge(feat_asof, on="project_id", how="inner")
pro = pro[~pro.project_id.isin(linked_ids)]                          # developer-linked projects excluded (label noise)
pro["watch_2024"] = (pro.tier_2024 == "watch").astype(int)
pro["log_size"] = np.log10(pro.total_issued.clip(lower=1))
pro["afolu"] = pro.project_type.astype(str).str.contains("Forestry|Land Use", case=False).astype(int)
pro["energy_demand"] = pro.project_type.astype(str).str.contains("Energy demand", case=False).astype(int)
pro["issuance_to_claim_ratio"] = pro.project_id.map(claim_map)
pro["y"] = pro.moved_into_risk.astype(int)
PRO_FEATURES = ["watch_2024", "log_size", "n_batches", "share_credits_retired", "mean_issuance_lag_days",
                "batch_size_cv", "issuance_to_claim_ratio", "afolu", "energy_demand"]
print(f"Prospective cohort: {len(pro)} VCS projects in normal/watch at Dec 2024 with issuance; "
      f"{pro.y.sum()} entered risk by Apr 2026 ({100 * pro.y.mean():.2f}%)")

def Xpro(df, feats):
    X = df[feats].astype(float).copy()
    for c in ["n_batches", "mean_issuance_lag_days"]:
        if c in X: X[c] = np.log1p(X[c].clip(lower=0))
    if "issuance_to_claim_ratio" in X: X["issuance_to_claim_ratio"] = np.log1p(X["issuance_to_claim_ratio"].clip(lower=0))
    return X.values

PRO_MODELS = {
    "Registry only (watch state)": (["watch_2024"], lambda: make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=2000))),
    "Logistic (all agents' features)": (PRO_FEATURES, lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                                                           LogisticRegression(max_iter=5000, C=0.5))),
    "Random Forest (all agents' features)": (PRO_FEATURES, lambda: make_pipeline(SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=GLOBAL_SEED, n_jobs=-1))),
    "Random Forest (without claim-consistency)": ([f for f in PRO_FEATURES if f != "issuance_to_claim_ratio"],
        lambda: make_pipeline(SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=GLOBAL_SEED, n_jobs=-1))),
}
pro_res, pro_oof = [], {}
yv = pro.y.values
for name, (feats, fac) in PRO_MODELS.items():
    X = Xpro(pro, feats); oof_sum = np.zeros(len(yv)); aucs, aps = [], []
    for rep in range(V5["CV_REPEATS"]):
        oof = np.zeros(len(yv))
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=GLOBAL_SEED + rep).split(X, yv):
            mdl = fac().fit(X[tr], yv[tr]); oof[te] = mdl.predict_proba(X[te])[:, 1]
        aucs.append(roc_auc_score(yv, oof)); aps.append(average_precision_score(yv, oof)); oof_sum += oof
    oof_mean = oof_sum / V5["CV_REPEATS"]; pro_oof[name] = oof_mean
    lo, hi = auc_ci(yv, oof_mean, n_boot=V5["N_BOOT"])
    conf = pro.project_id.isin(fraud_cluster).values
    pct = pd.Series(oof_mean).rank(pct=True).values
    pro_res.append({"model": name, "n": len(yv), "events": int(yv.sum()), "AUC_mean_of_repeats": np.mean(aucs),
                    "AUC_95CI_low": lo, "AUC_95CI_high": hi, "avg_precision": np.mean(aps), "AP_no_skill": yv.mean(),
                    "confirmed_fraud_in_cohort": int(conf.sum()),
                    "confirmed_median_percentile": 100 * np.median(pct[conf]) if conf.any() else np.nan,
                    "confirmed_in_top10pct": int((pct[conf] >= 0.9).sum()) if conf.any() else 0})
pro_res = pd.DataFrame(pro_res); print(pro_res.round(3).to_string(index=False)); save5(pro_res, "prospective_early_warning.csv")
pro["oof_rf"] = pro_oof["Random Forest (all agents' features)"]
save5(pro[["project_id", "y", "watch_2024", "issuance_to_claim_ratio", "oof_rf"]], "prospective_scores.csv")
rf_full = PRO_MODELS["Random Forest (all agents' features)"][1]().fit(Xpro(pro, PRO_FEATURES), yv)
perm_imp = []
from sklearn.inspection import permutation_importance
pi = permutation_importance(rf_full, Xpro(pro, PRO_FEATURES), yv, scoring="roc_auc", n_repeats=10, random_state=GLOBAL_SEED, n_jobs=-1)
pro_imp = pd.DataFrame({"feature": PRO_FEATURES, "perm_importance_AUC": pi.importances_mean, "sd": pi.importances_std}
                       ).sort_values("perm_importance_AUC", ascending=False)
print(pro_imp.round(4).to_string(index=False)); save5(pro_imp, "prospective_feature_importance.csv")
tick("V5-5")

In [ ]:
# ===== V5-5b  Leakage check for V5-5: developer-grouped CV and performance on NON-confirmed risk entrants =====
from sklearn.model_selection import StratifiedGroupKFold
pro_dev = pro.project_id.map(vp24.drop_duplicates("ID_num").set_index("ID_num")["Proponent"]).fillna("unk_" + pro.project_id.astype(str))
grp = []
conf_mask = pro.project_id.isin(fraud_cluster).values
for name in ["Logistic (all agents' features)", "Random Forest (all agents' features)"]:
    feats, fac = PRO_MODELS[name]; X = Xpro(pro, feats); oofs = []
    for rep in range(V5["CV_REPEATS"]):
        oof = np.zeros(len(yv))
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=GLOBAL_SEED + rep).split(X, yv, pro_dev):
            oof[te] = fac().fit(X[tr], yv[tr]).predict_proba(X[te])[:, 1]
        oofs.append(oof)
    o = np.mean(oofs, 0); lo, hi = auc_ci(yv, o, n_boot=V5["N_BOOT"])
    nc = ~conf_mask; lo2, hi2 = auc_ci(yv[nc], o[nc], n_boot=V5["N_BOOT"])
    pct = pd.Series(o).rank(pct=True).values
    grp.append({"model": name, "grouped_CV_AUC": roc_auc_score(yv, o), "CI95_low": lo, "CI95_high": hi,
                "AUC_non_confirmed_events_only": roc_auc_score(yv[nc], o[nc]), "CI95_low_nc": lo2, "CI95_high_nc": hi2,
                "n_non_confirmed_events": int(yv[nc].sum()), "confirmed_in_top10pct": int((pct[conf_mask] >= 0.9).sum()),
                "avg_precision": average_precision_score(yv, o)})
grp = pd.DataFrame(grp); print(grp.round(3).to_string(index=False)); save5(grp, "prospective_grouped_cv.csv")
print("Interpretation: grouped CV keeps sibling projects of one developer out of each other's training folds.")
tick("V5-5b")

In [ ]:
# ===== V5-6  External check: train on registry outcomes (surrogate labels), test on confirmed fraud =====
test_ids = set(pd.to_numeric(matched_v4.project_id, errors="coerce").dropna().astype(int))
train = pro[~pro.project_id.isin(test_ids) & ~pro.project_id.isin(fraud_cluster)]
ext = []
for name in ["Logistic (all agents' features)", "Random Forest (all agents' features)"]:
    feats, fac = PRO_MODELS[name]
    mdl = fac().fit(Xpro(train, feats), train.y.values)
    tst = matched_v4.copy(); tst["project_id"] = pd.to_numeric(tst.project_id, errors="coerce")
    tst = tst.merge(pro[["project_id", "watch_2024", "afolu", "energy_demand"]], on="project_id", how="left")
    tst["watch_2024"] = tst["watch_2024"].fillna((tst.cluster == "Guizhou") & (tst.role == "case")).astype(float)
    tst["afolu"] = tst.project_type.astype(str).str.contains("Forestry|Land Use", case=False).astype(int)
    tst["energy_demand"] = tst.project_type.astype(str).str.contains("Energy demand", case=False).astype(int)
    tst["log_size"] = np.log10(tst.total_issued.clip(lower=1))
    p = mdl.predict_proba(Xpro(tst, feats))[:, 1]; yt = tst.is_fraud.astype(int).values
    lo, hi = auc_ci(yt, p, n_boot=V5["N_BOOT"])
    p_nw = mdl.predict_proba(Xpro(tst.assign(watch_2024=0.0), feats))[:, 1]
    ext.append({"model_trained_on_registry_outcomes": name, "train_n": len(train), "train_events": int(train.y.sum()),
                "test": "10 confirmed fraud vs 48 matched controls", "AUC": roc_auc_score(yt, p), "CI95_low": lo, "CI95_high": hi,
                "AUC_without_watch_signal": roc_auc_score(yt, p_nw)})
ext = pd.DataFrame(ext); print(ext.round(3).to_string(index=False)); save5(ext, "external_surrogate_validation.csv")
tick("V5-6")

In [ ]:
# ===== V5-7  Developer-cluster bootstrap of the matched-sample AUCs =====
prop_map = vp24.drop_duplicates("ID_num").set_index("ID_num")["Proponent"]
mm = matched_v4.reset_index(drop=True).copy()
mm["developer"] = pd.to_numeric(mm.project_id, errors="coerce").map(prop_map).fillna("unknown_" + mm.project_id.astype(str))
yy = mm.is_fraud.astype(int).values; groups = mm.developer.values; ug = np.unique(groups)
rng = np.random.default_rng(GLOBAL_SEED); clus = []
for name, p in preds_v4.items():
    vals = []
    for _ in range(V5["N_BOOT"]):
        pick = rng.choice(ug, len(ug), replace=True)
        idx = np.concatenate([np.where(groups == gname)[0] for gname in pick])
        if len(np.unique(yy[idx])) == 2: vals.append(roc_auc_score(yy[idx], p[idx]))
    clus.append({"model": name, "AUC": roc_auc_score(yy, p), "developer_cluster_CI95_low": np.percentile(vals, 2.5),
                 "developer_cluster_CI95_high": np.percentile(vals, 97.5), "n_developers": len(ug)})
clus = pd.DataFrame(clus); print(clus.round(3).to_string(index=False)); save5(clus, "cluster_bootstrap_auc.csv")
tick("V5-7")

In [ ]:
# ===== V5-8  Decision utility: precision-recall, review workload, expected cost =====
from sklearn.metrics import precision_recall_curve
dec = []
for name, p in preds_v4.items():
    ap = average_precision_score(y_v4, p); rng = np.random.default_rng(GLOBAL_SEED); bs = []
    pos, neg = np.where(y_v4 == 1)[0], np.where(y_v4 == 0)[0]
    for _ in range(V5["N_BOOT"]):
        i = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]); bs.append(average_precision_score(y_v4[i], p[i]))
    order = np.argsort(-p)
    row = {"model": name, "avg_precision": ap, "AP_CI95_low": np.percentile(bs, 2.5), "AP_CI95_high": np.percentile(bs, 97.5),
           "no_skill_AP": y_v4.mean()}
    for top in [5, 10, 15]:
        row[f"recall_at_top{top}"] = y_v4[order[:top]].sum() / y_v4.sum()
        row[f"precision_at_top{top}"] = y_v4[order[:top]].mean()
    dec.append(row)
dec = pd.DataFrame(dec); print(dec.round(3).to_string(index=False)); save5(dec, "decision_utility_matched.csv")

# Registry-surveillance as a screen over the whole market
N_all = int((panel_v4.n_snapshots_observed >= 2).sum()); n_pool = int(panel_v4.moved_into_risk.sum())
conf_in_pool = len(set(candidate_pool_v4.project_key) & {f"VCS{i}" for i in fraud_cluster})
rec_lo, rec_hi = wilson(conf_in_pool, len(fraud_cluster))
screen = pd.DataFrame([{"projects_monitored": N_all, "flagged_for_review": n_pool, "review_share_%": 100 * n_pool / N_all,
                        "confirmed_cases_recovered": f"{conf_in_pool}/{len(fraud_cluster)}",
                        "recall_CI95": f"{100 * rec_lo:.0f}-{100 * rec_hi:.0f}%",
                        "precision_lower_bound_%": 100 * conf_in_pool / n_pool,
                        "lift_vs_random_review": (conf_in_pool / n_pool) / (len(fraud_cluster) / N_all)}])
print(screen.round(2).to_string(index=False)); save5(screen, "surveillance_screen_utility.csv")

# Expected-cost scenarios per 10,000 projects: review cost = 1 unit; a missed problematic project costs R units
cost = []
for prev in [0.005, 0.01, 0.02]:
    for R in [10, 50, 200]:
        for strat, review_share, recall in [("Review nothing", 0, 0), ("Review everything", 1, 1),
                                            ("Review surveillance pool only", n_pool / N_all, rec_lo)]:
            n_bad = 10000 * prev
            cost.append({"prevalence_%": 100 * prev, "miss_cost_R": R, "strategy": strat,
                         "reviews": 10000 * review_share, "missed": n_bad * (1 - recall),
                         "expected_cost_units": 10000 * review_share + R * n_bad * (1 - recall)})
cost = pd.DataFrame(cost); save5(cost, "expected_cost_scenarios.csv")
print(cost.pivot_table(index=["prevalence_%", "miss_cost_R"], columns="strategy", values="expected_cost_units").round(0))
print("NOTE: surveillance recall uses the conservative Wilson lower bound; recall is measured only on registry-confirmed cases.")
tick("V5-8")

In [ ]:
# ===== V5-9  Evidence-based agentic arbiter (reason codes, contradictions, human review) + ablation =====
from dataclasses import dataclass, field, asdict
@dataclass
class Evidence:
    agent: str; signal: str; direction: int; strength: float; note: str   # direction: +1 risk, -1 reassuring, 0 missing

def registry_agent(tier_now, moved, ever_watch, status_now):
    ev = []
    if tier_now is None or (isinstance(tier_now, float) and np.isnan(tier_now)):
        return [Evidence("Registry", "no_current_status", 0, 0, "No status in latest snapshot")]
    adverse = isinstance(status_now, str) and any(w in status_now.lower() for w in ["reject", "denied", "cancel", "unsuccessful"])
    if tier_now == "risk_flag":
        ev.append(Evidence("Registry", "adverse_registry_decision" if adverse else "risk_status", 1, 1.0 if adverse else 0.8, str(status_now)))
    elif tier_now == "watch":
        ev.append(Evidence("Registry", "watch_status", 1, 0.5, str(status_now)))
    else:
        ev.append(Evidence("Registry", "normal_status", -1, 0.4, str(status_now)))
    if moved: ev.append(Evidence("Registry", "transition_into_risk", 1, 0.6, "normal/watch -> risk_flag"))
    if ever_watch and tier_now == "normal": ev.append(Evidence("Registry", "past_watch", 1, 0.3, "was on watch earlier"))
    return ev

def transaction_agent(score_pct):
    if score_pct is None or np.isnan(score_pct): return [Evidence("Transaction", "no_issuance_data", 0, 0, "no VCU records")]
    if score_pct >= 0.95: return [Evidence("Transaction", "anomalous_issuance_pattern", 1, 0.5, f"model percentile {score_pct:.2f}")]
    if score_pct <= 0.50: return [Evidence("Transaction", "typical_issuance_pattern", -1, 0.2, f"model percentile {score_pct:.2f}")]
    return [Evidence("Transaction", "unremarkable", 0, 0.1, f"model percentile {score_pct:.2f}")]

def claim_agent(ratio):
    if ratio is None or np.isnan(ratio): return [Evidence("ClaimConsistency", "no_claim_data", 0, 0, "no EAER")]
    if ratio >= 2.0: return [Evidence("ClaimConsistency", "issued_far_above_own_claim", 1, 0.7, f"ratio {ratio:.2f}")]
    if ratio >= 1.2: return [Evidence("ClaimConsistency", "issued_above_own_claim", 1, 0.4, f"ratio {ratio:.2f}")]
    return [Evidence("ClaimConsistency", "within_own_claim", -1, 0.2, f"ratio {ratio:.2f}")]

def onchain_agent(tokenized):
    if tokenized is None: return [Evidence("OnChain", "not_checked", 0, 0, "Carbonmark not queried")]
    return [Evidence("OnChain", "tokenized" if tokenized else "not_tokenized", 0, 0.1,
                     "tokenized exposure (informational)" if tokenized else "no on-chain exposure")]

def arbiter(evs):
    risk = [e for e in evs if e.direction > 0]; calm = [e for e in evs if e.direction < 0]
    missing = [e.agent for e in evs if e.direction == 0 and e.strength == 0]
    agents_risk = {e.agent for e in risk if e.strength >= 0.4}
    contradiction = any(e.strength >= 0.6 for e in risk) and any(e.strength >= 0.4 and e.agent != "Registry" for e in calm)
    if any(e.signal == "no_current_status" for e in evs) and not risk:
        tier = "Insufficient evidence"
    elif any(e.signal == "adverse_registry_decision" for e in evs) or (any(e.signal == "risk_status" for e in evs) and
                                                                      any(e.signal == "transition_into_risk" for e in evs)):
        tier = "Red"
    elif len(agents_risk) >= 2 or any(e.strength >= 0.5 for e in risk):
        tier = "Amber"
    else:
        tier = "Green"
    conf = "high" if not missing and not contradiction else ("low" if len(missing) >= 2 or contradiction else "medium")
    action = {"Red": "Escalate to sustainability/compliance committee; request evidence; no automatic rejection",
              "Amber": "Enhanced due diligence: request documents and verification reports",
              "Green": "Routine monitoring (not an approval)",
              "Insufficient evidence": "Data request / manual verification"}[tier]
    return {"tier": tier, "confidence": conf, "contradiction": contradiction, "human_review": tier != "Green" or contradiction,
            "reasons": "; ".join(f"{e.agent}:{e.signal}" for e in sorted(risk, key=lambda e: -e.strength)) or "none",
            "missing": ", ".join(missing) or "none", "action": action}

# Transaction-agent percentile: prospective RF out-of-fold score (no leakage), ranked within the cohort
tx_pct = pd.Series(pd.Series(pro["oof_rf"].values).rank(pct=True).values, index=pro.project_id.values)
status_now = panel_v4[SNAPS].ffill(axis=1).iloc[:, -1]
tier_now = status_now.map(STATUS_ONTOLOGY)
ever_watch = (tiers_v4 == "watch").any(axis=1)
decisions = []
for idx, r in panel_v4.iterrows():
    pid = int(r.project_key[3:]) if r.registry == "VCS" and r.project_key[3:].isdigit() else None
    evs = registry_agent(tier_now[idx] if pd.notna(tier_now[idx]) else None, bool(r.moved_into_risk), bool(ever_watch[idx]), status_now[idx])
    if pid is not None:
        evs += transaction_agent(tx_pct.get(pid, np.nan)) + claim_agent(claim_map.get(pid, np.nan))
    evs += onchain_agent(None)
    d = arbiter(evs); d.update({"project_key": r.project_key, "registry": r.registry,
                                "confirmed_fraud": r.project_key in {f"VCS{i}" for i in fraud_cluster},
                                "in_candidate_pool": bool(r.moved_into_risk)})
    decisions.append(d)
    if d["confirmed_fraud"] or (len(decisions) % 2000 == 1):
        with open(out5("audit_trail_examples.jsonl"), "a") as fh:
            fh.write(json.dumps({"project": r.project_key, "evidence": [asdict(e) for e in evs], "decision": d}) + "\n")
decisions = pd.DataFrame(decisions)
decisions["issued_MtCO2e"] = decisions.project_key.map(iss_by_key).fillna(0) / 1e6
tiers_tab = decisions.groupby("tier").agg(projects=("project_key", "size"), issued_MtCO2e=("issued_MtCO2e", "sum"),
                                          confirmed_fraud=("confirmed_fraud", "sum"), candidate_pool=("in_candidate_pool", "sum"),
                                          human_review=("human_review", "sum"), contradictions=("contradiction", "sum"))
print(tiers_tab.round(2)); save5(decisions, "arbiter_decisions_all_projects.csv"); save5(tiers_tab, "triage_tiers.csv", index=True)

# Ablation AS OF DEC 2024 (prospective): which agents would have flagged future risk entrants / confirmed cases?
abl = []
cohort = pro.copy()
cohort["pct_tx"] = cohort.project_id.map(tx_pct)
for name, use_tx, use_claim in [("Registry only", False, False), ("Registry + transaction", True, False),
                                ("Registry + claim consistency", False, True), ("All agents", True, True)]:
    flag = []
    for _, r in cohort.iterrows():
        evs = registry_agent(r.tier_2024, False, r.tier_2024 == "watch", "watch" if r.tier_2024 == "watch" else "Registered")
        if use_tx: evs += transaction_agent(r.pct_tx)
        if use_claim: evs += claim_agent(r.issuance_to_claim_ratio)
        flag.append(arbiter(evs)["tier"] in ("Amber", "Red"))
    flag = np.array(flag); conf = cohort.project_id.isin(fraud_cluster).values
    abl.append({"agents": name, "flagged_Dec2024": int(flag.sum()), "flagged_share_%": 100 * flag.mean(),
                "future_risk_entrants_caught": f"{int((flag & (cohort.y == 1)).sum())}/{int(cohort.y.sum())}",
                "confirmed_fraud_caught": f"{int((flag & conf).sum())}/{int(conf.sum())}",
                "precision_%": 100 * (flag & (cohort.y == 1)).sum() / max(flag.sum(), 1)})
abl = pd.DataFrame(abl); print(abl.round(2).to_string(index=False)); save5(abl, "agent_ablation_dec2024.csv")
tick("V5-9")

In [ ]:
# ===== V5-10  Scoring-weight sensitivity (rule points and consensus weights) =====
cur_risk = (tier_now == "risk_flag").astype(int); moved = panel_v4.moved_into_risk.astype(int)
lag_flag = pd.Series(0, index=panel_v4.index)
ctrl_med_lag = feat_asof.loc[feat_asof.project_id.isin(eligible_ids), "mean_issuance_lag_days"].median()
lag_by_pid = feat_asof.set_index("project_id")["mean_issuance_lag_days"]
vcs_idx = panel_v4.registry == "VCS"
lag_flag[vcs_idx] = (pd.to_numeric(panel_v4.loc[vcs_idx, "project_key"].str[3:], errors="coerce").map(lag_by_pid)
                     < 0.6 * ctrl_med_lag).fillna(False).astype(int).values
weight_sets = {"Published 40/20/15": (40, 20, 15), "30/30/15": (30, 30, 15), "50/10/15": (50, 10, 15),
               "40/20/0 (no lag rule)": (40, 20, 0), "Equal 25/25/25": (25, 25, 25)}
from scipy.stats import kendalltau
base_score = 40 * cur_risk + 20 * moved + 15 * lag_flag
ws = []
for name, (a, b, c) in weight_sets.items():
    s = a * cur_risk + b * moved + c * lag_flag
    red = set(panel_v4.project_key[s >= 60]); base_red = set(panel_v4.project_key[base_score >= 60])
    ws.append({"weights": name, "projects_score>=60": len(red), "jaccard_vs_published": len(red & base_red) / max(len(red | base_red), 1),
               "kendall_tau_vs_published": kendalltau(s, base_score)[0],
               "confirmed_score>=60": int(panel_v4.project_key[s >= 60].isin({f"VCS{i}" for i in fraud_cluster}).sum())})
# consensus weights: the narrative score is bounded 0-100, so bound its influence on the final tier
for w in [0.0, 0.2, 0.3, 0.4, 0.5]:
    lo_s, hi_s = (1 - w) * base_score + w * 0, (1 - w) * base_score + w * 100
    ws.append({"weights": f"Consensus registry {1 - w:.1f} / narrative {w:.1f}",
               "projects_score>=60": f"{int((lo_s >= 60).sum())}-{int((hi_s >= 60).sum())} (narrative 0 vs 100)",
               "jaccard_vs_published": np.nan, "kendall_tau_vs_published": np.nan,
               "confirmed_score>=60": f"{int((lo_s[panel_v4.project_key.isin({f'VCS{i}' for i in fraud_cluster})] >= 60).sum())}"
                                      f"-{int((hi_s[panel_v4.project_key.isin({f'VCS{i}' for i in fraud_cluster})] >= 60).sum())}"})
ws = pd.DataFrame(ws); print(ws.to_string(index=False)); save5(ws, "weight_sensitivity.csv")
tick("V5-10")

In [ ]:
# ===== V5-11  Fairness / unintended exclusion + uncertainty on the inclusive-finance shares =====
hh2 = hh.drop_duplicates("key").copy()
hh2["in_pool"] = hh2.key.isin(set(candidate_pool_v4.project_key))
hh2["registry_std"] = hh2.key.map(registry_from_key)
fair = []
for col in ["Scope", "Region", "registry_std"]:
    for val, gdf in hh2.groupby(col):
        n, k = len(gdf), int(gdf.in_pool.sum()); lo, hi = wilson(k, n)
        fair.append({"dimension": col, "group": val, "projects": n, "flagged": k, "flag_rate_%": 100 * k / n,
                     "CI95_low_%": 100 * lo, "CI95_high_%": 100 * hi,
                     "share_of_credits_flagged_%": 100 * gdf.loc[gdf.in_pool, c_iss].sum() / max(gdf[c_iss].sum(), 1)})
fair = pd.DataFrame(fair); overall = 100 * hh2.in_pool.mean()
print(f"Overall per-project flag rate: {overall:.2f}%")
print(fair[fair.projects >= 50].sort_values(["dimension", "flag_rate_%"], ascending=[True, False]).round(2).to_string(index=False))
save5(fair, "fairness_flag_rates.csv")

# Bootstrap CI on the credit-weighted household share of the pool + permutation vs random 113-project pools
pool_df = hh2[hh2.in_pool]; rng = np.random.default_rng(GLOBAL_SEED)
def hshare(d): return 100 * d.loc[d.household_community, c_iss].sum() / d[c_iss].sum()
obs = hshare(pool_df)
boot = [hshare(pool_df.sample(len(pool_df), replace=True, random_state=int(rng.integers(1e9)))) for _ in range(V5["N_BOOT"])]
null = [hshare(hh2.sample(len(pool_df), random_state=int(rng.integers(1e9)))) for _ in range(V5["N_BOOT"])]
incl_unc = pd.DataFrame([{"metric": "Household & community share of pool credits (%)", "observed": obs,
                          "bootstrap_CI95_low": np.percentile(boot, 2.5), "bootstrap_CI95_high": np.percentile(boot, 97.5),
                          "random_113_pool_median": np.median(null), "perm_p_one_sided": (np.sum(np.array(null) >= obs) + 1) / (len(null) + 1),
                          "project_count_share_%": 100 * pool_df.household_community.mean()}])
vcs_all = hh2[hh2.registry_std == "VCS"]; vcs_pool = vcs_all[vcs_all.in_pool]
null_v = [hshare(vcs_all.sample(len(vcs_pool), random_state=int(rng.integers(1e9)))) for _ in range(V5["N_BOOT"])]
incl_unc = pd.concat([incl_unc, pd.DataFrame([{"metric": "Same, VCS-only baseline (pool is 112/113 VCS)", "observed": hshare(vcs_pool),
    "bootstrap_CI95_low": np.nan, "bootstrap_CI95_high": np.nan, "random_113_pool_median": np.median(null_v),
    "perm_p_one_sided": (np.sum(np.array(null_v) >= hshare(vcs_pool)) + 1) / (len(null_v) + 1),
    "project_count_share_%": 100 * vcs_pool.household_community.mean(), "market_share_%": hshare(vcs_all)}])], ignore_index=True)
incl_unc.loc[0, "market_share_%"] = hshare(hh2)
print(incl_unc.round(3).to_string(index=False)); save5(incl_unc, "inclusive_share_uncertainty.csv")
tick("V5-11")

In [ ]:
# ===== V5-12  Coverage: why do 384 ACR projects lack a 2026-04 status? =====
acr = hh2[hh2.registry_std == "ACR"].copy()
acr_status_col = "Voluntary Status"
acr["missing_status"] = acr[acr_status_col].isna()
arb_col = [c for c in hh2.columns if str(c).startswith("ARB / WA") and "Project" in str(c)][0]
cov = acr.groupby([arb_col, "missing_status"]).size().unstack(fill_value=0)
print(cov); save5(cov, "acr_missing_by_arb.csv", index=True)
from scipy.stats import chi2_contingency
chi = chi2_contingency(cov.values)
print(f"Chi-square missing-status vs ARB/WA programme: chi2={chi[0]:.1f}, p={chi[1]:.2e}")
print(acr.groupby("missing_status")[c_iss].agg(["count", "median", "sum"]))
ACR_COVERAGE = {"chi2": chi[0], "p": chi[1]}
tick("V5-12")

In [ ]:
# ===== V5-13  Ground-truth expansion worksheet: graded labels from registry decisions and notes =====
notes_col = find_col(df_harmonized, "notes from registry"); docs_col = find_col(df_harmonized, "registry documents")
dev_col = find_col(df_harmonized, "project developer")
info = hh.drop_duplicates("key").set_index("key")
URL = {"VCS": "https://registry.verra.org/app/projectDetail/VCS/{n}", "GS": "https://registry.goldstandard.org/projects/details/{n}",
       "ACR": "https://acr2.apx.com/mymodule/reg/prjView.asp?id1={n}", "CAR": "https://thereserve2.apx.com/mymodule/reg/prjView.asp?id1={n}",
       "ART": "https://art.apx.com/mymodule/reg/prjView.asp?id1={n}"}
ws_rows = []
for _, r in candidate_pool_v4.iterrows():
    k = r.project_key; reg = registry_from_key(k); num = re.sub(r"^[A-Z]+", "", k)
    path = " -> ".join(str(r[s]) for s in SNAPS if pd.notna(r[s]))
    final = str(status_now[panel_v4.project_key == k].iloc[0])
    grade = ("A: registry adverse decision" if any(w in final.lower() for w in ["reject", "denied", "cancel", "unsuccessful"])
             else "B: withdrawn" if "withdrawn" in final.lower() else "C: inactive/other")
    ws_rows.append({"project_key": k, "registry": reg, "confirmed_fraud": k in {f"VCS{i}" for i in fraud_cluster},
                    "status_path": path, "evidence_grade": grade,
                    "issued_credits": info[c_iss].get(k, np.nan), "retired_credits": info[c_ret].get(k, np.nan),
                    "developer": info[dev_col].get(k, "") if dev_col else "", "country": info["Country"].get(k, ""),
                    "scope": info["Scope"].get(k, ""), "registry_notes": info[notes_col].get(k, "") if notes_col else "",
                    "registry_url": URL.get(reg, "").format(n=num),
                    "verification_outcome": "", "source_url": "", "verified_by": "", "verified_on": ""})
gt = pd.DataFrame(ws_rows).sort_values(["confirmed_fraud", "evidence_grade", "issued_credits"], ascending=[False, True, False])
print(gt.evidence_grade.value_counts()); print(gt.groupby("evidence_grade")["issued_credits"].sum() / 1e6)
has_note = gt.registry_notes.fillna("").astype(str).str.strip().str.len() > 3
print(f"Unverified candidates with registry notes: {int((has_note & ~gt.confirmed_fraud).sum())} of {int((~gt.confirmed_fraud).sum())}")
save5(gt, "ground_truth_expansion_worksheet.csv")
tick("V5-13")

In [ ]:
# ===== V5-14  Who used at-risk credits for claims? (aggregated retirement beneficiaries) =====
vr = df_vcus.copy(); vr["ID"] = pd.to_numeric(vr["ID"], errors="coerce")
vr["Quantity Issued"] = pd.to_numeric(vr["Quantity Issued"], errors="coerce").fillna(0)
vr = vr[vr["Retirement/Cancellation Date"].notna()]
pool_vcs = {int(k[3:]) for k in candidate_pool_v4.project_key if k.startswith("VCS") and k[3:].isdigit()}
def benef_summary(ids, label):
    d = vr[vr.ID.isin(ids)]
    reason = d["Retirement Reason"].fillna("unspecified").str.lower()
    cancelled = reason.str.contains("cancel|buffer|correct|replace", regex=True)
    ret = d[~cancelled]
    return {"group": label, "retired_or_cancelled_credits": d["Quantity Issued"].sum(),
            "retired_for_claims_credits": ret["Quantity Issued"].sum(),
            "distinct_named_beneficiaries": ret["Retirement Beneficiary"].dropna().nunique(),
            "share_with_named_beneficiary_%": 100 * ret.loc[ret["Retirement Beneficiary"].notna(), "Quantity Issued"].sum()
                                                / max(ret["Quantity Issued"].sum(), 1)}
benef = pd.DataFrame([benef_summary(set(fraud_cluster), "Confirmed fraud (VCS)"),
                      benef_summary(pool_vcs, "Candidate pool (VCS projects)")])
print(benef.round(1).to_string(index=False)); save5(benef, "claim_exposure_beneficiaries.csv")
top_b = (vr[vr.ID.isin(pool_vcs)].groupby("Retirement Beneficiary")["Quantity Issued"].sum().sort_values(ascending=False).head(15))
save5(top_b.reset_index(), "top_beneficiaries_candidate_pool_APPENDIX_ONLY.csv")
print("Top beneficiaries saved for the appendix only; do not name companies in the main text without checking context.")
tick("V5-14")

## 9. Figures 1, 5 and 6, and reproducibility bundle

In [ ]:
# ===== V5-15  Figures v5 (addresses figure comments + new evidence) =====
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10})
# Fig 1 — transition matrix with colorbar
tm = transition_matrix_v4
fig, ax = plt.subplots(figsize=(4.8, 3.8))
im = ax.imshow(np.log10(tm.values + 1), cmap="cividis")
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.set_label("log10(events + 1)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{tm.values[i, j]:,}", ha="center", va="center",
                color="white" if np.log10(tm.values[i, j] + 1) < np.log10(tm.values.max() + 1) * 0.6 else "black")
ax.set_xticks(range(3)); ax.set_xticklabels(["normal", "watch", "risk flag"]); ax.set_yticks(range(3)); ax.set_yticklabels(["normal", "watch", "risk flag"])
ax.set_xlabel("State at later snapshot"); ax.set_ylabel("State at earlier snapshot"); ax.set_title("Registry status transitions (events)")
plt.tight_layout(); plt.savefig(out5("fig1_transitions_v5.png"), dpi=300); plt.close()

# Fig 2 — effect sizes (current matched run) incl. claim-consistency
f2 = pd.concat([stats_v4, claim_test], ignore_index=True)
labels = {"n_batches": "Issuance batches", "share_credits_retired": "Share of credits retired",
          "mean_issuance_lag_days": "Mean issuance lag", "batch_size_cv": "Batch-size CV",
          "issuance_to_claim_ratio": "Issued / own claimed reductions"}
bl = [c for c in stats_v4.columns if c.endswith("_bonf_low")][0]; bh = [c for c in stats_v4.columns if c.endswith("_bonf_high")][0]
fig, ax = plt.subplots(figsize=(6.4, 3.3))
for i, r in enumerate(f2.itertuples()):
    lo_adj = getattr(r, bl) if pd.notna(getattr(r, bl, np.nan)) else r.CI95_low
    hi_adj = getattr(r, bh) if pd.notna(getattr(r, bh, np.nan)) else r.CI95_high
    ax.plot([lo_adj, hi_adj], [i, i], color=CB["grey"], lw=1)
    ax.plot([r.CI95_low, r.CI95_high], [i, i], color=CB["blue"], lw=3)
    ax.plot(r.rank_biserial_r, i, "o", ms=7, mfc="white", mec=CB["blue"], mew=1.5)
    ax.text(1.02, i, f"r={r.rank_biserial_r:+.2f}", va="center", fontsize=8, transform=ax.get_yaxis_transform())
ax.axvline(0, color="black", ls=":", lw=1); ax.set_xlim(-1.05, 1.05)
ax.set_yticks(range(len(f2))); ax.set_yticklabels([labels.get(f, f) for f in f2.feature])
ax.set_xlabel("Rank-biserial r (positive = higher in confirmed fraud)")
ax.set_title(f"Confirmed fraud (n={int(f2.n_fraud.iloc[0])}) vs matched controls (n={int(f2.n_control.iloc[0])})")
plt.tight_layout(); plt.savefig(out5("fig2_effects_v5.png"), dpi=300); plt.close()

# Fig 3 — inclusive shares with bootstrap CI (credit-weighted denominator stated on axis)
fig, ax = plt.subplots(figsize=(5.2, 2.8))
vals = [inclusive.iloc[0, 5], obs, inclusive.iloc[2, 5]]
err_lo = [0, obs - incl_unc.bootstrap_CI95_low[0], 0]; err_hi = [0, incl_unc.bootstrap_CI95_high[0] - obs, 0]
ax.bar(range(3), vals, color=[CB["grey"], CB["vermillion"], CB["blue"]], hatch=[None, "//", ".."], edgecolor="black", lw=0.3,
       yerr=[err_lo, err_hi], capsize=4)
for i, v_ in enumerate(vals): ax.text(i, v_ + 3, f"{v_:.0f}%", ha="center", fontsize=8)
ax.set_xticks(range(3)); ax.set_xticklabels(["All projects\n(n=11,343)", "Candidate pool\n(n=113)", "Confirmed fraud\n(n=11)"])
ax.set_ylabel("% of credits issued\nfrom household & community"); ax.set_ylim(0, 70)
ax.set_title("Household & community share of issued credits")
plt.tight_layout(); plt.savefig(out5("fig3_household_share_v5.png"), dpi=300); plt.close()

# Fig 4 — prospective early warning: precision-recall
fig, ax = plt.subplots(figsize=(4.8, 3.4))
for (name, p), c, ls in zip(pro_oof.items(), [CB["grey"], CB["blue"], CB["vermillion"], CB["green"] if "green" in CB else "#009E73"], ["-", "--", "-", ":"]):
    pr, rc, _ = precision_recall_curve(yv, p); ax.plot(rc, pr, color=c, ls=ls, lw=1.5, label=f"{name} (AP={average_precision_score(yv, p):.2f})")
ax.axhline(yv.mean(), color="black", ls=":", lw=1, label=f"No skill ({100 * yv.mean():.1f}%)")
ax.set_xlabel("Recall (share of future risk entrants caught)"); ax.set_ylabel("Precision")
ax.set_title("Early warning from Dec 2024 information"); ax.legend(fontsize=6, frameon=False)
plt.tight_layout(); plt.savefig(out5("fig4_early_warning_pr_v5.png"), dpi=300); plt.close()
print("Figures saved to", V5["OUT_DIR"])
tick("V5-15")

In [ ]:
# ===== V5-17  Reproducibility + footprint: hashes, runtimes, versions, workbook =====
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
inputs = [FILE_HARMONIZED, FILE_2024, FILE_2025, FILE_2026_02]
hashes = pd.DataFrame([{"file": os.path.basename(p), "bytes": os.path.getsize(p), "sha256": sha256(p)} for p in inputs])
print(hashes.to_string(index=False)); save5(hashes, "input_file_hashes.csv")
tick("V5-17")
runtime = pd.DataFrame(list(V5_TIMES.items()), columns=["stage_completed", "elapsed_seconds"])
total_s = runtime.elapsed_seconds.max()
footprint = {"v5_runtime_seconds": total_s, "projects_screened": len(decisions),
             "cpu_seconds_per_1000_projects": 1000 * total_s / len(decisions),
             "approx_energy_Wh_at_65W_laptop": 65 * total_s / 3600,
             "note": "Screening itself (arbiter over all projects) is a small fraction of runtime; most time is bootstraps/CV."}
print(json.dumps(footprint, indent=2))
with open(out5("footprint.json"), "w") as fh: json.dump(footprint, fh, indent=2)
sheets5 = {"early_warning_interval": ew, "early_warning_eventual": rr, "lead_time": lead, "ontology_sensitivity": onto,
           "matching_sensitivity": msens, "balance_full": love, "claim_consistency": claim_test,
           "prospective_model": pro_res, "prospective_grouped_cv": grp, "prospective_importance": pro_imp, "external_validation": ext,
           "cluster_bootstrap": clus, "decision_utility": dec, "surveillance_screen": screen, "expected_cost": cost,
           "triage_tiers": tiers_tab.reset_index(), "agent_ablation": abl, "weight_sensitivity": ws, "fairness": fair,
           "inclusive_uncertainty": incl_unc, "acr_coverage": cov.reset_index(), "ground_truth_worksheet": gt,
           "claim_exposure": benef, "input_hashes": hashes, "runtime": runtime}
with pd.ExcelWriter(out5("Supplementary_Appendix_v5.xlsx")) as xw:
    for n_, d_ in sheets5.items(): d_.to_excel(xw, sheet_name=n_[:31], index=False)
print("Wrote", out5("Supplementary_Appendix_v5.xlsx"))

## 10. Expanded confirmed set (23 cases) and third (rice) scheme

In [ ]:
# ===================== EXPANDED CONFIRMED SET (Verra C-Quest review table) =====================
# Source of labels: Verra, "Verra Cancels 5 Million Overissued Credits Linked to C-Quest Capital"
# (per-project 'VCUs compensated' table) + Verra 12 Dec 2025 statement (4 Guizhou Baiheng rejections).
import os, shutil, warnings, numpy as np, pandas as pd
from scipy.stats import mannwhitneyu
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
warnings.simplefilter("ignore")

DATA, OUT = "/content", "/content/v9_outputs"; os.makedirs(OUT, exist_ok=True)
CUTOFF, SEED = pd.Timestamp("2024-12-31"), 42
F = {"2024-12": "VROD-registry-files--2024-12-year-end.xlsx",
     "2025-12": "VROD-registry-files--2025-12.xlsx",
     "2026-02": "VROD-registry-files--2026-02.xlsx"}
FH = "Voluntary-Registry-Offsets-Database--v2026-04.xlsx"

# ---- Labels ----
CQC_COMP = {2340:243980, 2341:471945, 2342:1307556, 2349:186407, 2351:22058, 2366:104173, 2371:138528,
            2372:2035142, 2409:19668, 2521:5281, 2522:29705, 2523:36495, 2711:91262, 2712:249645,
            2886:9490, 2888:638, 2890:2543, 2981:37441, 4157:12958}          # VCUs compensated > 0
GZ = [1847, 1864, 1865, 2070]
PAPER_SET = {2340, 2341, 2371, 2380, 2890, 2981, 4157} | set(GZ)             # the 11 used in v8
NEW_SET = set(CQC_COMP) | set(GZ)                                            # 23, Verra-documented
SCHEME = {**{i: "C-Quest" for i in CQC_COMP}, **{i: "Guizhou" for i in GZ}}

ONT = {"Listed":"normal","Registered":"normal","Completed":"normal","Under development":"normal",
 "Under validation":"normal","under_validation":"normal","validated":"normal","Transitioned":"normal",
 "Gold Standard Certified Design":"normal","Gold Standard Certified Project":"normal",
 "Transfer to Another GHG Program":"normal","Units Transferred from Approved GHG Program":"normal",
 "Requantification Requested":"normal","Registration requested":"normal","Verification approval requested":"normal",
 "Crediting Period Renewal Requested":"normal","Crediting Period Renewal and Verification Approval Requested":"normal",
 "Registration and verification approval requested":"normal",
 "On Hold - see notification letter":"watch","Late to verify":"watch",
 "Canceled":"risk_flag","cancelled":"risk_flag","Withdrawn":"risk_flag","Inactive":"risk_flag",
 "Rejected by Administrator":"risk_flag","Registration request denied":"risk_flag",
 "Registration and verification approval request denied":"risk_flag","Verification approval request denied":"risk_flag",
 "Crediting Period Renewal Request Denied":"risk_flag",
 "Crediting Period Renewal and Verification Approval Request Denied":"risk_flag","validation_unsuccessful":"risk_flag"}

def wilson(k, n, z=1.96):
    p = k/n; d = 1 + z*z/n; c = (p + z*z/(2*n))/d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return round(100*(c-h),1), round(100*(c+h),1)
def rank_biserial(x, y):                      # r > 0 => cases HIGHER than controls
    try: u, p = mannwhitneyu(x, y, alternative="two-sided")
    except ValueError: return np.nan, 1.0
    return 2*u/(len(x)*len(y)) - 1, p
def boot_r(x, y, n=2000):
    rng = np.random.default_rng(SEED); out = []
    for _ in range(n):
        r = rank_biserial(rng.choice(x, len(x)), rng.choice(y, len(y)))[0]
        if np.isfinite(r): out.append(r)
    return np.percentile(out, [2.5, 97.5])
def bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p); q = np.empty(n); prev = 1.0
    for rank, i in reversed(list(enumerate(o, 1))):
        prev = min(prev, p[i]*n/rank); q[i] = prev
    return q

# ---- 1. Registry status panel (Verra) ----
print("Loading files..."); snaps = ["2024-12", "2025-12", "2026-02", "2026-04"]
vp = {s: pd.read_excel(f"{DATA}/{f}", sheet_name="Verra Projects") for s, f in F.items()}
for s in vp:
    vp[s]["ID"] = pd.to_numeric(vp[s]["ID"], errors="coerce")
    vp[s] = vp[s].dropna(subset=["ID"]).drop_duplicates("ID"); vp[s]["ID"] = vp[s]["ID"].astype(int)
hz = pd.read_excel(f"{DATA}/{FH}", sheet_name="PROJECTS", header=3)
hz.columns = [str(c).replace("\n", "").strip() for c in hz.columns]
hz = hz[hz["Voluntary Registry"].astype(str).str.strip().str.upper().eq("VCS")].copy()
hz["ID"] = pd.to_numeric(hz["Project ID"].astype(str).str.upper().str.replace("VCS", "", regex=False)
                         .str.replace(r"\.0$", "", regex=True), errors="coerce")
hz = hz.dropna(subset=["ID"]).drop_duplicates("ID"); hz["ID"] = hz["ID"].astype(int)
st = pd.DataFrame({s: vp[s].set_index("ID")["Status"] for s in vp})
st["2026-04"] = hz.set_index("ID")["Voluntary Status"]
st = st[snaps].apply(lambda c: c.astype("string").str.strip())
tier = st.apply(lambda c: c.map(ONT))
def entered_risk(row):
    obs = [t for t in row if isinstance(t, str)]
    return any(a in ("normal", "watch") and b == "risk_flag" for a, b in zip(obs, obs[1:]))
pool = set(tier.index[tier.apply(entered_risk, axis=1)])
print(f"Verra candidate pool: {len(pool)} (v8 reports 112 Verra of 113)")

# ---- 2. Recall of the candidate pool ----
rec = []
for name, S in [("v8 label set (11)", PAPER_SET), ("Verra-documented set (23)", NEW_SET)]:
    k = len(S & pool); lo, hi = wilson(k, len(S))
    rec.append({"label_set": name, "n": len(S), "in_pool": k, "recall_%": round(100*k/len(S), 1), "wilson95": f"{lo}-{hi}"})
rec = pd.DataFrame(rec); print("\n#1 RECALL OF THE CANDIDATE POOL"); print(rec.to_string(index=False))
missed = sorted(NEW_SET - pool)
miss_tab = pd.DataFrame({"project": [f"VCS{i}" for i in missed], "scheme": [SCHEME[i] for i in missed],
                         "VCUs_compensated": [CQC_COMP.get(i, np.nan) for i in missed]}).join(
                         st.reindex(missed).reset_index(drop=True))
print("\nConfirmed projects NOT in the pool:"); print(miss_tab.to_string(index=False))
print(f"VCUs compensated inside pool: {sum(v for k, v in CQC_COMP.items() if k in pool):,} of {sum(CQC_COMP.values()):,}")
print("VCS2380 in Verra review table:", 2380 in CQC_COMP, "| statuses:", list(st.loc[2380]) if 2380 in st.index else "not found")

# ---- 3. Leakage-safe features at 31 Dec 2024 ----
with pd.ExcelFile(f"{DATA}/{F['2026-02']}") as b:
    vcu = pd.read_excel(b, sheet_name=[n for n in b.sheet_names if n.strip().casefold() == "verra vcus"][0])
v = vcu.copy(); v["ID"] = pd.to_numeric(v["ID"], errors="coerce"); v = v.dropna(subset=["ID"]); v["ID"] = v["ID"].astype(int)
for c in ["Issuance Date", "Vintage Start", "Vintage End", "Retirement/Cancellation Date"]:
    v[c] = pd.to_datetime(v[c], errors="coerce")
v["q"] = pd.to_numeric(v["Quantity Issued"], errors="coerce").fillna(0)
v = v[v["Issuance Date"] <= CUTOFF]
v["ret"] = v["Retirement/Cancellation Date"].le(CUTOFF)
def union_years(g):
    iv = sorted((s, e + pd.Timedelta(days=1)) for s, e in zip(g["Vintage Start"], g["Vintage End"])
                if pd.notna(s) and pd.notna(e) and e >= s)
    if not iv: return np.nan
    tot = pd.Timedelta(0); cs, ce = iv[0]
    for s, e in iv[1:]:
        if s <= ce: ce = max(ce, e)
        else: tot += ce - cs; cs, ce = s, e
    return (tot + ce - cs).days / 365.25
def feats(g):
    lag = (g["Issuance Date"] - g["Vintage End"]).dt.days; tq = g.q.sum()
    return pd.Series({"n_batches": len(g), "share_retired": g.loc[g.ret, "q"].sum()/tq if tq > 0 else np.nan,
                      "mean_issuance_lag_days": lag.mean(), "batch_size_cv": g.q.std()/g.q.mean() if g.q.mean() > 0 else np.nan,
                      "issued": tq, "years_covered": union_years(g)})
X = v.groupby("ID").apply(feats)
eaer = pd.to_numeric(vp["2024-12"].set_index("ID")["Estimated Annual Emission Reductions"], errors="coerce")
X["ccr"] = X.issued / (eaer.reindex(X.index).where(lambda e: e > 0) * X.years_covered.clip(lower=1/12))
info = vp["2026-02"].set_index("ID")[["Proponent", "Project Type"]].combine_first(
       vp["2025-12"].set_index("ID")[["Proponent", "Project Type"]]).combine_first(
       vp["2024-12"].set_index("ID")[["Proponent", "Project Type"]])
print(f"\nVerra projects with issuance before 2025: {len(X)} (v8 reports 1,969)")

# ---- 4. Controls: never flagged, Registered/Completed, no developer link ----
last = st.apply(lambda r: r.dropna().iloc[-1] if r.notna().any() else pd.NA, axis=1)
never = tier.index[~tier.isin(["watch", "risk_flag"]).any(axis=1)]
ctrl = set(never) & set(last.index[last.isin(["Registered", "Completed"])])
props = set(info.reindex(list(NEW_SET | {2380}))["Proponent"].dropna()) - {"Multiple Proponents"}
bad = info.index[info["Proponent"].isin(props) |
                 info["Proponent"].astype(str).str.contains("C-Quest|Bridge Carbon|Baiheng", case=False)]
ctrl = (ctrl - set(bad) - NEW_SET - {2380}) & set(X.index)
print(f"Eligible controls: {len(ctrl)} (v8 reports 1,491)")

# ---- 5. Claim-consistency ratio vs never-flagged population ----
pop = X.loc[sorted(ctrl), "ccr"].dropna()
rows = []
for name, S in [("all confirmed (23)", NEW_SET), ("confirmed IN pool", NEW_SET & pool), ("confirmed MISSED by pool", NEW_SET - pool),
                ("v8 set (11)", PAPER_SET)]:
    x = X.reindex(sorted(S))["ccr"].dropna()
    r, p = rank_biserial(x.values, pop.values)
    rows.append({"group": name, "n": len(x), "median_ccr": round(x.median(), 3), "control_median": round(pop.median(), 3),
                 "rank_biserial_r": round(r, 3), "p": round(p, 4)})
claim = pd.DataFrame(rows); print("\n#2 CLAIM-CONSISTENCY RATIO vs never-flagged Verra controls"); print(claim.to_string(index=False))

# ---- 6. Matched controls (exact type, nearest log10 credits, caliper 0.5 pooled SD, k<=5, no replacement) ----
cases = X.reindex(sorted(NEW_SET)).dropna(subset=["issued"]).copy()
cases["type"] = info.reindex(cases.index)["Project Type"].values; cases["lc"] = np.log10(cases.issued.clip(lower=1))
C = X.loc[sorted(ctrl)].copy(); C["type"] = info.reindex(C.index)["Project Type"].values; C["lc"] = np.log10(C.issued.clip(lower=1))
cal = 0.5*np.sqrt((cases.lc.var() + C.lc.var())/2); used, pairs = set(), []
for cid in cases.sort_values("issued", ascending=False).index:
    cand = C[(C.type == cases.at[cid, "type"]) & (~C.index.isin(used))].copy()
    cand["d"] = (cand.lc - cases.at[cid, "lc"]).abs()
    for j in cand[cand.d <= cal].nsmallest(5, "d").index: used.add(j); pairs.append((cid, j))
FE = ["n_batches", "share_retired", "mean_issuance_lag_days", "batch_size_cv", "ccr"]
data = pd.concat([cases[FE].assign(set_id=cases.index, y=1),
                  X.loc[[j for _, j in pairs], FE].assign(set_id=[c for c, _ in pairs], y=0)])
data["scheme"] = data.set_id.map(SCHEME); data["in_pool"] = data.set_id.isin(pool)
print(f"\nCases with issuance: {len(cases)} | matched controls: {len(pairs)} | caliper {cal:.3f} | "
      f"cases without a match: {sorted(set(cases.index) - {c for c, _ in pairs})}")

# ---- 7. Feature tests (4 pre-specified, BH-FDR; CCR post hoc) ----
ca, co = data[data.y == 1], data[data.y == 0]; tests = []
for f in FE:
    x, y = ca[f].dropna().values, co[f].dropna().values; r, p = rank_biserial(x, y); lo, hi = boot_r(x, y)
    tests.append({"feature": f, "n_case": len(x), "n_ctrl": len(y), "case_median": round(np.median(x), 3),
                  "ctrl_median": round(np.median(y), 3), "r": round(r, 3), "CI95": f"[{lo:+.2f}, {hi:+.2f}]", "p": p})
tests = pd.DataFrame(tests); pre = tests.feature != "ccr"
tests.loc[pre, "FDR_q"] = bh(tests.loc[pre, "p"]); tests["p"] = tests.p.round(4); tests["FDR_q"] = tests.FDR_q.round(3)
print("\n#3 CONFIRMED (23) vs MATCHED CONTROLS, as of 31 Dec 2024 (ccr = post hoc, outside FDR)"); print(tests.to_string(index=False))

# ---- 8. Out-of-pool replication + cross-scheme transfer ----
ARMS = {"transaction": FE[:4], "claim": ["ccr"], "full": FE}
def evaluate(train_mask, test_mask, cols, nboot=2000):
    tr, te = data[train_mask], data[test_mask]
    if te.y.nunique() < 2 or tr.y.nunique() < 2: return np.nan, np.nan, np.nan
    med = tr[cols].median()
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=SEED, n_jobs=-1).fit(tr[cols].fillna(med), tr.y)
    te = te.assign(s=rf.predict_proba(te[cols].fillna(med))[:, 1]); auc = roc_auc_score(te.y, te.s)
    rng = np.random.default_rng(SEED); sets = te.set_id.unique(); bs = []
    for _ in range(nboot):
        b = pd.concat([te[te.set_id == s] for s in rng.choice(sets, len(sets))])
        if b.y.nunique() == 2: bs.append(roc_auc_score(b.y, b.s))
    return round(auc, 3), round(np.percentile(bs, 2.5), 2), round(np.percentile(bs, 97.5), 2)
res = []
for arm, cols in ARMS.items():
    res.append(("Train IN-pool cases -> test MISSED cases", arm, *evaluate(data.in_pool, ~data.in_pool, cols)))
    res.append(("Train C-Quest -> test Guizhou", arm, *evaluate(data.scheme == "C-Quest", data.scheme == "Guizhou", cols)))
    res.append(("Train Guizhou -> test C-Quest", arm, *evaluate(data.scheme == "Guizhou", data.scheme == "C-Quest", cols)))
res = pd.DataFrame(res, columns=["design", "arm", "AUC", "CI_low", "CI_high"])
print("\n#4 OUT-OF-POOL REPLICATION AND CROSS-SCHEME TRANSFER (Random Forest, CI clustered by match set)"); print(res.to_string(index=False))

# ---- 9. Save everything ----
for n, d in {"recall": rec, "missed_confirmed": miss_tab, "claim_ratio": claim, "feature_tests": tests,
             "ml_results": res, "analysis_data": data.reset_index()}.items():
    d.to_csv(f"{OUT}/{n}.csv", index=False)
shutil.make_archive("/content/v9_outputs", "zip", OUT)
print("\nDONE -> download /content/v9_outputs.zip.")

In [ ]:
# ===================== THIRD SCHEME (Verra rice rejections, 28 Aug 2024) + NEW FIGURES 2–4 =====================
# Run in the SAME session right after the v9 cell.
# Source: Verra, "Verra Rejects China Rice Cultivation Projects, Sanctions Auditing Firms and Project Proponents" (28 Aug 2024);
#         Verra, "Update on Compensation for VCUs from Rejected Rice Cultivation Projects" (9 Jan 2025): 25 projects over-issued.
import matplotlib; import matplotlib.pyplot as plt

RICE_ISSUED = [2362, 2396, 2397, 2398, 2399, 2476, 2477, 2504, 2506, 2507, 2629, 2631, 2632, 2633, 2634,
               2769, 2771, 2899, 2902, 2903, 2909, 2910, 2911, 2912, 2917]          # the 25 with issued VCUs (asterisked)
RICE_CUTOFF = pd.Timestamp("2024-06-30")                                            # before the Aug 2024 rejection: leakage-safe

# ---- 1. Status check ----
print("Rice projects found in panel:", sum(i in st.index for i in RICE_ISSUED), "of 25")
print(st.reindex(RICE_ISSUED)["2024-12"].value_counts(dropna=False).to_string())

# ---- 2. Features as of 30 June 2024 (rice cases and their controls) ----
vr = vcu.copy(); vr["ID"] = pd.to_numeric(vr["ID"], errors="coerce"); vr = vr.dropna(subset=["ID"]); vr["ID"] = vr["ID"].astype(int)
for c in ["Issuance Date", "Vintage Start", "Vintage End", "Retirement/Cancellation Date"]:
    vr[c] = pd.to_datetime(vr[c], errors="coerce")
vr["q"] = pd.to_numeric(vr["Quantity Issued"], errors="coerce").fillna(0)
vr = vr[vr["Issuance Date"] <= RICE_CUTOFF]; vr["ret"] = vr["Retirement/Cancellation Date"].le(RICE_CUTOFF)
XR = vr.groupby("ID").apply(feats)
XR["ccr"] = XR.issued / (eaer.reindex(XR.index).where(lambda e: e > 0) * XR.years_covered.clip(lower=1/12))
rcases = XR.reindex(RICE_ISSUED).dropna(subset=["issued"]).copy()
print("Rice cases with issuance before cutoff:", len(rcases))

# ---- 3. Matched controls for rice (unused in training, no shared proponent) ----
rice_props = set(info.reindex(RICE_ISSUED)["Proponent"].dropna()) - {"Multiple Proponents"}
train_ctrl = {j for _, j in pairs}
rpool = [i for i in ctrl if i not in train_ctrl and i in XR.index and info["Proponent"].get(i) not in rice_props]
RC = XR.loc[rpool].copy(); RC["type"] = info.reindex(RC.index)["Project Type"].values; RC["lc"] = np.log10(RC.issued.clip(lower=1))
rcases["type"] = info.reindex(rcases.index)["Project Type"].values; rcases["lc"] = np.log10(rcases.issued.clip(lower=1))
rcal = 0.5*np.sqrt((rcases.lc.var() + RC.lc.var())/2); rused, rpairs = set(), []
for cid in rcases.sort_values("issued", ascending=False).index:
    cand = RC[(RC.type == rcases.at[cid, "type"]) & (~RC.index.isin(rused))].copy()
    cand["d"] = (cand.lc - rcases.at[cid, "lc"]).abs()
    for j in cand[cand.d <= rcal].nsmallest(5, "d").index: rused.add(j); rpairs.append((cid, j))
rdata = pd.concat([rcases[FE].assign(set_id=rcases.index, y=1),
                   XR.loc[[j for _, j in rpairs], FE].assign(set_id=[c for c, _ in rpairs], y=0)])
rdata = rdata[rdata.set_id.isin({c for c, _ in rpairs})]                            # keep matched sets only
print(f"Rice: {rdata.y.sum()} matched cases, {len(rpairs)} controls, caliper {rcal:.3f}")

# ---- 4. Do the signals replicate on the third scheme? ----
rt = []
for f in FE:
    x, y = rdata.loc[rdata.y == 1, f].dropna().values, rdata.loc[rdata.y == 0, f].dropna().values
    if len(x) < 2 or len(y) < 2: continue
    r, p = rank_biserial(x, y); lo, hi = boot_r(x, y)
    rt.append({"feature": f, "n_case": len(x), "n_ctrl": len(y), "case_median": round(np.median(x), 3),
               "ctrl_median": round(np.median(y), 3), "r": round(r, 3), "CI95": f"[{lo:+.2f}, {hi:+.2f}]", "p": round(p, 4)})
rt = pd.DataFrame(rt); print("\n#5 THIRD SCHEME (rice) vs matched controls, features as of 30 Jun 2024"); print(rt.to_string(index=False))

def auc_ci(df, score, nboot=2000):
    df = df.assign(s=score); auc = roc_auc_score(df.y, df.s); rng = np.random.default_rng(SEED); sets = df.set_id.unique(); bs = []
    for _ in range(nboot):
        b = pd.concat([df[df.set_id == s] for s in rng.choice(sets, len(sets))])
        if b.y.nunique() == 2: bs.append(roc_auc_score(b.y, b.s))
    return round(auc, 3), round(np.percentile(bs, 2.5), 2), round(np.percentile(bs, 97.5), 2)
ext = []
for arm, cols in ARMS.items():                                                       # frozen models trained on all 23 cases
    med = data[cols].median()
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=SEED, n_jobs=-1).fit(data[cols].fillna(med), data.y)
    ext.append(("Train C-Quest+Guizhou -> test rice (3rd scheme)", arm, *auc_ci(rdata, rf.predict_proba(rdata[cols].fillna(med))[:, 1])))
for f in ["mean_issuance_lag_days", "ccr"]:                                           # no training: direction fixed in advance
    d = rdata.dropna(subset=[f]); ext.append(("Rule, no training (lower = riskier) -> rice", f, *auc_ci(d, -d[f])))
ext = pd.DataFrame(ext, columns=["design", "arm", "AUC", "CI_low", "CI_high"])
print("\n#6 EXTERNAL-SCHEME TEST"); print(ext.to_string(index=False))

# ---- 5. Figures 2–4 (23-case analysis + third scheme) ----
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9})
BLUE, ORANGE, GREY, GREEN = "#0072B2", "#E69F00", "#7F7F7F", "#009E73"
NAMES = {"n_batches": "Issuance batches", "share_retired": "Share of credits retired",
         "mean_issuance_lag_days": "Mean issuance lag", "batch_size_cv": "Batch-size CV", "ccr": "Issued ÷ own claimed reductions*"}
# Fig 2: effect sizes
fig, ax = plt.subplots(figsize=(6.4, 3.2))
for k, f in enumerate(FE):
    x, y = data.loc[data.y == 1, f].dropna().values, data.loc[data.y == 0, f].dropna().values
    r = rank_biserial(x, y)[0]; lo, hi = boot_r(x, y); col = ORANGE if f == "ccr" else BLUE
    ax.plot([lo, hi], [k + .12, k + .12], color=col, lw=2.5); ax.plot(r, k + .12, "o", color=col, mfc="white", ms=6)
    row = rt[rt.feature == f]
    if len(row):
        lo2, hi2 = [float(z) for z in row.CI95.iloc[0].strip("[]").split(",")]
        ax.plot([lo2, hi2], [k - .12, k - .12], color=GREEN, lw=1.5); ax.plot(row.r.iloc[0], k - .12, "D", color=GREEN, ms=5)
ax.axvline(0, color=GREY, ls=":", lw=1); ax.set_yticks(range(len(FE))); ax.set_yticklabels([NAMES[f] for f in FE])
ax.set_xlim(-1, 1); ax.set_xlabel("Rank-biserial r (positive = higher in confirmed projects)")
ax.set_title(f"Confirmed (n={int(data.y.sum())}) vs matched controls (n={int((data.y == 0).sum())}); rice scheme in green", fontsize=9)
fig.tight_layout(); fig.savefig(f"{OUT}/Figure2_effect_sizes_v9.png", dpi=300); plt.show()
# Fig 3: claim-consistency ratio by group
groups = [("Never-flagged\ncontrols", X.loc[sorted(ctrl), "ccr"].dropna(), GREY),
          ("Confirmed\nin pool", X.reindex(sorted(NEW_SET & pool))["ccr"].dropna(), BLUE),
          ("Confirmed\nmissed by pool", X.reindex(sorted(NEW_SET - pool))["ccr"].dropna(), ORANGE),
          ("Rice scheme\n(3rd scheme)", rcases["ccr"].dropna(), GREEN)]
fig, ax = plt.subplots(figsize=(6.4, 3.4)); rng = np.random.default_rng(SEED)
for k, (lab, vals, col) in enumerate(groups):
    vals = vals[vals > 0]
    if k == 0: ax.boxplot(vals, positions=[k], widths=.5, showfliers=False)
    else: ax.scatter(k + rng.uniform(-.12, .12, len(vals)), vals, color=col, s=18, alpha=.85)
    ax.hlines(np.median(vals), k - .25, k + .25, color="black", lw=1.5)
ax.axhline(1, color=GREY, ls=":", lw=1); ax.set_yscale("log"); ax.set_xticks(range(4)); ax.set_xticklabels([g[0] for g in groups])
ax.set_ylabel("Issued ÷ own claimed reductions (log)"); ax.set_title("Claim-consistency ratio; black line = median", fontsize=9)
fig.tight_layout(); fig.savefig(f"{OUT}/Figure3_claim_ratio_v9.png", dpi=300); plt.show()
# Fig 4: AUC forest plot
allres = pd.concat([res, ext], ignore_index=True)
fig, ax = plt.subplots(figsize=(6.4, 0.32*len(allres) + 0.8))
for k, r_ in allres.iloc[::-1].reset_index(drop=True).iterrows():
    col = GREEN if "rice" in r_.design else (ORANGE if "MISSED" in r_.design else BLUE)
    ax.plot([r_.CI_low, r_.CI_high], [k, k], color=col, lw=2); ax.plot(r_.AUC, k, "o", color=col, mfc="white", ms=5)
ax.set_yticks(range(len(allres))); ax.set_yticklabels([f"{d} · {a}" for d, a in zip(allres.design[::-1], allres.arm[::-1])], fontsize=7)
ax.axvline(.5, color=GREY, ls=":", lw=1); ax.set_xlim(0, 1); ax.set_xlabel("AUC with 95% CI (dotted line = chance)")
fig.tight_layout(); fig.savefig(f"{OUT}/Figure4_auc_v9.png", dpi=300); plt.show()

rt.to_csv(f"{OUT}/rice_feature_tests.csv", index=False); ext.to_csv(f"{OUT}/rice_external_auc.csv", index=False)
shutil.make_archive("/content/v9_outputs", "zip", OUT)
print("\nDONE -> download /content/v9_outputs.zip.")

## 11. Revision analyses
Matching on methodology and vintage, simple baselines, leave-one-scheme-out tests and sensitivity analyses.

In [ ]:
# ===================== Revision analyses =====================
# Run after the 23-case and rice cells (Section 10); reuses their variables.
#  A  Headline recall (11 of 23 primary; 12 of 24 as a sensitivity check)
#  B  Watch state on the prospective cohort (relative risk and pooled AUC)
#  C  Re-matching on methodology, vintage and size, with a balance table
#  D  Feature tests on the new matched sample (FDR over all five and over the four pre-specified)
#  E  Simple baselines against the Random Forest, leave-one-scheme-out including rice
#  F  Sensitivity analyses on the 23-case set
#  G  Population-level baselines, including a cookstove-methodology flag
import os, re, shutil, itertools, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

OUT10 = "/content/v10_outputs"; os.makedirs(OUT10, exist_ok=True)
def save(df, name): df.to_csv(f"{OUT10}/{name}.csv", index=False); return df
SIGN = {"n_batches": +1, "share_retired": +1, "mean_issuance_lag_days": -1, "batch_size_cv": +1, "ccr": -1}
PRESPEC = ["n_batches", "share_retired", "mean_issuance_lag_days", "batch_size_cv"]       # ccr was added post hoc

def boot_auc(y, s, groups=None, nboot=2000):
    """AUC with 95% CI; resamples match sets when `groups` is given, else stratified by class."""
    y, s = np.asarray(y, int), np.asarray(s, float); ok = ~np.isnan(s); y, s = y[ok], s[ok]
    if len(np.unique(y)) < 2: return np.nan, np.nan, np.nan
    auc = roc_auc_score(y, s); rng = np.random.default_rng(SEED); bs = []
    if groups is not None:
        g = np.asarray(groups)[ok]; ug = np.unique(g); idx = {k: np.where(g == k)[0] for k in ug}
        for _ in range(nboot):
            i = np.concatenate([idx[k] for k in rng.choice(ug, len(ug))])
            if len(np.unique(y[i])) == 2: bs.append(roc_auc_score(y[i], s[i]))
    else:
        pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
        for _ in range(nboot):
            i = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]; bs.append(roc_auc_score(y[i], s[i]))
    return round(auc, 3), round(np.percentile(bs, 2.5), 2), round(np.percentile(bs, 97.5), 2)

# ---------------------------------------------------------------- A. HEADLINE RECALL
A = []
for name, S in [("PRIMARY: Verra-documented set (23)", NEW_SET), ("Sensitivity: + VCS2380 (24)", NEW_SET | {2380})]:
    k = len(S & pool); lo, hi = wilson(k, len(S))
    A.append({"set": name, "n": len(S), "caught_by_pool": k, "recall_%": round(100 * k / len(S), 1), "wilson95_%": f"{lo}-{hi}"})
print("A. RECALL"); print(save(pd.DataFrame(A), "A_recall").to_string(index=False))

# ---------------------------------------------------------------- B. WATCH STATE (prospective cohort)
# Cohort: Verra projects with issuance before 2025 that were normal or watch on 31 Dec 2024.
# Outcome: risk_flag at any later snapshot. This is the right place to test watch state, because the
# matched sample's controls were chosen to have NEVER been on watch (so watch is fixed by design there).
coh = tier.reindex(X.index)
coh = coh[coh["2024-12"].isin(["normal", "watch"])]
later = [c for c in tier.columns if c != "2024-12"]
yB = (coh[later] == "risk_flag").any(axis=1).astype(int).values
wB = (coh["2024-12"] == "watch").astype(int).values
a, b = int(((wB == 1) & (yB == 1)).sum()), int(((wB == 1) & (yB == 0)).sum())
c, d = int(((wB == 0) & (yB == 1)).sum()), int(((wB == 0) & (yB == 0)).sum())
A_, B_, C_, D_ = (x + 0.5 for x in (a, b, c, d)) if 0 in (a, b, c, d) else (a, b, c, d)
rr = (A_ / (A_ + B_)) / (C_ / (C_ + D_)); se = np.sqrt(1 / A_ - 1 / (A_ + B_) + 1 / C_ - 1 / (C_ + D_))
auc_w = boot_auc(yB, wB)
B = pd.DataFrame([{"cohort_n": len(yB), "events": int(yB.sum()), "watch&event": a, "watch&no_event": b,
                   "normal&event": c, "normal&no_event": d, "RR": round(rr, 2),
                   "RR_95CI": f"{np.exp(np.log(rr) - 1.96 * se):.2f}-{np.exp(np.log(rr) + 1.96 * se):.2f}",
                   "pooled_AUC": auc_w[0], "AUC_95CI": f"{auc_w[1]}-{auc_w[2]}"}])
print("\nB. WATCH STATE, prospective cohort (report THIS; drop the 0.14 from the matched-sample LOGO)")
print(save(B, "B_watch_state").to_string(index=False))
_ctrl_watch = int(tier.reindex(sorted(ctrl)).isin(["watch"]).any(axis=1).sum())
print(f"   Matched-sample controls ever on watch: {_ctrl_watch} of {len(ctrl)} (zero by construction), "
      "so a watch-based AUC in the matched sample is not interpretable.")

# ---------------------------------------------------------------- C. RE-MATCHING: methodology + vintage + size
def first_code(m):
    m = str(m).upper().replace(" ", "")
    hit = re.findall(r"(VMR\d{4}|VM\d{4}|ACM\d{4}|AM\d{4}|AMS-[IV]+\.[A-Z]+|AR-[A-Z]+\d*|VMD\d{4})", m)
    return hit[0] if hit else (m if m not in ("", "NAN", "NONE") else np.nan)
meth_raw = pd.concat([vp[s].set_index("ID")["Methodology"] for s in ["2026-02", "2025-12", "2024-12"] if "Methodology" in vp[s]])
meth_raw = meth_raw[~meth_raw.index.duplicated()]
METH = meth_raw.map(first_code)
VINT = v.groupby("ID")["Vintage Start"].min().dt.year            # first vintage year, issuance before 2025
VINT_R = vr.groupby("ID")["Vintage Start"].min().dt.year         # same for the rice cutoff

def build(df_feat, ids, vint):
    t = df_feat.reindex(sorted(ids)).dropna(subset=["issued"]).copy()
    t["type"] = info.reindex(t.index)["Project Type"].values
    t["meth"] = METH.reindex(t.index).values
    t["vint"] = vint.reindex(t.index).values
    t["lc"] = np.log10(t.issued.clip(lower=1)); return t

def match(cs, cp, exact="meth", vint_win=1, k=5, cal_mult=0.5, fallback=True):
    """Exact on methodology (fallback: project type), vintage within +/- vint_win years,
    nearest log10 credits within caliper, k <= 5, no replacement, largest cases first."""
    cal = cal_mult * np.sqrt((cs.lc.var() + cp.lc.var()) / 2); used, pr, rep = set(), [], []
    for cid in cs.sort_values("issued", ascending=False).index:
        level = exact
        def cands(col):
            q = cp[(cp[col] == cs.at[cid, col]) & (~cp.index.isin(used))]
            if vint_win is not None and pd.notna(cs.at[cid, "vint"]):
                q = q[(q.vint - cs.at[cid, "vint"]).abs() <= vint_win]
            q = q.assign(dist=(q.lc - cs.at[cid, "lc"]).abs()); return q[q.dist <= cal].nsmallest(k, "dist")
        q = cands(exact) if pd.notna(cs.at[cid, exact]) else cp.iloc[0:0]
        if len(q) == 0 and fallback and exact == "meth": level, q = "type", cands("type")
        used.update(q.index); pr += [(cid, j) for j in q.index]
        rep.append({"case": cid, "scheme": SCHEME.get(cid, "Rice"), "methodology": cs.at[cid, "meth"],
                    "vintage": cs.at[cid, "vint"], "matched_on": level if len(q) else "UNMATCHED", "n_controls": len(q)})
    return pr, pd.DataFrame(rep), cal

def assemble(cs, feat_src, pr):
    keep = sorted({c for c, _ in pr})
    out = pd.concat([cs.loc[keep, FE].assign(set_id=keep, y=1),
                     feat_src.loc[[j for _, j in pr], FE].assign(set_id=[c for c, _ in pr], y=0)])
    out["scheme"] = out.set_id.map(lambda i: SCHEME.get(i, "Rice")); return out

def smd(a, b):
    a, b = pd.Series(a, dtype=float).dropna(), pd.Series(b, dtype=float).dropna()
    s = np.sqrt((a.var() + b.var()) / 2); return round((a.mean() - b.mean()) / s, 2) if s > 0 else 0.0

def balance(cs, cp, pr, label):
    ctrl_ids = [j for _, j in pr]; case_ids = sorted({c for c, _ in pr}); rows = []
    for nm, x1, x0 in [("log10 credits", cs.loc[case_ids, "lc"], cp.loc[ctrl_ids, "lc"]),
                       ("first vintage year", cs.loc[case_ids, "vint"], cp.loc[ctrl_ids, "vint"])]:
        rows.append({"design": label, "covariate": nm, "SMD_matched": smd(x1, x0)})
    same = np.mean([cs.at[c, "meth"] == cp.at[j, "meth"] for c, j in pr]) if pr else np.nan
    rows.append({"design": label, "covariate": "share of pairs with same methodology", "SMD_matched": round(same, 2)})
    return rows

cs23 = build(X, NEW_SET, VINT); cp23 = build(X, ctrl, VINT)
pr10, rep10, cal10 = match(cs23, cp23, exact="meth", vint_win=1)
data10 = assemble(cs23, X, pr10)
bal = balance(cs23, cp23, pr10, "v10: methodology+vintage±1+size") + \
      balance(cs23, cp23, pairs, "v9: project type+size")
print(f"\nC. RE-MATCHING  caliper={cal10:.3f} | cases matched: {int((rep10.n_controls > 0).sum())}/{len(rep10)} "
      f"| via methodology: {(rep10.matched_on == 'meth').sum()} | via type fallback: {(rep10.matched_on == 'type').sum()} "
      f"| controls: {len(pr10)}")
save(rep10, "C_matching_report"); print(save(pd.DataFrame(bal), "C_balance").to_string(index=False))
print("   Unmatched cases:", rep10.loc[rep10.n_controls == 0, "case"].tolist())

# ---------------------------------------------------------------- D. FEATURE TESTS on the v10 matched sample
def feature_tests(dd, label):
    ca, co = dd[dd.y == 1], dd[dd.y == 0]; rows = []
    for f in FE:
        x, y = ca[f].dropna().values, co[f].dropna().values
        if len(x) < 3 or len(y) < 3: continue
        r, p = rank_biserial(x, y); lo, hi = boot_r(x, y)
        rows.append({"design": label, "feature": f, "n_case": len(x), "n_ctrl": len(y), "case_median": round(np.median(x), 3),
                     "ctrl_median": round(np.median(y), 3), "r": round(r, 3), "CI95": f"[{lo:+.2f}, {hi:+.2f}]", "p": p})
    t = pd.DataFrame(rows)
    if len(t):
        t["q_all5"] = bh(t.p.values).round(4)                                         # FDR over all features
        pre = t.feature.isin(PRESPEC); t.loc[pre, "q_prespec4"] = bh(t.loc[pre, "p"].values).round(4)
        t["p"] = t.p.round(4)
    return t
D = feature_tests(data10, "v10 matched (methodology+vintage)")
print("\nD. FEATURE TESTS, v10 matched sample  (q_all5 = FDR over all five; q_prespec4 = the four pre-specified)")
print(save(D, "D_feature_tests_v10").to_string(index=False))

# ---------------------------------------------------------------- E. BASELINES + LEAVE-ONE-SCHEME-OUT incl. rice
ARMS10 = {"RF transaction": FE[:4], "RF claim": ["ccr"], "RF full": FE}
def rf_eval(tr, te, cols):
    med = tr[cols].median()
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=SEED, n_jobs=-1)
    rf.fit(tr[cols].fillna(med), tr.y); return boot_auc(te.y, rf.predict_proba(te[cols].fillna(med))[:, 1], te.set_id)
def rule_score(te, f): return SIGN[f] * te[f].astype(float)
def two_rule(te):  # no training: average rank of "fast issuance" and "low claim ratio"
    return (rule_score(te, "mean_issuance_lag_days").rank(pct=True) + rule_score(te, "ccr").rank(pct=True)) / 2

def loso_table(dd, label):
    rows = []
    for s in sorted(dd.scheme.unique()):
        tr, te = dd[dd.scheme != s], dd[dd.scheme == s]
        if te.y.nunique() < 2 or tr.y.nunique() < 2: continue
        n = int(te.y.sum())
        for arm, cols in ARMS10.items(): rows.append((label, f"held out: {s}", n, arm, *rf_eval(tr, te, cols)))
        for f in FE: rows.append((label, f"held out: {s}", n, f"Rule: {f}", *boot_auc(te.y, rule_score(te, f), te.set_id)))
        rows.append((label, f"held out: {s}", n, "Rule: lag + ccr (rank average)", *boot_auc(te.y, two_rule(te), te.set_id)))
    t = pd.DataFrame(rows, columns=["sample", "design", "n_cases_test", "scorer", "AUC", "CI_low", "CI_high"])
    return t.sort_values(["design", "AUC"], ascending=[True, False])

rice_all = rdata.assign(scheme="Rice")
E = pd.concat([loso_table(pd.concat([data, rice_all]), "v9 matching + rice"),
               loso_table(data10, "v10 matching (C-Quest, Guizhou)")])
print("\nE. LEAVE-ONE-SCHEME-OUT: trained RF vs no-training rules (CI clustered by match set)")
print(save(E, "E_loso_baselines").to_string(index=False))

# ---------------------------------------------------------------- F. SENSITIVITY on the 23-case set
F_rows = []
grid = list(itertools.product(["meth", "type"], [1, 2, None], [1, 3, 5], [0.5]))  + \
       [("meth", 1, 5, 0.25), ("meth", 1, 5, 1.0)]
for exact, vw, k, cm in grid:
    pr_, rep_, _ = match(cs23, cp23, exact=exact, vint_win=vw, k=k, cal_mult=cm)
    if len(pr_) == 0: continue
    lab = f"exact={exact}, vintage={'±'+str(vw) if vw is not None else 'free'}, k={k}, caliper={cm}SD"
    t = feature_tests(assemble(cs23, X, pr_), lab)
    for _, r in t[t.feature.isin(["ccr", "mean_issuance_lag_days"])].iterrows():
        F_rows.append({"variant": lab, "cases_matched": int((rep_.n_controls > 0).sum()), "controls": len(pr_),
                       "feature": r.feature, "r": r.r, "CI95": r.CI95, "q_all5": r.q_all5})
# developer sensitivity: Guizhou only (C-Quest removed) against the never-flagged population
gz = X.reindex(sorted(i for i in NEW_SET if SCHEME[i] == "Guizhou")); pop = X.loc[sorted(ctrl)]
for f in ["ccr", "mean_issuance_lag_days"]:
    r, p = rank_biserial(gz[f].dropna().values, pop[f].dropna().values)
    F_rows.append({"variant": "C-Quest removed (Guizhou only) vs all never-flagged", "cases_matched": int(gz[f].notna().sum()),
                   "controls": int(pop[f].notna().sum()), "feature": f, "r": round(r, 3), "CI95": "", "q_all5": np.nan,
                   "p_unadjusted": round(p, 4)})
F = pd.DataFrame(F_rows)
print("\nF. SENSITIVITY (23 confirmed cases): claim ratio and issuance lag across matching designs")
print(save(F, "F_sensitivity_23cases").to_string(index=False))

# ---------------------------------------------------------------- G. POPULATION-LEVEL BASELINES (no matching)
P = pd.concat([cs23.assign(y=1), cp23.assign(y=0)])
cook = P.meth.astype(str).str.contains(r"VMR0006|AMS-II\.G|VMR0001|AMS-I\.E", regex=True) | \
       P.type.astype(str).str.contains("Energy demand", case=False)
G = []
for nm, s in [("Rule: low claim ratio (ccr)", -P.ccr), ("Rule: fast issuance (lag)", -P.mean_issuance_lag_days),
              ("Rule: lag + ccr (rank average)", (-P.ccr).rank(pct=True) + (-P.mean_issuance_lag_days).rank(pct=True)),
              ("Rule: cookstove / energy-demand methodology flag", cook.astype(float)),
              ("Rule: larger issuance (log10 credits)", P.lc)]:
    G.append({"scorer": nm, **dict(zip(["AUC", "CI_low", "CI_high"], boot_auc(P.y, s)))})
print(f"\nG. POPULATION BASELINES: 23 confirmed vs {int((P.y == 0).sum())} eligible never-flagged controls")
print(save(pd.DataFrame(G), "G_population_baselines").to_string(index=False))

shutil.make_archive("/content/v10_outputs", "zip", OUT10)
print("\nDONE -> download /content/v10_outputs.zip.")

In [ ]:
mm = rep10.loc[rep10.matched_on == "meth", "case"]
sub = data10[data10.set_id.isin(mm)]
print(rep10.groupby(["scheme", "matched_on"]).size(), "\n")
for f in ["ccr", "mean_issuance_lag_days"]:
    x, y = sub.loc[sub.y == 1, f].dropna().values, sub.loc[sub.y == 0, f].dropna().values
    r, p = rank_biserial(x, y); lo, hi = boot_r(x, y)
    print(f"{f}: cases {len(x)}, controls {len(y)}, r = {r:+.3f} [{lo:+.2f}, {hi:+.2f}], p = {p:.4f}")

In [ ]:
cm = sorted(set(cs23.meth.dropna()))
rows = []
for m in cm:
    xc, xk = cs23.loc[cs23.meth == m, "ccr"].dropna(), cp23.loc[cp23.meth == m, "ccr"].dropna()
    rows.append({"methodology": m, "n_cases": len(xc), "case_median_ccr": round(xc.median(), 3),
                 "n_controls": len(xk), "control_median_ccr": round(xk.median(), 3) if len(xk) else None})
print(pd.DataFrame(rows).to_string(index=False))
x = cs23.ccr.dropna().values; y = cp23.loc[cp23.meth.isin(cm), "ccr"].dropna().values
r, p = rank_biserial(x, y); lo, hi = boot_r(x, y)
print(f"\nAll cases vs ALL same-methodology controls: cases {len(x)}, controls {len(y)}, "
      f"r = {r:+.3f} [{lo:+.2f}, {hi:+.2f}], p = {p:.4f}")
print("Median ccr, all eligible controls:", round(cp23.ccr.median(), 3))

## 12. Registry corrections and independent Verra reviews
Visibility of corrections in public issuance data, the December 2025 China reviews, and the pooled within-methodology test.

In [ ]:
# ===================== CORRECTIONS ACROSS ALL VERRA PROJECTS =====================
# Question: how often does Verra correct a project's credits WITHOUT the project's status changing?
# Run in the SAME Colab session after v9 (uses vp, tier, pool, info, NEW_SET, CQC_COMP, SCHEME).
#
# Two independent signals that a project's credits were corrected:
#   S1  "Vanished credits": batches issued on or before 31 Dec 2024 that appear in an earlier VCU snapshot
#       but whose issued quantity is lower (or missing) in a later snapshot.
#   S2  "Correction cancellations": cancelled batches whose Retirement Reason / Details mention
#       compensation, over-issuance, correction, error, invalidation and similar wording.
# A correction is SILENT if the project's status never moved to watch or risk_flag in any snapshot.
# Validation: both signals are checked against the 19 C-Quest projects Verra itself listed as compensated.
import os, re, shutil, numpy as np, pandas as pd

OUT11 = "/content/v11_outputs"; os.makedirs(OUT11, exist_ok=True)
def save(df, name): df.to_csv(f"{OUT11}/{name}.csv", index=False); return df
DATA11 = "/content"                             # folder holding the VROD registry files
FILES11 = {"2024-12": "VROD-registry-files--2024-12-year-end.xlsx",     # own copy: does not rely on v9's `F`
           "2025-12": "VROD-registry-files--2025-12.xlsx",
           "2026-02": "VROD-registry-files--2026-02.xlsx"}
ISSUED_BY = pd.Timestamp("2024-12-31")          # compare only batches that existed in every snapshot
MIN_DROP_VCU = 1                                # minimum drop counted as a correction (raise to 100 to ignore noise)

def load_vcus(path):
    with pd.ExcelFile(path) as b:
        sh = [n for n in b.sheet_names if n.strip().casefold() == "verra vcus"][0]
        t = pd.read_excel(b, sheet_name=sh)
    t["ID"] = pd.to_numeric(t["ID"], errors="coerce"); t = t.dropna(subset=["ID"]); t["ID"] = t["ID"].astype(int)
    for c in ["Issuance Date", "Vintage Start", "Vintage End", "Retirement/Cancellation Date"]:
        if c in t: t[c] = pd.to_datetime(t[c], errors="coerce")
    t["q"] = pd.to_numeric(t["Quantity Issued"], errors="coerce").fillna(0)
    return t

print("Loading VCU sheets from each snapshot (this takes a few minutes)...")
VCU = {s: load_vcus(f"{DATA11}/{f}") for s, f in FILES11.items()}
snaps_v = list(VCU)                                  # e.g. ['2024-12', '2025-12', '2026-02']
for s in snaps_v: print(f"  {s}: {len(VCU[s]):,} rows, {VCU[s].ID.nunique():,} projects")

# ---------------------------------------------------------------- S1. Vanished credits between snapshots
# Key each issuance batch by project + vintage window + issuance date; sum quantities for the batch.
def batch_totals(t):
    t = t[t["Issuance Date"] <= ISSUED_BY]
    k = t.groupby(["ID", "Vintage Start", "Vintage End", "Issuance Date"], dropna=False)["q"].sum()
    return k
BT = pd.concat({s: batch_totals(VCU[s]) for s in snaps_v}, axis=1).fillna(0)
first = snaps_v[0]
drops = []
for a, b in zip(snaps_v[:-1], snaps_v[1:]):
    d = (BT[a] - BT[b]).clip(lower=0)
    d = d[d >= MIN_DROP_VCU].groupby(level="ID").sum()
    drops.append(d.rename(f"drop_{a}_to_{b}"))
S1 = pd.concat(drops, axis=1).fillna(0)
S1["vanished_vcus"] = S1.sum(axis=1)
# Net change of the project's total (same batches): guards against batches that were merely re-labelled
tot = BT.groupby(level="ID").sum()
S1["net_change_vcus"] = (tot[snaps_v[-1]] - tot[first]).reindex(S1.index).values
S1 = S1[(S1.vanished_vcus > 0) & (S1.net_change_vcus < 0)]
proj_tot = VCU[first][VCU[first]["Issuance Date"] <= ISSUED_BY].groupby("ID")["q"].sum()
S1["issued_by_2024"] = proj_tot.reindex(S1.index).values
S1["vanished_share"] = (S1.vanished_vcus / S1.issued_by_2024).round(4)

# ---------------------------------------------------------------- S2. Correction-type cancellations
KEY = (r"compensat|over[\s-]?issu|overissu|over[\s-]?credit|correct|erroneous|error|invalid|"
       r"replac|reconcil|remediat|registry administrator|by verra|cancelled by verra|quantification")
last = VCU[snaps_v[-1]].copy()
txt = (last.get("Retirement Reason", pd.Series("", index=last.index)).astype(str) + " | " +
       last.get("Retirement Details", pd.Series("", index=last.index)).astype(str) + " | " +
       last.get("Retirement Beneficiary", pd.Series("", index=last.index)).astype(str))
last["is_corr"] = txt.str.contains(KEY, case=False, regex=True, na=False)
S2 = last[last.is_corr].groupby("ID")["q"].sum().rename("correction_cancelled_vcus").to_frame()

# Diagnostics for manual checking: which reason strings exist, which texts matched
reasons = last.get("Retirement Reason", pd.Series(dtype=str)).astype(str).value_counts().head(40)
save(reasons.rename_axis("retirement_reason").reset_index(name="batches"), "diag_retirement_reasons")
matched_txt = txt[last.is_corr].str.slice(0, 200).value_counts().head(60)
save(matched_txt.rename_axis("matched_text").reset_index(name="batches"), "diag_matched_texts")
print("\nTop retirement reasons (check that the correction keywords make sense):")
print(reasons.head(15).to_string())
print("\nMost common texts flagged as corrections (check these by eye):")
print(matched_txt.head(15).to_string())

# ---------------------------------------------------------------- Combine with status history
allp = sorted(set(S1.index) | set(S2.index))
R = pd.DataFrame(index=pd.Index(allp, name="ID"))
R = R.join(S1[["vanished_vcus", "net_change_vcus", "vanished_share", "issued_by_2024"]]).join(S2).fillna(
    {"vanished_vcus": 0, "correction_cancelled_vcus": 0})
R["S1_vanished"] = R.vanished_vcus > 0
R["S2_correction_text"] = R.correction_cancelled_vcus > 0
T = tier.reindex(R.index)
R["ever_watch_or_risk"] = T.isin(["watch", "risk_flag"]).any(axis=1).values
R["in_candidate_pool"] = R.index.isin(pool)
R["silent"] = ~R.ever_watch_or_risk
R["project_type"] = info.reindex(R.index)["Project Type"].values
meth_col = vp["2026-02"].set_index("ID")["Methodology"] if "Methodology" in vp["2026-02"] else pd.Series(dtype=str)
R["methodology"] = meth_col.reindex(R.index).values
R["proponent"] = info.reindex(R.index)["Proponent"].values
R["known_scheme"] = [SCHEME.get(i, "") for i in R.index]
R = R.reset_index()
save(R, "corrected_projects_all")

# ---------------------------------------------------------------- Validation against Verra's C-Quest table
VAL = []
cq = set(CQC_COMP)
for sig in ["S1_vanished", "S2_correction_text"]:
    hit = set(R.loc[R[sig], "ID"])
    VAL.append({"signal": sig, "C-Quest compensated projects": len(cq),
                "detected": len(cq & hit), "detection_%": round(100 * len(cq & hit) / len(cq), 1),
                "missed": sorted(cq - hit)})
hit_any = set(R.loc[R.S1_vanished | R.S2_correction_text, "ID"])
VAL.append({"signal": "S1 or S2", "C-Quest compensated projects": len(cq), "detected": len(cq & hit_any),
            "detection_%": round(100 * len(cq & hit_any) / len(cq), 1), "missed": sorted(cq - hit_any)})
print("\nVALIDATION: do the signals find the 19 projects Verra itself listed as compensated?")
print(save(pd.DataFrame(VAL), "validation_cquest").to_string(index=False))

# ---------------------------------------------------------------- Headline: how common are silent corrections?
def summary(mask, label):
    sub = R[mask]
    return {"projects": label, "n_corrected": len(sub), "n_silent": int(sub.silent.sum()),
            "silent_%": round(100 * sub.silent.mean(), 1) if len(sub) else np.nan,
            "n_in_candidate_pool": int(sub.in_candidate_pool.sum()),
            "VCUs_vanished": int(sub.vanished_vcus.sum()), "VCUs_correction_cancelled": int(sub.correction_cancelled_vcus.sum())}
H = pd.DataFrame([
    summary(R.S1_vanished | R.S2_correction_text, "Any signal, all Verra projects"),
    summary(R.S1_vanished & R.S2_correction_text, "Both signals agree"),
    summary(R.S1_vanished, "S1 vanished credits"),
    summary(R.S2_correction_text, "S2 correction cancellations"),
    summary((R.S1_vanished | R.S2_correction_text) & (R.known_scheme == ""), "Any signal, EXCLUDING the known schemes"),
])
print("\nHEADLINE: corrections and how many were silent (status never left 'normal')")
print(save(H, "headline_silent_corrections").to_string(index=False))

BY = (R[R.S1_vanished | R.S2_correction_text].assign(silent=lambda d: d.silent.astype(int))
      .groupby("project_type").agg(n_corrected=("ID", "size"), n_silent=("silent", "sum"),
                                   VCUs_vanished=("vanished_vcus", "sum"))
      .sort_values("n_corrected", ascending=False).reset_index())
print("\nBy project type:"); print(save(BY, "by_project_type").head(15).to_string(index=False))

top = (R[(R.S1_vanished | R.S2_correction_text) & (R.known_scheme == "")]
       .sort_values("vanished_vcus", ascending=False).head(25))
print("\nLargest corrections OUTSIDE the known schemes (check these by hand against Verra's project pages):")
print(save(top, "top_unknown_corrections")[["ID", "project_type", "proponent", "vanished_vcus", "net_change_vcus", "vanished_share",
                                              "correction_cancelled_vcus", "silent"]].to_string(index=False))

shutil.make_archive("/content/v11_outputs", "zip", OUT11)
print("\nDONE -> download /content/v11_outputs.zip.")

In [ ]:
# ===================== OUT-OF-SAMPLE TEST: VERRA'S DEC 2025 CHINA REVIEWS =====================
# Source: Verra, "Serious Allegations Prompt Verra to Reject China Projects and Launch Broader Reviews" (12 Dec 2025).
# 35 projects put on hold for quality-control review + 10 withdrawn AFOLU projects under review.
# These are UNDER REVIEW, not confirmed: treat them as an independent, graded label, never as fraud cases.
# Question: using only information available by 31 Dec 2024 (a year before the announcement),
# would the screen have flagged them? Run after v9 and v10 (uses X, tier, st, pool, ctrl, info, METH, rank_biserial, boot_r).
import os, shutil, numpy as np, pandas as pd

OUT12 = "/content/v12_outputs"; os.makedirs(OUT12, exist_ok=True)
def save(df, name): df.to_csv(f"{OUT12}/{name}.csv", index=False); return df

QC_REVIEW = [1332, 1361, 1529, 1542, 1577, 1664, 1715, 1718, 1825, 1826, 1832, 1866, 1895, 1896, 1935, 2249, 2326,
             2327, 2343, 2370, 2375, 2391, 2405, 2413, 2418, 2451, 2458, 2565, 2606, 2722, 2745, 2748, 2852, 3319, 4102]
WITHDRAWN_REVIEW = [1855, 1930, 2082, 2083, 2087, 2310, 2378, 2379, 2387, 2652]
GROUPS = {"QC review (35, on hold)": QC_REVIEW, "Withdrawn under review (10)": WITHDRAWN_REVIEW}
ALL45 = QC_REVIEW + WITHDRAWN_REVIEW

# 1. Who are they? (developer spread answers the "one developer" criticism)
prof = pd.DataFrame({"ID": ALL45, "group": ["QC review"] * 35 + ["Withdrawn review"] * 10})
prof["proponent"] = info.reindex(ALL45)["Proponent"].values
prof["project_type"] = info.reindex(ALL45)["Project Type"].values
prof["methodology"] = METH.reindex(ALL45).values
prof["has_issuance_by_2024"] = prof.ID.isin(X.index)
prof["ccr"] = X.reindex(ALL45)["ccr"].values
prof["in_candidate_pool"] = prof.ID.isin(pool)
for s in tier.columns: prof[f"tier_{s}"] = tier.reindex(ALL45)[s].values
save(prof, "china45_profile")
print(f"Found in status panel: {tier.index.isin(ALL45).sum()} of 45 | with issuance by 2024: {prof.has_issuance_by_2024.sum()}")
print(f"Distinct proponents: {prof.proponent.nunique()} | largest proponent share: "
      f"{prof.proponent.value_counts(normalize=True).iloc[0]:.0%}")
print(prof.methodology.value_counts(dropna=False).head(10).to_string())

# 2. Did status surveillance see them? (before vs after the announcement)
rows = []
for g, ids in GROUPS.items():
    t = tier.reindex(ids)
    for s in tier.columns:
        rows.append({"group": g, "snapshot": s, "normal": int((t[s] == "normal").sum()), "watch": int((t[s] == "watch").sum()),
                     "risk_flag": int((t[s] == "risk_flag").sum()), "missing": int(t[s].isna().sum())})
    rows.append({"group": g, "snapshot": "in candidate pool (any snapshot)", "normal": np.nan, "watch": np.nan,
                 "risk_flag": int(pd.Index(ids).isin(pool).sum()), "missing": np.nan})
print("\nStatus of the 45 at each snapshot (announcement was 12 Dec 2025):")
print(save(pd.DataFrame(rows), "china45_status").to_string(index=False))

# 3. Would the claim ratio have flagged them by 2024? vs all never-flagged controls AND vs same-methodology controls
cp = X.loc[sorted(ctrl)].assign(meth=METH.reindex(sorted(ctrl)).values)
res = []
for g, ids in list(GROUPS.items()) + [("All 45", ALL45)]:
    x = X.reindex(ids)["ccr"].dropna()
    if len(x) < 3: continue
    same = cp.loc[cp.meth.isin(set(METH.reindex(ids).dropna())), "ccr"].dropna()
    for ref_name, ref in [("all never-flagged controls", cp.ccr.dropna()), ("same-methodology controls", same)]:
        if len(ref) < 3: continue
        r, p = rank_biserial(x.values, ref.values); lo, hi = boot_r(x.values, ref.values)
        res.append({"group": g, "compared_with": ref_name, "n_group": len(x), "n_ref": len(ref),
                    "median_ccr_group": round(x.median(), 3), "median_ccr_ref": round(ref.median(), 3),
                    "r": round(r, 3), "CI95": f"[{lo:+.2f}, {hi:+.2f}]", "p": round(p, 4)})
print("\nClaim-consistency ratio by Dec 2024 (negative r = group issued less than claimed, relative to reference):")
print(save(pd.DataFrame(res), "china45_ccr").to_string(index=False))

shutil.make_archive("/content/v12_outputs", "zip", OUT12)
print("\nDONE -> download /content/v12_outputs.zip.")

In [ ]:
from scipy.stats import wilcoxon
flag = pd.DataFrame({"ID": sorted(NEW_SET) + ALL45,
                     "group": [SCHEME[i] for i in sorted(NEW_SET)] + ["China QC review"] * 35 + ["China withdrawn review"] * 10})
flag = flag.drop_duplicates("ID")
flag["meth"] = METH.reindex(flag.ID).values; flag["ccr"] = X.reindex(flag.ID)["ccr"].values
flag = flag.dropna(subset=["meth", "ccr"]); ref = cp23[["meth", "ccr"]].dropna()
def pct(row):
    r = ref.loc[ref.meth == row.meth, "ccr"]
    return np.nan if len(r) < 5 else ((r < row.ccr).sum() + 0.5 * (r == row.ccr).sum()) / len(r)
flag["pct_within_meth"] = flag.apply(pct, axis=1); f = flag.dropna(subset=["pct_within_meth"])
print(f.groupby("group").pct_within_meth.agg(["count", "median"]).round(2))
stat, p = wilcoxon(f.pct_within_meth - 0.5)
print(f"\nPooled: n = {len(f)}, median percentile within methodology = {f.pct_within_meth.median():.2f} "
      f"(0.50 = no difference), Wilcoxon p = {p:.4f}")

cc = VCU["2026-02"]; cc = cc[cc["Retirement Reason"].astype(str).eq("Credit Cancellation")]
print("\n'Credit Cancellation' rows by project:"); print(cc.groupby("ID")["q"].sum().sort_values(ascending=False).to_string())
print(cc[["ID", "Retirement Details", "Retirement Beneficiary"]].drop_duplicates().head(40).to_string())

In [ ]:
f = f.assign(proponent=info.reindex(f.ID)["Proponent"].fillna("unknown").values)
f.loc[f.proponent.eq("Multiple Proponents"), "proponent"] = "multi_" + f.ID.astype(str)
rng = np.random.default_rng(SEED)
def clustered(d, B=5000):
    devs = d.proponent.unique(); meds = []
    for _ in range(B):
        pick = rng.choice(devs, len(devs))
        meds.append(pd.concat([d[d.proponent == k] for k in pick]).pct_within_meth.median())
    return np.percentile(meds, [2.5, 97.5])
for name, d in [("All 47", f), ("Confirmed only (C-Quest + Guizhou)", f[f.group.isin(["C-Quest", "Guizhou"])]),
                ("Without China QC review", f[f.group != "China QC review"])]:
    lo, hi = clustered(d); stat, p = wilcoxon(d.pct_within_meth - 0.5)
    print(f"{name}: n = {len(d)}, developers = {d.proponent.nunique()}, median = {d.pct_within_meth.median():.2f}, "
          f"developer-clustered 95% CI [{lo:.2f}, {hi:.2f}], Wilcoxon p = {p:.4f}")

## 13. Revised figures

In [ ]:
# ===================== REVISED FIGURES 2, 3b, 4b, 7 AND 8 =====================
# Run in the SAME Colab session after v9, v9b, v10 and v12 (uses data, data10, rdata, FE, X, cp23, METH, info,
# tier, pool, NEW_SET, SCHEME, ALL45, QC_REVIEW, WITHDRAWN_REVIEW, E, rank_biserial, boot_r, boot_auc, SEED).
# If the session was restarted, run v9 -> v9b -> v10 -> v12 first, then this cell.
# Output: /content/v13_figures.zip with five 300-dpi PNGs and the data behind each figure (CSV).
import os, shutil, numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

OUT13 = "/content/v13_figures"; os.makedirs(OUT13, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GREEN, GREY, PURPLE, RED = "#0072B2", "#E69F00", "#009E73", "#7F7F7F", "#CC79A7", "#D55E00"
NAMES = {"n_batches": "Issuance batches", "share_retired": "Share of credits retired",
         "mean_issuance_lag_days": "Mean issuance lag", "batch_size_cv": "Batch-size CV",
         "ccr": "Issued ÷ own claimed reductions*"}
def savefig(fig, name): fig.tight_layout(); fig.savefig(f"{OUT13}/{name}.png", dpi=300, bbox_inches="tight"); plt.show(); plt.close(fig)

# ---------------------------------------------------------------- Figure 2: effect sizes under both matching designs + rice
designs = [("Type + size matching (v9; 23 cases, 84 controls)", data, BLUE, +0.22),
           ("Methodology + vintage + size matching (20 cases, 50 controls)", data10, ORANGE, 0.0),
           ("Rice scheme (8 cases, 33 controls)", rdata, GREEN, -0.22)]
rows = []
fig, ax = plt.subplots(figsize=(6.6, 3.6))
for lab, dd, col, off in designs:
    for k, f in enumerate(FE):
        x, y = dd.loc[dd.y == 1, f].dropna().values, dd.loc[dd.y == 0, f].dropna().values
        if len(x) < 3 or len(y) < 3: continue
        r = rank_biserial(x, y)[0]; lo, hi = boot_r(x, y)
        rows.append({"design": lab, "feature": f, "r": round(r, 3), "lo": round(lo, 3), "hi": round(hi, 3)})
        ax.plot([lo, hi], [k + off, k + off], color=col, lw=2.2)
        ax.plot(r, k + off, "o", color=col, mfc="white", ms=5, label=lab if k == 0 else None)
ax.axvline(0, color=GREY, ls=":", lw=1); ax.set_yticks(range(len(FE))); ax.set_yticklabels([NAMES[f] for f in FE])
ax.set_xlim(-1, 1); ax.set_xlabel("Rank-biserial r with 95% bootstrap interval\n(negative = lower in flagged projects)")
ax.invert_yaxis(); ax.legend(loc="upper center", fontsize=7, frameon=False, bbox_to_anchor=(0.4, -0.36), ncol=1)
ax.set_title("Issuance lag loses its effect once vintage is matched;\nthe claim ratio keeps it", loc="left")
savefig(fig, "Figure2_effect_sizes_v13"); pd.DataFrame(rows).to_csv(f"{OUT13}/Figure2_data.csv", index=False)

# ---------------------------------------------------------------- Figure 3: claim ratio within methodology
grp_of = {**{i: SCHEME[i] for i in NEW_SET}, **{i: "China QC review" for i in QC_REVIEW},
          **{i: "China withdrawn review" for i in WITHDRAWN_REVIEW}}
COLS = {"C-Quest": BLUE, "Guizhou": ORANGE, "China QC review": PURPLE, "China withdrawn review": RED}
flagged = pd.DataFrame({"ID": list(grp_of)}).assign(group=lambda d: d.ID.map(grp_of))
flagged["meth"] = METH.reindex(flagged.ID).values; flagged["ccr"] = X.reindex(flagged.ID)["ccr"].values
flagged = flagged.dropna(subset=["ccr"]); flagged = flagged[flagged.ccr > 0]
ref = cp23[["meth", "ccr"]].dropna(); ref = ref[ref.ccr > 0]
top = flagged.meth.value_counts(); top = [m for m in top.index if (ref.meth == m).sum() >= 3][:5]
fig, ax = plt.subplots(figsize=(6.8, 3.6)); rng = np.random.default_rng(SEED); pos = 0; ticks, labels, f3 = [], [], []
allc = ref.ccr.values
ax.boxplot(allc, positions=[pos], widths=0.5, showfliers=False, medianprops={"color": "black"}); ax.hlines(np.median(allc), pos - .3, pos + .3, color="black", lw=1.5)
ticks.append(pos); labels.append(f"All never-\nflagged\n(n={len(allc)})"); pos += 1.4
for m in top:
    rc, fc = ref.loc[ref.meth == m, "ccr"].values, flagged[flagged.meth == m]
    ax.boxplot(rc, positions=[pos], widths=0.45, showfliers=False, medianprops={"color": "black"})
    ax.scatter(pos + rng.uniform(-.12, .12, len(rc)), rc, color=GREY, s=7, alpha=.5)
    ax.hlines(np.median(rc), pos - .28, pos + .28, color="black", lw=1.5)
    for g, gd in fc.groupby("group"):
        ax.scatter(pos + 0.55 + rng.uniform(-.1, .1, len(gd)), gd.ccr, color=COLS[g], s=16, alpha=.9, edgecolor="white", lw=.3)
    ax.hlines(np.median(fc.ccr), pos + .3, pos + .8, color="black", lw=1.5, ls="--")
    ticks.append(pos + .28); labels.append(f"{m}\n(controls {len(rc)},\nflagged {len(fc)})"); pos += 1.8
    f3.append({"methodology": m, "n_controls": len(rc), "control_median": round(np.median(rc), 3),
               "n_flagged": len(fc), "flagged_median": round(fc.ccr.median(), 3)})
ax.axhline(1, color=GREY, ls=":", lw=1); ax.set_yscale("log"); ax.set_xticks(ticks); ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel("Issued ÷ own claimed reductions (log)")
for g, c in COLS.items(): ax.scatter([], [], color=c, s=16, label=g)
ax.legend(fontsize=7, frameon=False, loc="upper right", ncol=2)
ax.set_title("Grey: never-flagged projects of the same methodology (solid line = median)\n"
             "Coloured: flagged projects (dashed line = median)", loc="left", fontsize=8)
savefig(fig, "Figure3_claim_ratio_by_methodology_v13"); pd.DataFrame(f3).to_csv(f"{OUT13}/Figure3_data.csv", index=False)

# ---------------------------------------------------------------- Figure 4: trained model vs no-training rules
e = E[E["sample"] == "v9 matching + rice"].copy()
pick = []
for d in e.design.unique():
    sub = e[e.design == d]; rf = sub[sub.scorer == "RF full"].iloc[0]
    rules = sub[sub.scorer.str.startswith("Rule")].sort_values("AUC", ascending=False).iloc[0]
    claim = sub[sub.scorer == "Rule: ccr"].iloc[0]
    trio = [("Random Forest (full)", rf), (f"Best rule: {rules.scorer[6:]}", rules)]
    if rules.scorer != "Rule: ccr": trio.append(("Claim-ratio rule", claim))
    for kind, row in trio:
        pick.append({"test": f"{d.replace('held out: ', '')} held out (n={int(row.n_cases_test)})", "scorer": kind,
                     "AUC": row.AUC, "lo": row.CI_low, "hi": row.CI_high})
Ppop = pd.concat([cs23.assign(y=1), cp23.assign(y=0)])
cook = Ppop.meth.astype(str).str.contains(r"VMR0006|AMS-II\.G|VMR0001|AMS-I\.E", regex=True) | \
       Ppop.type.astype(str).str.contains("Energy demand", case=False)
for kind, s in [("Cookstove-methodology flag", cook.astype(float)),
                ("Best rule: lag + ccr (rank average)", (-Ppop.ccr).rank(pct=True) + (-Ppop.mean_issuance_lag_days).rank(pct=True))]:
    a, lo, hi = boot_auc(Ppop.y, s); pick.append({"test": f"All controls (23 vs {int((Ppop.y == 0).sum())})", "scorer": kind, "AUC": a, "lo": lo, "hi": hi})
P4 = pd.DataFrame(pick)
fig, ax = plt.subplots(figsize=(6.6, 0.33 * len(P4) + 1.0))
cmap = lambda s: BLUE if s.startswith("Random") else (GREEN if s.startswith("Best") else (ORANGE if s.startswith("Claim") else PURPLE))
yl = []
for k, r in P4.iloc[::-1].reset_index(drop=True).iterrows():
    c = cmap(r.scorer); ax.plot([r.lo, r.hi], [k, k], color=c, lw=2); ax.plot(r.AUC, k, "o", color=c, mfc="white", ms=5)
    yl.append(f"{r.test} · {r.scorer}")
ax.set_yticks(range(len(P4))); ax.set_yticklabels(yl, fontsize=7); ax.axvline(.5, color=GREY, ls=":", lw=1)
ax.set_xlim(0, 1); ax.set_xlabel("AUC with 95% interval (dotted line = chance)")
ax.set_title("No trained model beats the best no-training rule", loc="left")
savefig(fig, "Figure4_rules_vs_model_v13"); P4.to_csv(f"{OUT13}/Figure4_data.csv", index=False)

# ---------------------------------------------------------------- Figure 7 (new): within-methodology percentile by group
fl = flagged.copy(); refm = cp23[["meth", "ccr"]].dropna()
def pct(row):
    r = refm.loc[refm.meth == row.meth, "ccr"]
    return np.nan if len(r) < 5 else ((r < row.ccr).sum() + 0.5 * (r == row.ccr).sum()) / len(r)
fl["pct"] = fl.apply(pct, axis=1); fl = fl.dropna(subset=["pct"])
fl["proponent"] = info.reindex(fl.ID)["Proponent"].fillna("unknown").values
fl.loc[fl.proponent.eq("Multiple Proponents"), "proponent"] = "multi_" + fl.ID.astype(str)
def clustered(d, B=5000):
    devs = d.proponent.unique(); r = np.random.default_rng(SEED); meds = []
    for _ in range(B):
        meds.append(pd.concat([d[d.proponent == k] for k in r.choice(devs, len(devs))]).pct.median())
    return np.percentile(meds, [2.5, 97.5])
order = ["China QC review", "China withdrawn review", "C-Quest", "Guizhou"]
fig, ax = plt.subplots(figsize=(6.4, 3.2)); rows7 = []
for k, g in enumerate(order + ["All flagged"]):
    d = fl if g == "All flagged" else fl[fl.group == g]
    if len(d) == 0: continue
    c = GREY if g == "All flagged" else COLS[g]
    ax.scatter(np.full(len(d), k) + rng.uniform(-.15, .15, len(d)), d.pct, color=c, s=16, alpha=.8, edgecolor="white", lw=.3)
    med = d.pct.median(); lo, hi = clustered(d) if d.proponent.nunique() > 1 else (np.nan, np.nan)
    ax.hlines(med, k - .28, k + .28, color="black", lw=1.8)
    if np.isfinite(lo): ax.plot([k + .34, k + .34], [lo, hi], color="black", lw=1.2)
    p = wilcoxon(d.pct - 0.5).pvalue if len(d) >= 6 else np.nan
    rows7.append({"group": g, "n": len(d), "developers": d.proponent.nunique(), "median_pct": round(med, 3),
                  "clustered_lo": round(lo, 3) if np.isfinite(lo) else None, "clustered_hi": round(hi, 3) if np.isfinite(hi) else None,
                  "wilcoxon_p": round(p, 4) if np.isfinite(p) else None})
ax.axhline(.5, color=GREY, ls=":", lw=1); ax.set_ylim(0, 1)
ax.set_xticks(range(len(rows7))); ax.set_xticklabels([f"{r['group']}\n(n={r['n']})" for r in rows7], fontsize=7)
ax.set_ylabel("Percentile of claim ratio among\nsame-methodology never-flagged projects")
ax.set_title("Flagged projects sit below the median of similar projects\n"
             "(black bar = median; vertical line = developer-clustered 95% CI)", loc="left", fontsize=8)
savefig(fig, "Figure7_within_methodology_percentile"); pd.DataFrame(rows7).to_csv(f"{OUT13}/Figure7_data.csv", index=False)

# ---------------------------------------------------------------- Figure 8 (new): status before and after Verra acted
STATES, SC = ["normal", "watch", "risk_flag"], {"normal": "#BBBBBB", "watch": ORANGE, "risk_flag": RED}
missed_cq = sorted(i for i in NEW_SET if SCHEME[i] == "C-Quest" and i not in pool)
panels = [("China QC review (35)\nVerra action: Dec 2025", QC_REVIEW),
          ("C-Quest projects missed by the pool (12)\nVerra compensation: Oct 2024", missed_cq)]
fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.9), sharey=True); rows8 = []
for ax, (title, ids) in zip(axes, panels):
    t = tier.reindex(ids); bottom = np.zeros(len(t.columns))
    for s in STATES:
        v = (t == s).sum().values; ax.bar(range(len(t.columns)), v, bottom=bottom, color=SC[s], label=s.replace("_", " "), width=.65)
        bottom += v
        for snap, n in zip(t.columns, v): rows8.append({"panel": title.split("\n")[0], "snapshot": snap, "state": s, "n": int(n)})
    ax.set_xticks(range(len(t.columns))); ax.set_xticklabels(t.columns, fontsize=8); ax.set_title(title, fontsize=8, loc="left")
axes[0].set_ylabel("Projects"); axes[1].legend(fontsize=7, frameon=False, loc="upper right")
fig.suptitle("Registry status before and after Verra acted", x=0.01, ha="left", fontsize=9)
savefig(fig, "Figure8_status_before_after"); pd.DataFrame(rows8).to_csv(f"{OUT13}/Figure8_data.csv", index=False)

shutil.make_archive("/content/v13_figures", "zip", OUT13)
print("DONE -> download /content/v13_figures.zip.")

## 14. Package versions and download of all outputs

In [ ]:
import sys, platform, importlib.metadata as md
pkgs = ["numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "matplotlib", "openpyxl", "xgboost", "shap", "requests", "Pillow"]
with open("requirements_used.txt", "w") as fh:
    fh.write(f"# Python {platform.python_version()} ({sys.platform})\n")
    for p in pkgs:
        try: fh.write(f"{p}=={md.version(p)}\n")
        except md.PackageNotFoundError: pass
print(open("requirements_used.txt").read())

import os, shutil
folders = [f for f in ["v4_outputs", "v5_outputs", "/content/v9_outputs", "/content/v10_outputs", "/content/v11_outputs",
                       "/content/v12_outputs", "/content/v13_figures"] if os.path.isdir(f)]
os.makedirs("/content/aman_carbon_ai_outputs", exist_ok=True)
for f in folders:
    shutil.copytree(f, os.path.join("/content/aman_carbon_ai_outputs", os.path.basename(f.rstrip("/"))), dirs_exist_ok=True)
for extra in ["requirements_used.txt", "v5_outputs/input_file_hashes.csv"]:
    if os.path.exists(extra): shutil.copy(extra, "/content/aman_carbon_ai_outputs")
shutil.make_archive("/content/aman_carbon_ai_outputs", "zip", "/content/aman_carbon_ai_outputs")
print("Zipped:", ", ".join(os.path.basename(f) for f in folders))
try:
    from google.colab import files
    files.download("/content/aman_carbon_ai_outputs.zip")
except ImportError:
    print("Not running in Colab: the zip is at /content/aman_carbon_ai_outputs.zip")